In [ ]:
# Notebook 11

# Feature Selection

**Project:** Machine Learning-Based Prediction of Degradation in PEM Fuel Cells Using Time-Series Operational Data

This notebook performs systematic feature selection for the PEM fuel cell voltage-prediction problem established in the preceding analysis.

The candidate predictor set contains original operational variables together with six physics-informed engineered features developed during feature engineering.

Feature selection is performed using a structured hybrid methodology combining methodological validity screening, statistical filter methods, multicollinearity assessment, embedded regularised selection, nonlinear model-based selection, and validation-based confirmation.

Because the PEMFC dataset contains strong predictor relationships, repeated dynamic operating behaviour, and a chronological durability structure, features are not selected solely according to their individual association with voltage. Target relevance, predictor redundancy, multicollinearity, nonlinear dependence, physical interpretation, prediction-time validity, and temporal generalisation are considered jointly.

The final objective is to establish a scientifically defensible predictor set for subsequent comparison of Ridge Regression, XGBoost, and feed-forward Artificial Neural Network models.

Final test data will remain excluded from feature-selection decisions to prevent information leakage and preserve an unbiased evaluation of model generalisation to later durability stages.

In [ ]:
## Objectives

The objectives of this notebook are to:

- Define the legitimate candidate predictor set for voltage prediction.
- Establish a chronological training, validation, and final test structure based on durability stages.
- Identify and exclude variables that would introduce target leakage.
- Evaluate individual predictor relevance to voltage using Pearson correlation, Spearman rank correlation, and Mutual Information.
- Assess pairwise predictor redundancy using Pearson and Spearman associations.
- Evaluate multivariate linear redundancy using Variance Inflation Factor (VIF).
- Obtain regularised linear feature-selection evidence using Elastic Net.
- Obtain nonlinear feature-selection evidence using an XGBoost-based Boruta approach.
- Integrate statistical, model-based, physical, and redundancy evidence.
- Compare candidate feature subsets using temporally appropriate validation data.
- Confirm predictive contribution using permutation importance where appropriate.
- Assess feature-selection stability across durability stages or temporal validation folds.
- Establish and document the final feature set for subsequent predictive modelling.

No variable will be retained or removed solely on the basis of a single statistical threshold.

In [ ]:
## Notebook Workflow

This notebook follows the structure below:

### E1 Feature Selection

- E1.1 Introduction and Objective
- E1.2 Feature-Selection Dataset Preparation
- E1.3 Chronological Train–Validation–Test Definition
- E1.4 Predictor Validity and Leakage Screening
- E1.5 Filter-Based Target Relevance
  - E1.5.1 Pearson Correlation
  - E1.5.2 Spearman Rank Correlation
  - E1.5.3 Mutual Information
  - E1.5.4 Integrated Target-Relevance Assessment
- E1.6 Predictor Redundancy Assessment
  - E1.6.1 Pairwise Pearson Redundancy
  - E1.6.2 Pairwise Spearman Redundancy
  - E1.6.3 Redundancy Groups
- E1.7 Multicollinearity Assessment
  - E1.7.1 Variance Inflation Factor
- E1.8 Embedded Linear Feature Selection
  - E1.8.1 Elastic Net
- E1.9 Nonlinear Hybrid Feature Selection
  - E1.9.1 XGBoost-Boruta
- E1.10 Integrated Feature-Selection Evidence
- E1.11 Candidate Feature-Set Construction
- E1.12 Validation-Based Feature-Set Comparison
- E1.13 Permutation Importance
- E1.14 Temporal Feature-Stability Assessment
- E1.15 Final Feature-Selection Decision
- E1.16 Save Selected Feature Specification
- E1.17 Feature Selection Summary

In [1]:
# ============================================================
# Import Required Libraries
# ============================================================

import warnings
warnings.filterwarnings("default")

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

print("Required libraries imported successfully.")

Required libraries imported successfully.


In [2]:
# ============================================================
# Notebook Configuration
# ============================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.4f}".format)

plt.style.use("default")

print("Notebook configured successfully.")

Notebook configured successfully.


In [3]:
# ============================================================
# Define Project Paths
# ============================================================

project_root = Path.cwd().parent

raw_data_dir = project_root / "data" / "raw"
processed_data_dir = project_root / "data" / "processed"

figures_dir = project_root / "figures" / "feature_selection"
results_dir = project_root / "results" / "feature_selection"

figures_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

print("Project Root       :", project_root)
print("Processed Data     :", processed_data_dir)
print("Selection Figures  :", figures_dir)
print("Selection Results  :", results_dir)

Project Root       : C:\Users\usman\Desktop\PEMFC_Dissertation
Processed Data     : C:\Users\usman\Desktop\PEMFC_Dissertation\data\processed
Selection Figures  : C:\Users\usman\Desktop\PEMFC_Dissertation\figures\feature_selection
Selection Results  : C:\Users\usman\Desktop\PEMFC_Dissertation\results\feature_selection


In [4]:
# ============================================================
# Configure Project Source Package
# ============================================================

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root added to Python path successfully.")

Project root added to Python path successfully.


In [ ]:
## Load Feature-Engineered Dataset

The feature-engineered dataset produced in Notebook 10 is used as the starting point for feature selection.

This dataset contains the original cleaned PEMFC operational variables together with the six physics-informed engineered features.

Feature selection is therefore performed on the complete candidate feature space rather than on the original operational variables alone.

In [5]:
# ============================================================
# Load Feature-Engineered Dataset
# ============================================================

feature_engineered_file = (
    processed_data_dir / "pemfc_feature_engineered.csv"
)

df = pd.read_csv(feature_engineered_file)

print("Feature-engineered dataset loaded successfully.")
print("Dataset shape:", df.shape)

Feature-engineered dataset loaded successfully.
Dataset shape: (3629680, 24)


In [ ]:
## Verify Dataset

The loaded dataset is verified before feature-selection analysis begins.

The verification confirms that the expected number of observations and variables has been preserved and that the feature-engineered dataset has been loaded correctly.

In [6]:
# ============================================================
# Verify Dataset
# ============================================================

print("Number of observations :", f"{df.shape[0]:,}")
print("Number of variables    :", df.shape[1])

print("\nMissing values:")
print(df.isna().sum().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

Number of observations : 3,629,680
Number of variables    : 24

Missing values:
0

Duplicate rows:
0


In [ ]:
## Dataset Structure

The available variables are reviewed to confirm the complete candidate feature space before predictor roles are assigned.

At this stage, no variable is removed. Variables will subsequently be classified according to their role as target, operational predictor, engineered predictor, temporal variable, or leakage-prone variable.

In [7]:
# ============================================================
# Dataset Structure
# ============================================================

dataset_structure = pd.DataFrame({
    "Variable": df.columns,
    "Data_Type": df.dtypes.astype(str).values
})

dataset_structure

,Variable,Data_Type
0,operating_hour,int64
1,time,float64
2,current,float64
3,voltage,float64
4,power,float64
5,pressure_anode_inlet,float64
6,pressure_anode_outlet,float64
7,pressure_cathode_inlet,float64
8,pressure_cathode_outlet,float64
9,temp_anode_endplate,float64


In [ ]:
## Preview Dataset

The first observations are displayed to verify that the original and engineered variables are aligned correctly and to inspect the structure of the dataset before defining the feature-selection predictor space.

In [8]:
# ============================================================
# Preview Dataset
# ============================================================

df.head()

,operating_hour,time,current,voltage,power,pressure_anode_inlet,pressure_anode_outlet,pressure_cathode_inlet,pressure_cathode_outlet,temp_anode_endplate,temp_anode_dewpoint_water,temp_anode_inlet,temp_anode_outlet,temp_cathode_dewpoint_water,temp_cathode_inlet,temp_cathode_outlet,total_anode_stack_flow,total_cathode_stack_flow,anode_pressure_diff,cathode_pressure_diff,anode_temp_diff,cathode_temp_diff,anode_dewpoint_offset,cathode_dewpoint_offset
0,50,1.7610,0.0000,0.9375,0.0000,109.9013,110.3273,109.8001,108.7921,83.1282,54.4647,71.9515,39.4532,64.4643,69.6736,56.3375,0.0700,0.2910,-0.4259,1.0080,-32.4983,-13.3360,17.4867,5.2093
1,50,2.7610,0.0000,0.9375,0.0000,110.1037,110.3273,109.8001,108.8934,83.0788,54.4647,71.9021,39.3870,64.4390,69.6612,56.3249,0.0700,0.2910,-0.2236,0.9067,-32.5150,-13.3363,17.4373,5.2222
2,50,3.7610,0.0000,0.9372,0.0000,110.3060,110.3273,109.8001,108.7921,83.0788,54.5293,71.8897,39.4135,64.4866,69.6859,56.2997,0.0700,0.2910,-0.0212,1.0080,-32.4763,-13.3863,17.3604,5.1993
3,50,4.7610,0.0000,0.9375,0.0000,110.1037,110.3273,109.8001,108.7921,83.1035,54.4777,71.9391,39.3870,64.4390,69.6859,56.3249,0.0700,0.2910,-0.2236,1.0080,-32.5521,-13.3610,17.4615,5.2469
4,50,5.7610,0.0000,0.9372,0.0000,109.9013,110.3273,109.6990,108.8934,83.0911,54.4777,71.8897,39.3738,64.4895,69.6612,56.2492,0.0700,0.2910,-0.4259,0.8056,-32.5159,-13.4121,17.4121,5.1717


In [ ]:
## Verify Engineered Features

The six engineered features created in Notebook 10 are explicitly checked before feature selection begins.

This ensures that all engineered candidates are available for comparison with their original parent variables.

In [9]:
# ============================================================
# Verify Engineered Features
# ============================================================

engineered_features = [
    "anode_pressure_diff",
    "cathode_pressure_diff",
    "anode_temp_diff",
    "cathode_temp_diff",
    "anode_dewpoint_offset",
    "cathode_dewpoint_offset"
]

engineered_feature_check = pd.DataFrame({
    "Feature": engineered_features,
    "Present": [
        feature in df.columns
        for feature in engineered_features
    ]
})

engineered_feature_check

,Feature,Present
0,anode_pressure_diff,True
1,cathode_pressure_diff,True
2,anode_temp_diff,True
3,cathode_temp_diff,True
4,anode_dewpoint_offset,True
5,cathode_dewpoint_offset,True


In [ ]:
## E1.2 Feature-Selection Dataset Preparation

Before applying feature-selection methods, the variables in the feature-engineered dataset are classified according to their role in the prediction problem.

The selected prediction target is stack voltage. Candidate predictors consist of valid operational measurements and the six physics-informed engineered features developed previously.

Variables that contain target information, represent temporal or experimental structure, or are otherwise not intended to act as ordinary operational predictors are identified separately before statistical feature selection.

This step establishes the candidate predictor space that will subsequently be used for chronological data splitting and feature-selection analysis.

In [10]:
# ============================================================
# E1.2 Define Target and Available Variables
# ============================================================

target_variable = "voltage"

print("Target variable:", target_variable)

print("\nAvailable variables:")
for i, column in enumerate(df.columns, start=1):
    print(f"{i:2d}. {column}")

Target variable: voltage

Available variables:
 1. operating_hour
 2. time
 3. current
 4. voltage
 5. power
 6. pressure_anode_inlet
 7. pressure_anode_outlet
 8. pressure_cathode_inlet
 9. pressure_cathode_outlet
10. temp_anode_endplate
11. temp_anode_dewpoint_water
12. temp_anode_inlet
13. temp_anode_outlet
14. temp_cathode_dewpoint_water
15. temp_cathode_inlet
16. temp_cathode_outlet
17. total_anode_stack_flow
18. total_cathode_stack_flow
19. anode_pressure_diff
20. cathode_pressure_diff
21. anode_temp_diff
22. cathode_temp_diff
23. anode_dewpoint_offset
24. cathode_dewpoint_offset


In [11]:
# ============================================================
# E1.2 Variable Role Classification
# ============================================================

engineered_features = [
    "anode_pressure_diff",
    "cathode_pressure_diff",
    "anode_temp_diff",
    "cathode_temp_diff",
    "anode_dewpoint_offset",
    "cathode_dewpoint_offset"
]

variable_roles = []

for column in df.columns:

    if column == target_variable:
        role = "Target"

    elif column == "power":
        role = "Potential target leakage"

    elif column == "time":
        role = "Temporal / ordering variable"

    elif column == "operating_hour":
        role = "Durability-stage variable"

    elif column in engineered_features:
        role = "Engineered predictor"

    else:
        role = "Original operational predictor"

    variable_roles.append({
        "Variable": column,
        "Role": role
    })

variable_role_table = pd.DataFrame(variable_roles)

variable_role_table

,Variable,Role
0,operating_hour,Durability-stage variable
1,time,Temporal / ordering variable
2,current,Original operational predictor
3,voltage,Target
4,power,Potential target leakage
5,pressure_anode_inlet,Original operational predictor
6,pressure_anode_outlet,Original operational predictor
7,pressure_cathode_inlet,Original operational predictor
8,pressure_cathode_outlet,Original operational predictor
9,temp_anode_endplate,Original operational predictor


In [12]:
# ============================================================
# E1.2 Variable Role Summary
# ============================================================

role_summary = (
    variable_role_table["Role"]
    .value_counts()
    .rename_axis("Role")
    .reset_index(name="Count")
)

role_summary

,Role,Count
0,Original operational predictor,14
1,Engineered predictor,6
2,Durability-stage variable,1
3,Temporal / ordering variable,1
4,Target,1
5,Potential target leakage,1


In [ ]:
## E1.3 Chronological Train–Validation–Test Definition

Feature selection and subsequent predictive modelling must respect the chronological durability structure of the PEMFC experiment.

Random row-level splitting would allow observations from the same or later durability stages to appear across training, validation, and test sets. Because the measurements originate from repeated dynamic operating cycles, this could produce overly optimistic estimates of model generalisation.

The dataset is therefore partitioned according to `operating_hour`, preserving the progression of PEMFC durability stages.

The three subsets have distinct methodological roles:

- **Training set:** used to fit feature-selection procedures and predictive models.
- **Validation set:** used during model development to compare feature subsets and subsequently support model and hyperparameter selection.
- **Final test set:** reserved for final evaluation and excluded from all feature-selection and model-development decisions.

Before defining the split boundaries, the available durability stages and their observation counts are examined.

In [13]:
# ============================================================
# E1.3 Inspect Durability-Stage Structure
# ============================================================

durability_stage_counts = (
    df.groupby("operating_hour")
      .size()
      .reset_index(name="observations")
)

durability_stage_counts["percentage"] = (
    durability_stage_counts["observations"]
    / len(df)
    * 100
)

durability_stage_counts

,operating_hour,observations,percentage
0,50,179360,4.9415
1,100,179360,4.9415
2,150,179360,4.9415
3,200,179360,4.9415
4,250,179360,4.9415
5,300,179360,4.9415
6,350,179360,4.9415
7,400,179360,4.9415
8,450,179360,4.9415
9,500,179360,4.9415


In [14]:
# ============================================================
# E1.3 Durability-Stage Summary
# ============================================================

print("Number of durability stages:",
      df["operating_hour"].nunique())

print("\nDurability stages:")
print(
    sorted(df["operating_hour"].unique())
)

print("\nMinimum operating hour:",
      df["operating_hour"].min())

print("Maximum operating hour:",
      df["operating_hour"].max())

Number of durability stages: 20

Durability stages:
[np.int64(50), np.int64(100), np.int64(150), np.int64(200), np.int64(250), np.int64(300), np.int64(350), np.int64(400), np.int64(450), np.int64(500), np.int64(550), np.int64(600), np.int64(650), np.int64(700), np.int64(750), np.int64(800), np.int64(850), np.int64(900), np.int64(950), np.int64(1000)]

Minimum operating hour: 50
Maximum operating hour: 1000


In [15]:
# ============================================================
# E1.3 Check Dataset Stage Ordering
# ============================================================

stage_sequence = df["operating_hour"].to_numpy()

is_stage_ordered = np.all(
    stage_sequence[:-1] <= stage_sequence[1:]
)

print("Dataset ordered chronologically by durability stage:",
      is_stage_ordered)

Dataset ordered chronologically by durability stage: True


In [ ]:
### E1.3.1 Post-Feature-Engineering Temporal Integrity Verification

The temporal integrity of the operational datasets was comprehensively assessed during earlier preprocessing and time-series integrity analysis. However, because the datasets were subsequently merged and six engineered features were added, a concise verification is performed before establishing the chronological modelling partitions.

This verification is not intended to repeat the earlier time-series analysis. Its purpose is to confirm that feature engineering preserved the original experimental structure, including durability-stage membership, observation counts, chronological ordering, sampling intervals, and absence of abnormal time gaps.

In [16]:
# ============================================================
# E1.3.1 Post-Feature-Engineering Temporal Integrity Check
# ============================================================

temporal_checks = []

for stage, stage_df in df.groupby("operating_hour", sort=True):

    time_values = stage_df["time"].to_numpy()
    time_diff = np.diff(time_values)

    temporal_checks.append({
        "operating_hour": stage,
        "observations": len(stage_df),
        "mean_interval": time_diff.mean(),
        "minimum_interval": time_diff.min(),
        "maximum_interval": time_diff.max(),
        "non_positive_intervals": np.sum(time_diff <= 0),
        "large_time_gaps": np.sum(time_diff > 1.5)
    })

temporal_integrity = pd.DataFrame(temporal_checks)

temporal_integrity

,operating_hour,observations,mean_interval,minimum_interval,maximum_interval,non_positive_intervals,large_time_gaps
0,50,179360,1.0000,0.9990,1.0010,0,0
1,100,179360,1.0000,0.9990,1.0010,0,0
2,150,179360,1.0000,0.9990,1.0010,0,0
3,200,179360,1.0000,1.0000,1.0000,0,0
4,250,179360,1.0000,1.0000,1.0000,0,0
5,300,179360,1.0000,0.9990,1.0010,0,0
6,350,179360,1.0000,1.0000,1.0000,0,0
7,400,179360,1.0000,1.0000,1.0000,0,0
8,450,179360,1.0000,0.9700,1.0300,0,0
9,500,179360,1.0000,1.0000,1.0000,0,0


In [17]:
# ============================================================
# E1.3.1 Temporal Integrity Summary
# ============================================================

print("Durability stages       :", df["operating_hour"].nunique())
print("Total observations      :", f"{len(df):,}")
print("Non-positive intervals  :", temporal_integrity["non_positive_intervals"].sum())
print("Large time gaps         :", temporal_integrity["large_time_gaps"].sum())

sampling_preserved = np.allclose(
    temporal_integrity["mean_interval"],
    1.0,
    atol=0.001
)

print("~1-second sampling preserved:", sampling_preserved)

Durability stages       : 20
Total observations      : 3,629,680
Non-positive intervals  : 0
Large time gaps         : 0
~1-second sampling preserved: True


In [ ]:
### E1.3.2 Development and Final Test Definition

The durability dataset is divided chronologically into a development region and an untouched final test region.

Durability stages from 50 to 850 h are assigned to model development. These stages may be used for feature-selection analysis, forward validation, model comparison, and hyperparameter optimisation.

The latest durability stages, 900 to 1000 h, are reserved as the final test region.

The final test data are excluded from all feature-selection and model-development decisions so that they provide an independent assessment of generalisation to later, previously unseen durability conditions.

The resulting structure is:

- **Development data:** 50–850 h
- **Final test data:** 900–1000 h

The development region is subsequently divided into expanding chronological train–validation folds.

In [18]:
# ============================================================
# E1.3.2 Define Development and Final Test Regions
# ============================================================

DEVELOPMENT_STAGES = list(range(50, 851, 50))
FINAL_TEST_STAGES = list(range(900, 1001, 50))

development_df = df[
    df["operating_hour"].isin(DEVELOPMENT_STAGES)
].copy()

final_test_df = df[
    df["operating_hour"].isin(FINAL_TEST_STAGES)
].copy()

print("=" * 70)
print("Development and Final Test Definition")
print("=" * 70)

print("Development stages :", DEVELOPMENT_STAGES)
print("Final test stages  :", FINAL_TEST_STAGES)

print("\nDevelopment shape  :", development_df.shape)
print("Final test shape   :", final_test_df.shape)

Development and Final Test Definition
Development stages : [50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700, 750, 800, 850]
Final test stages  : [900, 950, 1000]

Development shape  : (3049120, 24)
Final test shape   : (580560, 24)


In [ ]:
### E1.3.3 Expanding Chronological Validation Folds

Feature-selection and model-development decisions are evaluated using expanding-window chronological validation within the development region.

In each fold, the training data contain only durability stages occurring before the corresponding validation stages. As durability progresses, the training window expands to incorporate additional historical information.

Four folds are defined:

- **Fold 1:** Train 50–450 h → Validate 500–550 h
- **Fold 2:** Train 50–550 h → Validate 600–650 h
- **Fold 3:** Train 50–650 h → Validate 700–750 h
- **Fold 4:** Train 50–750 h → Validate 800–850 h

Each validation block contains two complete durability stages (100 h), providing a consistent forward-looking assessment while preserving the experimental chronology.

This design allows feature-selection stability to be evaluated across progressively later durability conditions rather than relying on a single arbitrary validation boundary.

The final 900–1000 h test region remains completely excluded from these folds.

In [19]:
# ============================================================
# E1.3.3 Define Expanding Chronological Validation Folds
# ============================================================

temporal_folds = {
    "Fold_1": {
        "train_stages": list(range(50, 451, 50)),
        "validation_stages": [500, 550],
    },
    "Fold_2": {
        "train_stages": list(range(50, 551, 50)),
        "validation_stages": [600, 650],
    },
    "Fold_3": {
        "train_stages": list(range(50, 651, 50)),
        "validation_stages": [700, 750],
    },
    "Fold_4": {
        "train_stages": list(range(50, 751, 50)),
        "validation_stages": [800, 850],
    },
}

print("=" * 70)
print("Expanding Chronological Validation Folds")
print("=" * 70)

for fold_name, fold_info in temporal_folds.items():

    print(f"\n{fold_name}")

    print(
        "Training   :",
        f"{min(fold_info['train_stages'])}–"
        f"{max(fold_info['train_stages'])} h"
    )

    print(
        "Validation :",
        f"{min(fold_info['validation_stages'])}–"
        f"{max(fold_info['validation_stages'])} h"
    )

Expanding Chronological Validation Folds

Fold_1
Training   : 50–450 h
Validation : 500–550 h

Fold_2
Training   : 50–550 h
Validation : 600–650 h

Fold_3
Training   : 50–650 h
Validation : 700–750 h

Fold_4
Training   : 50–750 h
Validation : 800–850 h


In [20]:
# ============================================================
# E1.3.4 Verify Fold Chronology and Separation
# ============================================================

fold_integrity_results = []

for fold_name, fold_info in temporal_folds.items():

    train_stages = fold_info["train_stages"]
    validation_stages = fold_info["validation_stages"]

    stage_overlap = (
        set(train_stages)
        & set(validation_stages)
    )

    chronology_valid = (
        max(train_stages)
        < min(validation_stages)
    )

    final_test_overlap = (
        set(train_stages + validation_stages)
        & set(FINAL_TEST_STAGES)
    )

    fold_integrity_results.append(
        {
            "Fold": fold_name,
            "Train_Start": min(train_stages),
            "Train_End": max(train_stages),
            "Validation_Start": min(validation_stages),
            "Validation_End": max(validation_stages),
            "Train_Validation_Overlap": len(stage_overlap),
            "Chronology_Valid": chronology_valid,
            "Final_Test_Overlap": len(final_test_overlap),
        }
    )

fold_integrity_table = pd.DataFrame(
    fold_integrity_results
)

display(fold_integrity_table)

,Fold,Train_Start,Train_End,Validation_Start,Validation_End,Train_Validation_Overlap,Chronology_Valid,Final_Test_Overlap
0,Fold_1,50,450,500,550,0,True,0
1,Fold_2,50,550,600,650,0,True,0
2,Fold_3,50,650,700,750,0,True,0
3,Fold_4,50,750,800,850,0,True,0


In [21]:
# ============================================================
# E1.3.5 Fold Observation Summary
# ============================================================

fold_summary = []

for fold_name, fold_info in temporal_folds.items():

    train_mask = development_df[
        "operating_hour"
    ].isin(
        fold_info["train_stages"]
    )

    validation_mask = development_df[
        "operating_hour"
    ].isin(
        fold_info["validation_stages"]
    )

    train_rows = int(train_mask.sum())
    validation_rows = int(validation_mask.sum())

    fold_summary.append(
        {
            "Fold": fold_name,
            "Training_Stages": (
                f"{min(fold_info['train_stages'])}–"
                f"{max(fold_info['train_stages'])} h"
            ),
            "Validation_Stages": (
                f"{min(fold_info['validation_stages'])}–"
                f"{max(fold_info['validation_stages'])} h"
            ),
            "Training_Observations": train_rows,
            "Validation_Observations": validation_rows,
        }
    )

fold_summary_table = pd.DataFrame(
    fold_summary
)

display(fold_summary_table)

,Fold,Training_Stages,Validation_Stages,Training_Observations,Validation_Observations
0,Fold_1,50–450 h,500–550 h,1614240,358720
1,Fold_2,50–550 h,600–650 h,1972960,358720
2,Fold_3,50–650 h,700–750 h,2331680,358720
3,Fold_4,50–750 h,800–850 h,2690400,358720


In [22]:
# ============================================================
# E1.3.6 Verify Development / Final-Test Isolation
# ============================================================

development_stage_set = set(
    development_df["operating_hour"].unique()
)

final_test_stage_set = set(
    final_test_df["operating_hour"].unique()
)

stage_overlap = (
    development_stage_set
    & final_test_stage_set
)

total_partitioned_rows = (
    len(development_df)
    + len(final_test_df)
)

print("=" * 70)
print("Development / Final-Test Integrity")
print("=" * 70)

print(
    "Development stage range :",
    f"{min(development_stage_set)}–"
    f"{max(development_stage_set)} h"
)

print(
    "Final test stage range  :",
    f"{min(final_test_stage_set)}–"
    f"{max(final_test_stage_set)} h"
)

print(
    "\nStage overlap            :",
    stage_overlap
)

print(
    "Development observations:",
    f"{len(development_df):,}"
)

print(
    "Final test observations :",
    f"{len(final_test_df):,}"
)

print(
    "Total partitioned rows  :",
    f"{total_partitioned_rows:,}"
)

print(
    "Original dataset rows   :",
    f"{len(df):,}"
)

print(
    "\nAll observations preserved:",
    total_partitioned_rows == len(df)
)

Development / Final-Test Integrity
Development stage range : 50–850 h
Final test stage range  : 900–1000 h

Stage overlap            : set()
Development observations: 3,049,120
Final test observations : 580,560
Total partitioned rows  : 3,629,680
Original dataset rows   : 3,629,680

All observations preserved: True


In [23]:
# ============================================================
# E1.3.7 Modelling Split Summary
# ============================================================

split_summary = pd.DataFrame(
    {
        "Partition": [
            "Development",
            "Final Test",
        ],
        "Stage_Range": [
            "50–850 h",
            "900–1000 h",
        ],
        "Number_of_Stages": [
            len(DEVELOPMENT_STAGES),
            len(FINAL_TEST_STAGES),
        ],
        "Observations": [
            len(development_df),
            len(final_test_df),
        ],
    }
)

split_summary["Percentage_of_Data"] = (
    split_summary["Observations"]
    / len(df)
    * 100
)

display(split_summary)

,Partition,Stage_Range,Number_of_Stages,Observations,Percentage_of_Data
0,Development,50–850 h,17,3049120,84.0052
1,Final Test,900–1000 h,3,580560,15.9948


In [24]:
# ============================================================
# E1.3.8 Save Chronological Split Definition
# ============================================================

import json

split_definition = {
    "development_stages": DEVELOPMENT_STAGES,
    "final_test_stages": FINAL_TEST_STAGES,
    "temporal_folds": temporal_folds,
}

split_definition_file = (
    results_dir
    / "chronological_split_definition.json"
)

with open(
    split_definition_file,
    "w"
) as file:

    json.dump(
        split_definition,
        file,
        indent=4
    )

print(
    "Chronological split definition saved to:"
)

print(split_definition_file)

Chronological split definition saved to:
C:\Users\usman\Desktop\PEMFC_Dissertation\results\feature_selection\chronological_split_definition.json


In [ ]:
## E1.4 Predictor Validity and Leakage Screening

Before statistical or model-based feature selection is performed, each variable is assessed for methodological validity as a predictor of stack voltage.

This screening distinguishes between variables that are legitimate prediction inputs and variables that should be excluded because of their role in the dataset or their relationship with the prediction target.

The screening considers:

- target-variable exclusion;
- mathematical target leakage;
- temporal and experimental indexing variables;
- original operational measurements;
- physics-informed engineered predictors.

This stage does not evaluate whether a legitimate predictor is statistically important. Instead, it establishes which variables are eligible to enter the subsequent feature-selection process.

All screening decisions are made using the development data only in terms of model development. The final test region remains excluded from feature-selection decisions.

In [ ]:
### E1.4.1 Target-Variable Screening

Stack voltage is the selected prediction target and is therefore excluded from the predictor matrix.

The modelling problem is defined as:

\[
V = f(X)
\]

where \(V\) represents measured stack voltage and \(X\) contains the legitimate predictor variables.

Including voltage itself among the predictors would constitute direct target leakage and would invalidate predictive evaluation.

In [ ]:
### E1.4.2 Mathematical Target-Leakage Screening

The `power` variable is excluded from the candidate predictor set because electrical power is mathematically related to voltage and current:

\[
P = V \times I
\]

Since voltage is the prediction target, using measured power as an input would provide the model with information directly derived from the target itself.

Although power exhibited a strong statistical association with voltage during association analysis, this relationship is partly mathematical and cannot be treated as independent predictive information.

`power` is therefore classified as a target-leakage variable and removed before feature selection.

In [ ]:
### E1.4.3 Temporal and Durability-Stage Variables

Two variables describe temporal structure in the dataset: `time` and `operating_hour`.

`time` represents the within-block temporal coordinate of each observation. It is required for preserving and verifying time-series structure but is not treated as an ordinary physical predictor of voltage in the present modelling task.

`operating_hour` identifies the durability stage associated with each experimental block. It contains explicit information about accumulated test age and is essential for chronological partitioning, stage-wise analysis, and evaluation of generalisation across durability.

However, `operating_hour` is not included in the primary operational predictor set. Including it would produce an explicitly age-aware model and could allow the model to use durability-stage identity directly rather than learning voltage behaviour from measured operating and system-state variables.

Both variables are therefore retained in the dataset for temporal organisation and evaluation but excluded from the primary candidate predictor matrix.

In [25]:
# ============================================================
# E1.4.4 Establish Eligible Candidate Predictors
# ============================================================

TARGET_VARIABLE = "voltage"

excluded_variables = {
    "voltage": "Prediction target",
    "power": "Mathematical target leakage",
    "time": "Within-block temporal / ordering variable",
    "operating_hour": "Durability-stage / partitioning variable",
}

candidate_predictors = [
    column
    for column in development_df.columns
    if column not in excluded_variables
]

print("=" * 70)
print("Eligible Candidate Predictor Set")
print("=" * 70)

print(f"Dataset variables       : {development_df.shape[1]}")
print(f"Excluded variables      : {len(excluded_variables)}")
print(f"Candidate predictors    : {len(candidate_predictors)}")

print("\nExcluded variables:")
for variable, reason in excluded_variables.items():
    print(f"- {variable}: {reason}")

print("\nEligible candidate predictors:")
for i, variable in enumerate(candidate_predictors, start=1):
    print(f"{i:2d}. {variable}")

Eligible Candidate Predictor Set
Dataset variables       : 24
Excluded variables      : 4
Candidate predictors    : 20

Excluded variables:
- voltage: Prediction target
- power: Mathematical target leakage
- time: Within-block temporal / ordering variable
- operating_hour: Durability-stage / partitioning variable

Eligible candidate predictors:
 1. current
 2. pressure_anode_inlet
 3. pressure_anode_outlet
 4. pressure_cathode_inlet
 5. pressure_cathode_outlet
 6. temp_anode_endplate
 7. temp_anode_dewpoint_water
 8. temp_anode_inlet
 9. temp_anode_outlet
10. temp_cathode_dewpoint_water
11. temp_cathode_inlet
12. temp_cathode_outlet
13. total_anode_stack_flow
14. total_cathode_stack_flow
15. anode_pressure_diff
16. cathode_pressure_diff
17. anode_temp_diff
18. cathode_temp_diff
19. anode_dewpoint_offset
20. cathode_dewpoint_offset


In [26]:
# ============================================================
# E1.4.5 Classify Eligible Candidate Predictors
# ============================================================

engineered_features = [
    "anode_pressure_diff",
    "cathode_pressure_diff",
    "anode_temp_diff",
    "cathode_temp_diff",
    "anode_dewpoint_offset",
    "cathode_dewpoint_offset",
]

predictor_classification = pd.DataFrame({
    "Predictor": candidate_predictors
})

predictor_classification["Feature_Type"] = (
    predictor_classification["Predictor"].apply(
        lambda x:
        "Engineered predictor"
        if x in engineered_features
        else "Original operational predictor"
    )
)

display(predictor_classification)

,Predictor,Feature_Type
0,current,Original operational predictor
1,pressure_anode_inlet,Original operational predictor
2,pressure_anode_outlet,Original operational predictor
3,pressure_cathode_inlet,Original operational predictor
4,pressure_cathode_outlet,Original operational predictor
5,temp_anode_endplate,Original operational predictor
6,temp_anode_dewpoint_water,Original operational predictor
7,temp_anode_inlet,Original operational predictor
8,temp_anode_outlet,Original operational predictor
9,temp_cathode_dewpoint_water,Original operational predictor


In [27]:
# ============================================================
# E1.4.6 Predictor Validity Screening Table
# ============================================================

screening_records = []

for variable in development_df.columns:

    if variable == "voltage":
        role = "Target"
        decision = "Exclude from X"
        reason = "Prediction target"

    elif variable == "power":
        role = "Target-leakage variable"
        decision = "Exclude"
        reason = "Power is mathematically derived from voltage and current"

    elif variable == "time":
        role = "Temporal variable"
        decision = "Exclude from primary X"
        reason = "Used for within-block temporal ordering"

    elif variable == "operating_hour":
        role = "Durability-stage variable"
        decision = "Exclude from primary X"
        reason = (
            "Used for chronological partitioning and "
            "durability-stage evaluation"
        )

    elif variable in engineered_features:
        role = "Engineered predictor"
        decision = "Retain for feature selection"
        reason = "Methodologically eligible predictor"

    else:
        role = "Original operational predictor"
        decision = "Retain for feature selection"
        reason = "Methodologically eligible predictor"

    screening_records.append({
        "Variable": variable,
        "Role": role,
        "Screening_Decision": decision,
        "Reason": reason,
    })

predictor_screening_table = pd.DataFrame(
    screening_records
)

display(predictor_screening_table)

,Variable,Role,Screening_Decision,Reason
0,operating_hour,Durability-stage variable,Exclude from primary X,Used for chronological partitioning and durabi...
1,time,Temporal variable,Exclude from primary X,Used for within-block temporal ordering
2,current,Original operational predictor,Retain for feature selection,Methodologically eligible predictor
3,voltage,Target,Exclude from X,Prediction target
4,power,Target-leakage variable,Exclude,Power is mathematically derived from voltage a...
5,pressure_anode_inlet,Original operational predictor,Retain for feature selection,Methodologically eligible predictor
6,pressure_anode_outlet,Original operational predictor,Retain for feature selection,Methodologically eligible predictor
7,pressure_cathode_inlet,Original operational predictor,Retain for feature selection,Methodologically eligible predictor
8,pressure_cathode_outlet,Original operational predictor,Retain for feature selection,Methodologically eligible predictor
9,temp_anode_endplate,Original operational predictor,Retain for feature selection,Methodologically eligible predictor


In [28]:
# ============================================================
# E1.4.7 Verify Candidate Predictor Integrity
# ============================================================

expected_candidate_count = 20

checks = {
    "Target excluded": (
        TARGET_VARIABLE not in candidate_predictors
    ),
    "Power excluded": (
        "power" not in candidate_predictors
    ),
    "Time excluded": (
        "time" not in candidate_predictors
    ),
    "Operating hour excluded": (
        "operating_hour" not in candidate_predictors
    ),
    "All engineered features retained": (
        all(
            feature in candidate_predictors
            for feature in engineered_features
        )
    ),
    "Expected predictor count": (
        len(candidate_predictors)
        == expected_candidate_count
    ),
}

predictor_integrity_table = pd.DataFrame(
    {
        "Check": checks.keys(),
        "Passed": checks.values(),
    }
)

display(predictor_integrity_table)

,Check,Passed
0,Target excluded,True
1,Power excluded,True
2,Time excluded,True
3,Operating hour excluded,True
4,All engineered features retained,True
5,Expected predictor count,True


In [ ]:
### E1.4.8 Development Predictor and Target Definition

Following validity and leakage screening, the development predictor matrix contains only methodologically eligible operational and engineered variables.

The target vector contains measured stack voltage.

The final test predictor and target objects are not used during feature-selection analysis. They are retained separately for eventual final model evaluation.

In [29]:
# ============================================================
# E1.4.8 Create Development X and y
# ============================================================

X_development = development_df[
    candidate_predictors
].copy()

y_development = development_df[
    TARGET_VARIABLE
].copy()

print("=" * 70)
print("Development Predictor / Target Definition")
print("=" * 70)

print("X development shape :", X_development.shape)
print("y development shape :", y_development.shape)

print("\nNumber of predictors:", X_development.shape[1])

print(
    "Missing predictor values:",
    int(X_development.isna().sum().sum())
)

print(
    "Missing target values   :",
    int(y_development.isna().sum())
)

Development Predictor / Target Definition
X development shape : (3049120, 20)
y development shape : (3049120,)

Number of predictors: 20
Missing predictor values: 0
Missing target values   : 0


In [ ]:
## E1.5 Filter-Based Target Relevance

Filter-based feature-selection methods are used to evaluate the statistical relevance of each eligible candidate predictor to the voltage target independently of the final predictive models.

Three complementary measures are considered:

- Pearson correlation for linear association;
- Spearman rank correlation for monotonic association;
- Mutual Information for more general statistical dependence.

These measures capture different forms of predictor–target relationship and are therefore interpreted jointly rather than used as independent automatic selection rules.

All target-relevance calculations are performed using development data only. The final test region remains excluded from feature-selection decisions.

In [ ]:
### E1.5.1 Pearson Correlation

Pearson correlation is used to quantify the strength and direction of the linear association between each eligible candidate predictor and stack voltage.

For predictor \(X\) and voltage \(V\), the Pearson correlation coefficient is:

\[
r_{X,V} =
\frac{\operatorname{cov}(X,V)}
{\sigma_X \sigma_V}
\]

where \(r\) ranges from -1 to +1.

- \(r>0\) indicates a positive linear association.
- \(r<0\) indicates a negative linear association.
- Values of \(|r|\) closer to 1 indicate stronger linear association.
- Values of \(|r|\) closer to 0 indicate weaker linear association.

Because the dataset contains more than three million development observations, statistical significance alone may identify very small effects as statistically significant. Therefore, both the correlation coefficient and p-value are reported, but predictor relevance is interpreted primarily from association magnitude, physical meaning, and subsequent complementary feature-selection evidence.

Pearson correlation is not used as an automatic retain/remove criterion. A weak Pearson correlation does not imply that a predictor lacks nonlinear or monotonic predictive information.

In [30]:
# ============================================================
# E1.5.1 Calculate Predictor–Voltage Pearson Correlations
# ============================================================

from scipy.stats import pearsonr

pearson_results = []

for predictor in candidate_predictors:

    r_value, p_value = pearsonr(
        X_development[predictor],
        y_development
    )

    pearson_results.append(
        {
            "Predictor": predictor,
            "Pearson_r": r_value,
            "Absolute_r": abs(r_value),
            "p_value": p_value,
        }
    )

pearson_target_table = pd.DataFrame(
    pearson_results
)

pearson_target_table = (
    pearson_target_table
    .sort_values(
        by="Absolute_r",
        ascending=False
    )
    .reset_index(drop=True)
)

print("=" * 70)
print("Pearson Association of Candidate Predictors with Voltage")
print("=" * 70)

display(
    pearson_target_table.round(
        {
            "Pearson_r": 4,
            "Absolute_r": 4,
            "p_value": 6,
        }
    )
)

Pearson Association of Candidate Predictors with Voltage


,Predictor,Pearson_r,Absolute_r,p_value
0,current,-0.9681,0.9681,0.0000
1,total_cathode_stack_flow,-0.9200,0.9200,0.0000
2,total_anode_stack_flow,-0.9197,0.9197,0.0000
3,cathode_pressure_diff,-0.8539,0.8539,0.0000
4,temp_cathode_inlet,-0.7389,0.7389,0.0000
5,cathode_dewpoint_offset,-0.7161,0.7161,0.0000
6,pressure_cathode_inlet,-0.6716,0.6716,0.0000
7,anode_pressure_diff,-0.6263,0.6263,0.0000
8,temp_anode_endplate,-0.3943,0.3943,0.0000
9,temp_anode_inlet,-0.3906,0.3906,0.0000


In [31]:
# ============================================================
# E1.5.1 Classify Pearson Association Strength
# ============================================================

def classify_association_strength(value):

    absolute_value = abs(value)

    if absolute_value < 0.10:
        return "Very weak"

    elif absolute_value < 0.30:
        return "Weak"

    elif absolute_value < 0.50:
        return "Moderate"

    elif absolute_value < 0.70:
        return "Strong"

    else:
        return "Very strong"


pearson_target_table["Association_Strength"] = (
    pearson_target_table["Pearson_r"]
    .apply(classify_association_strength)
)

pearson_target_table["Direction"] = np.where(
    pearson_target_table["Pearson_r"] >= 0,
    "Positive",
    "Negative"
)

display(
    pearson_target_table[
        [
            "Predictor",
            "Pearson_r",
            "Absolute_r",
            "Association_Strength",
            "Direction",
            "p_value",
        ]
    ].round(
        {
            "Pearson_r": 4,
            "Absolute_r": 4,
            "p_value": 6,
        }
    )
)

,Predictor,Pearson_r,Absolute_r,Association_Strength,Direction,p_value
0,current,-0.9681,0.9681,Very strong,Negative,0.0000
1,total_cathode_stack_flow,-0.9200,0.9200,Very strong,Negative,0.0000
2,total_anode_stack_flow,-0.9197,0.9197,Very strong,Negative,0.0000
3,cathode_pressure_diff,-0.8539,0.8539,Very strong,Negative,0.0000
4,temp_cathode_inlet,-0.7389,0.7389,Very strong,Negative,0.0000
5,cathode_dewpoint_offset,-0.7161,0.7161,Very strong,Negative,0.0000
6,pressure_cathode_inlet,-0.6716,0.6716,Strong,Negative,0.0000
7,anode_pressure_diff,-0.6263,0.6263,Strong,Negative,0.0000
8,temp_anode_endplate,-0.3943,0.3943,Moderate,Negative,0.0000
9,temp_anode_inlet,-0.3906,0.3906,Moderate,Negative,0.0000


In [32]:
# ============================================================
# E1.5.1 Visualise Pearson Target Relevance
# ============================================================

pearson_plot_df = (
    pearson_target_table
    .sort_values(
        by="Pearson_r",
        ascending=True
    )
)

# Assign colours based on correlation direction
bar_colors = [
    "red" if value < 0 else "blue"
    for value in pearson_plot_df["Pearson_r"]
]

plt.figure(figsize=(10, 9))

bars = plt.barh(
    pearson_plot_df["Predictor"],
    pearson_plot_df["Pearson_r"],
    color=bar_colors
)

# Zero-reference line
plt.axvline(
    x=0,
    color="black",
    linewidth=1
)

# Add Pearson correlation values at the end of each bar
for bar, value in zip(
    bars,
    pearson_plot_df["Pearson_r"]
):
    y_position = bar.get_y() + bar.get_height() / 2

    if value < 0:
        plt.text(
            value - 0.02,
            y_position,
            f"{value:.3f}",
            va="center",
            ha="right",
            fontsize=9
        )
    else:
        plt.text(
            value + 0.02,
            y_position,
            f"{value:.3f}",
            va="center",
            ha="left",
            fontsize=9
        )

plt.xlim(-1.1, 1.1)

plt.xlabel("Pearson Correlation with Voltage (r)")
plt.ylabel("Candidate Predictor")

plt.title(
    "Linear Association of Candidate Predictors with Voltage"
)

plt.grid(
    axis="x",
    alpha=0.3
)

plt.tight_layout()

plt.show()

<Figure size 1000x900 with 1 Axes>

In [33]:
# ============================================================
# E1.5.1 Pearson Association Summary
# ============================================================

pearson_strength_summary = (
    pearson_target_table[
        "Association_Strength"
    ]
    .value_counts()
    .reindex(
        [
            "Very strong",
            "Strong",
            "Moderate",
            "Weak",
            "Very weak",
        ],
        fill_value=0
    )
    .rename_axis("Association_Strength")
    .reset_index(name="Number_of_Predictors")
)

display(pearson_strength_summary)

,Association_Strength,Number_of_Predictors
0,Very strong,6
1,Strong,2
2,Moderate,3
3,Weak,3
4,Very weak,6


In [ ]:
### E1.5.2 Spearman Rank Correlation

Spearman rank correlation is used to quantify the strength and direction of the monotonic association between each eligible candidate predictor and stack voltage.

Unlike Pearson correlation, Spearman correlation does not require the predictor–target relationship to be linear. Instead, it evaluates whether voltage tends to increase or decrease consistently as the predictor increases.

The Spearman rank correlation coefficient, denoted by \(\rho\), ranges from -1 to +1:

- \(\rho > 0\) indicates a positive monotonic association.
- \(\rho < 0\) indicates a negative monotonic association.
- Values of \(|\rho|\) closer to 1 indicate stronger monotonic association.
- Values of \(|\rho|\) closer to 0 indicate weaker monotonic association.

Spearman correlation complements Pearson correlation by identifying relationships that may be monotonic but not adequately represented by a linear correlation coefficient.

As with Pearson correlation, statistical significance is interpreted cautiously because the very large number of development observations can produce very small p-values even for weak effects.

Spearman correlation is therefore treated as complementary feature-selection evidence rather than an automatic retain/remove criterion.

All calculations are performed using development data only, while the final test region remains excluded.

In [34]:
# ============================================================
# E1.5.2 Calculate Predictor–Voltage Spearman Correlations
# ============================================================

from scipy.stats import spearmanr

spearman_results = []

for predictor in candidate_predictors:

    rho_value, p_value = spearmanr(
        X_development[predictor],
        y_development
    )

    spearman_results.append(
        {
            "Predictor": predictor,
            "Spearman_rho": rho_value,
            "Absolute_rho": abs(rho_value),
            "p_value": p_value,
        }
    )

spearman_target_table = pd.DataFrame(
    spearman_results
)

spearman_target_table = (
    spearman_target_table
    .sort_values(
        by="Absolute_rho",
        ascending=False
    )
    .reset_index(drop=True)
)

print("=" * 70)
print("Spearman Association of Candidate Predictors with Voltage")
print("=" * 70)

display(
    spearman_target_table.round(
        {
            "Spearman_rho": 4,
            "Absolute_rho": 4,
            "p_value": 6,
        }
    )
)

Spearman Association of Candidate Predictors with Voltage


,Predictor,Spearman_rho,Absolute_rho,p_value
0,total_anode_stack_flow,-0.9298,0.9298,0.0000
1,total_cathode_stack_flow,-0.9293,0.9293,0.0000
2,current,-0.9271,0.9271,0.0000
3,cathode_pressure_diff,-0.8190,0.8190,0.0000
4,pressure_cathode_inlet,-0.7621,0.7621,0.0000
5,temp_cathode_inlet,-0.6561,0.6561,0.0000
6,cathode_dewpoint_offset,-0.6398,0.6398,0.0000
7,anode_pressure_diff,-0.5539,0.5539,0.0000
8,temp_anode_endplate,-0.4408,0.4408,0.0000
9,pressure_anode_inlet,-0.4046,0.4046,0.0000


In [35]:
# ============================================================
# E1.5.2 Classify Spearman Association Strength
# ============================================================

spearman_target_table["Association_Strength"] = (
    spearman_target_table["Spearman_rho"]
    .apply(classify_association_strength)
)

spearman_target_table["Direction"] = np.where(
    spearman_target_table["Spearman_rho"] >= 0,
    "Positive",
    "Negative"
)

display(
    spearman_target_table[
        [
            "Predictor",
            "Spearman_rho",
            "Absolute_rho",
            "Association_Strength",
            "Direction",
            "p_value",
        ]
    ].round(
        {
            "Spearman_rho": 4,
            "Absolute_rho": 4,
            "p_value": 6,
        }
    )
)

,Predictor,Spearman_rho,Absolute_rho,Association_Strength,Direction,p_value
0,total_anode_stack_flow,-0.9298,0.9298,Very strong,Negative,0.0000
1,total_cathode_stack_flow,-0.9293,0.9293,Very strong,Negative,0.0000
2,current,-0.9271,0.9271,Very strong,Negative,0.0000
3,cathode_pressure_diff,-0.8190,0.8190,Very strong,Negative,0.0000
4,pressure_cathode_inlet,-0.7621,0.7621,Very strong,Negative,0.0000
5,temp_cathode_inlet,-0.6561,0.6561,Strong,Negative,0.0000
6,cathode_dewpoint_offset,-0.6398,0.6398,Strong,Negative,0.0000
7,anode_pressure_diff,-0.5539,0.5539,Strong,Negative,0.0000
8,temp_anode_endplate,-0.4408,0.4408,Moderate,Negative,0.0000
9,pressure_anode_inlet,-0.4046,0.4046,Moderate,Negative,0.0000


In [36]:
# ============================================================
# E1.5.2 Visualise Spearman Target Relevance
# ============================================================

spearman_plot_df = (
    spearman_target_table
    .sort_values(
        by="Spearman_rho",
        ascending=True
    )
)

# Assign colours based on correlation direction
bar_colors = [
    "red" if value < 0 else "blue"
    for value in spearman_plot_df["Spearman_rho"]
]

plt.figure(figsize=(10, 9))

bars = plt.barh(
    spearman_plot_df["Predictor"],
    spearman_plot_df["Spearman_rho"],
    color=bar_colors
)

# Zero-reference line
plt.axvline(
    x=0,
    color="black",
    linewidth=1
)

# Add Spearman coefficients at the end of each bar
for bar, value in zip(
    bars,
    spearman_plot_df["Spearman_rho"]
):
    y_position = (
        bar.get_y()
        + bar.get_height() / 2
    )

    if value < 0:
        plt.text(
            value - 0.02,
            y_position,
            f"{value:.3f}",
            va="center",
            ha="right",
            fontsize=9
        )

    else:
        plt.text(
            value + 0.02,
            y_position,
            f"{value:.3f}",
            va="center",
            ha="left",
            fontsize=9
        )

plt.xlim(-1.1, 1.1)

plt.xlabel(
    "Spearman Correlation with Voltage (ρ)"
)

plt.ylabel(
    "Candidate Predictor"
)

plt.title(
    "Monotonic Association of Candidate Predictors with Voltage"
)

plt.grid(
    axis="x",
    alpha=0.3
)

plt.tight_layout()

plt.show()

<Figure size 1000x900 with 1 Axes>

In [37]:
# ============================================================
# E1.5.2 Spearman Association Summary
# ============================================================

spearman_strength_summary = (
    spearman_target_table[
        "Association_Strength"
    ]
    .value_counts()
    .reindex(
        [
            "Very strong",
            "Strong",
            "Moderate",
            "Weak",
            "Very weak",
        ],
        fill_value=0
    )
    .rename_axis(
        "Association_Strength"
    )
    .reset_index(
        name="Number_of_Predictors"
    )
)

display(
    spearman_strength_summary
)

,Association_Strength,Number_of_Predictors
0,Very strong,5
1,Strong,3
2,Moderate,4
3,Weak,3
4,Very weak,5


In [38]:
# ============================================================
# E1.5.2 Save Spearman Target-Relevance Results
# ============================================================

spearman_output_file = (
    results_dir
    / "spearman_target_relevance.csv"
)

spearman_target_table.to_csv(
    spearman_output_file,
    index=False
)

print(
    "Spearman target-relevance results saved successfully."
)

print(
    "Saved to:",
    spearman_output_file
)

Spearman target-relevance results saved successfully.
Saved to: C:\Users\usman\Desktop\PEMFC_Dissertation\results\feature_selection\spearman_target_relevance.csv


In [ ]:
### E1.5.3 Mutual Information

Mutual Information (MI) is used to evaluate more general statistical dependence between each eligible candidate predictor and stack voltage.

Unlike Pearson and Spearman correlation, Mutual Information does not require the predictor–target relationship to be linear or monotonic. It can therefore identify dependencies that may not be adequately represented by conventional correlation coefficients.

For predictor \(X\) and target \(V\), Mutual Information measures the reduction in uncertainty about \(V\) obtained from knowledge of \(X\).

Conceptually:

\[
MI(X;V) = 0
\]

indicates that no statistical dependence has been detected, while larger positive values indicate greater statistical dependence.

Mutual Information differs from Pearson and Spearman correlation in several important respects:

- MI is non-negative.
- MI does not indicate the direction of a relationship.
- MI values are not bounded between -1 and +1.
- MI values should not be interpreted using Pearson or Spearman correlation-strength thresholds.
- MI is primarily interpreted comparatively across predictors within the same analysis.

To maintain consistency with the preceding Pearson and Spearman analyses, Mutual Information is calculated using all observations in the development region.

The final test region remains excluded from feature-selection analysis.

In [39]:
# ============================================================
# E1.5.3 Prepare Full Development Dataset for Mutual Information
# ============================================================

X_mi = X_development.copy()
y_mi = y_development.copy()

print("=" * 70)
print("Mutual Information Analysis Dataset")
print("=" * 70)

print("X MI shape :", X_mi.shape)
print("y MI shape :", y_mi.shape)

print(
    "\nMissing predictor values:",
    int(X_mi.isna().sum().sum())
)

print(
    "Missing target values   :",
    int(y_mi.isna().sum())
)

Mutual Information Analysis Dataset
X MI shape : (3049120, 20)
y MI shape : (3049120,)

Missing predictor values: 0
Missing target values   : 0


In [40]:
# ============================================================
# E1.5.3 Calculate Mutual Information with Voltage
# ============================================================

from sklearn.feature_selection import mutual_info_regression

MI_RANDOM_STATE = 42

mi_values = mutual_info_regression(
    X_mi,
    y_mi,
    discrete_features=False,
    random_state=MI_RANDOM_STATE
)

mi_target_table = pd.DataFrame(
    {
        "Predictor": candidate_predictors,
        "Mutual_Information": mi_values,
    }
)

mi_target_table = (
    mi_target_table
    .sort_values(
        by="Mutual_Information",
        ascending=False
    )
    .reset_index(drop=True)
)

print("=" * 70)
print("Mutual Information of Candidate Predictors with Voltage")
print("=" * 70)

display(
    mi_target_table.round(4)
)

Mutual Information of Candidate Predictors with Voltage


,Predictor,Mutual_Information
0,current,1.5338
1,total_cathode_stack_flow,1.4475
2,total_anode_stack_flow,1.4360
3,cathode_pressure_diff,1.4327
4,temp_anode_endplate,0.9806
5,pressure_cathode_inlet,0.8340
6,temp_cathode_inlet,0.7633
7,pressure_cathode_outlet,0.6991
8,cathode_dewpoint_offset,0.6157
9,anode_pressure_diff,0.5952


In [41]:
# ============================================================
# E1.5.3 Rank Predictors by Mutual Information
# ============================================================

mi_target_table["MI_Rank"] = (
    mi_target_table[
        "Mutual_Information"
    ]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)

mi_target_table["Relative_MI"] = (
    mi_target_table["Mutual_Information"]
    / mi_target_table["Mutual_Information"].max()
)

display(
    mi_target_table[
        [
            "Predictor",
            "Mutual_Information",
            "Relative_MI",
            "MI_Rank",
        ]
    ].round(4)
)

,Predictor,Mutual_Information,Relative_MI,MI_Rank
0,current,1.5338,1.0000,1
1,total_cathode_stack_flow,1.4475,0.9437,2
2,total_anode_stack_flow,1.4360,0.9362,3
3,cathode_pressure_diff,1.4327,0.9341,4
4,temp_anode_endplate,0.9806,0.6393,5
5,pressure_cathode_inlet,0.8340,0.5437,6
6,temp_cathode_inlet,0.7633,0.4976,7
7,pressure_cathode_outlet,0.6991,0.4558,8
8,cathode_dewpoint_offset,0.6157,0.4014,9
9,anode_pressure_diff,0.5952,0.3880,10


In [42]:
# ============================================================
# E1.5.3 Visualise Mutual Information Target Relevance
# ============================================================

mi_plot_df = (
    mi_target_table
    .sort_values(
        by="Mutual_Information",
        ascending=True
    )
)

plt.figure(figsize=(10, 9))

bars = plt.barh(
    mi_plot_df["Predictor"],
    mi_plot_df["Mutual_Information"]
)

for bar, value in zip(
    bars,
    mi_plot_df["Mutual_Information"]
):
    y_position = (
        bar.get_y()
        + bar.get_height() / 2
    )

    plt.text(
        value,
        y_position,
        f" {value:.3f}",
        va="center",
        ha="left",
        fontsize=9
    )

plt.xlabel(
    "Mutual Information with Voltage"
)

plt.ylabel(
    "Candidate Predictor"
)

plt.title(
    "General Statistical Dependence of Candidate Predictors with Voltage"
)

plt.grid(
    axis="x",
    alpha=0.3
)

plt.tight_layout()

plt.show()

<Figure size 1000x900 with 1 Axes>

In [43]:
# ============================================================
# E1.5.3 Save Mutual Information Results
# ============================================================

mi_output_file = (
    results_dir
    / "mutual_information_target_relevance.csv"
)

mi_target_table.to_csv(
    mi_output_file,
    index=False
)

print(
    "Mutual Information target-relevance results saved successfully."
)

print(
    "Saved to:",
    mi_output_file
)

Mutual Information target-relevance results saved successfully.
Saved to: C:\Users\usman\Desktop\PEMFC_Dissertation\results\feature_selection\mutual_information_target_relevance.csv


In [ ]:
### E1.5.4 Integrated Target-Relevance Assessment

The preceding analyses evaluated predictor relevance to stack voltage from three complementary statistical perspectives:

- **Pearson correlation** quantified linear association.
- **Spearman correlation** quantified monotonic association.
- **Mutual Information** quantified more general statistical dependence, including potentially nonlinear and non-monotonic relationships.

No single measure provides a complete assessment of predictor relevance. Therefore, the three measures are integrated into a common comparison table.

Agreement across the methods provides stronger evidence of marginal target relevance, while disagreement can reveal predictors whose relationships with voltage are nonlinear or otherwise inadequately represented by conventional correlation coefficients.

At this stage, the integrated assessment is used for evidence synthesis rather than automatic feature elimination. Predictor redundancy, multicollinearity, embedded selection, and nonlinear model-based selection are examined in subsequent sections before final feature-selection decisions are made.

All results incorporated in this assessment were obtained exclusively from the development region; the final test region remains excluded.

In [44]:
# ============================================================
# E1.5.4 Integrate Target-Relevance Evidence
# ============================================================

integrated_target_relevance = (
    pearson_target_table[
        [
            "Predictor",
            "Pearson_r",
            "Absolute_r"
        ]
    ]
    .merge(
        spearman_target_table[
            [
                "Predictor",
                "Spearman_rho",
                "Absolute_rho"
            ]
        ],
        on="Predictor",
        how="inner"
    )
    .merge(
        mi_target_table[
            [
                "Predictor",
                "Mutual_Information",
                "Relative_MI",
                "MI_Rank"
            ]
        ],
        on="Predictor",
        how="inner"
    )
)

print("=" * 80)
print("Integrated Predictor–Voltage Target-Relevance Evidence")
print("=" * 80)

display(
    integrated_target_relevance.round(4)
)

Integrated Predictor–Voltage Target-Relevance Evidence


,Predictor,Pearson_r,Absolute_r,Spearman_rho,Absolute_rho,Mutual_Information,Relative_MI,MI_Rank
0,current,-0.9681,0.9681,-0.9271,0.9271,1.5338,1.0000,1
1,total_cathode_stack_flow,-0.9200,0.9200,-0.9293,0.9293,1.4475,0.9437,2
2,total_anode_stack_flow,-0.9197,0.9197,-0.9298,0.9298,1.4360,0.9362,3
3,cathode_pressure_diff,-0.8539,0.8539,-0.8190,0.8190,1.4327,0.9341,4
4,temp_cathode_inlet,-0.7389,0.7389,-0.6561,0.6561,0.7633,0.4976,7
5,cathode_dewpoint_offset,-0.7161,0.7161,-0.6398,0.6398,0.6157,0.4014,9
6,pressure_cathode_inlet,-0.6716,0.6716,-0.7621,0.7621,0.8340,0.5437,6
7,anode_pressure_diff,-0.6263,0.6263,-0.5539,0.5539,0.5952,0.3880,10
8,temp_anode_endplate,-0.3943,0.3943,-0.4408,0.4408,0.9806,0.6393,5
9,temp_anode_inlet,-0.3906,0.3906,-0.3735,0.3735,0.5394,0.3517,11


In [45]:
# ============================================================
# E1.5.4 Integrated Evidence Integrity Check
# ============================================================

expected_predictor_count = len(candidate_predictors)

actual_predictor_count = (
    integrated_target_relevance["Predictor"]
    .nunique()
)

print(
    "Expected predictors :",
    expected_predictor_count
)

print(
    "Integrated predictors:",
    actual_predictor_count
)

print(
    "All candidate predictors retained:",
    actual_predictor_count == expected_predictor_count
)

print(
    "Duplicate predictors:",
    integrated_target_relevance["Predictor"].duplicated().sum()
)

print(
    "Missing values:",
    integrated_target_relevance.isna().sum().sum()
)

Expected predictors : 20
Integrated predictors: 20
All candidate predictors retained: True
Duplicate predictors: 0
Missing values: 0


In [46]:
# ============================================================
# E1.5.4 Calculate Target-Relevance Rankings
# ============================================================

integrated_target_relevance["Pearson_Rank"] = (
    integrated_target_relevance["Absolute_r"]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)

integrated_target_relevance["Spearman_Rank"] = (
    integrated_target_relevance["Absolute_rho"]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)

integrated_target_relevance["Mean_Relevance_Rank"] = (
    integrated_target_relevance[
        [
            "Pearson_Rank",
            "Spearman_Rank",
            "MI_Rank"
        ]
    ]
    .mean(axis=1)
)

integrated_target_relevance = (
    integrated_target_relevance
    .sort_values(
        "Mean_Relevance_Rank"
    )
    .reset_index(drop=True)
)

display(
    integrated_target_relevance[
        [
            "Predictor",
            "Pearson_r",
            "Pearson_Rank",
            "Spearman_rho",
            "Spearman_Rank",
            "Mutual_Information",
            "MI_Rank",
            "Mean_Relevance_Rank"
        ]
    ].round(4)
)

,Predictor,Pearson_r,Pearson_Rank,Spearman_rho,Spearman_Rank,Mutual_Information,MI_Rank,Mean_Relevance_Rank
0,current,-0.9681,1,-0.9271,3,1.5338,1,1.6667
1,total_cathode_stack_flow,-0.9200,2,-0.9293,2,1.4475,2,2.0000
2,total_anode_stack_flow,-0.9197,3,-0.9298,1,1.4360,3,2.3333
3,cathode_pressure_diff,-0.8539,4,-0.8190,4,1.4327,4,4.0000
4,temp_cathode_inlet,-0.7389,5,-0.6561,6,0.7633,7,6.0000
5,pressure_cathode_inlet,-0.6716,7,-0.7621,5,0.8340,6,6.0000
6,cathode_dewpoint_offset,-0.7161,6,-0.6398,7,0.6157,9,7.3333
7,temp_anode_endplate,-0.3943,9,-0.4408,9,0.9806,5,7.6667
8,anode_pressure_diff,-0.6263,8,-0.5539,8,0.5952,10,8.6667
9,temp_anode_inlet,-0.3906,10,-0.3735,11,0.5394,11,10.6667


In [47]:
# ============================================================
# E1.5.4 Visualise Integrated Target-Relevance Rankings
# ============================================================

ranking_plot_df = (
    integrated_target_relevance[
        [
            "Predictor",
            "Pearson_Rank",
            "Spearman_Rank",
            "MI_Rank"
        ]
    ]
    .set_index("Predictor")
)

# ------------------------------------------------------------
# Convert ranks to upward bar heights
# ------------------------------------------------------------
# This transformation is ONLY for visualisation:
#
# Rank 1  -> tallest bar
# Rank 20 -> shortest bar
#
# The original rank values are preserved and displayed
# outside the bars.

max_rank = len(candidate_predictors)

ranking_plot_upward = (
    max_rank + 1 - ranking_plot_df
)

# ------------------------------------------------------------
# Create plot
# ------------------------------------------------------------

ax = ranking_plot_upward.plot(
    kind="bar",
    figsize=(14, 7)
)

# ------------------------------------------------------------
# Add ACTUAL rank magnitude outside each bar
# ------------------------------------------------------------

for container, column in zip(
    ax.containers,
    ranking_plot_df.columns
):
    original_ranks = ranking_plot_df[column].values

    for bar, rank in zip(
        container,
        original_ranks
    ):
        bar_height = bar.get_height()

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar_height + 0.25,
            f"{int(rank)}",
            ha="center",
            va="bottom",
            fontsize=8
        )

# ------------------------------------------------------------
# Preserve original rank values on Y-axis
# ------------------------------------------------------------

rank_ticks = list(
    range(1, max_rank + 1)
)

tick_positions = [
    max_rank + 1 - rank
    for rank in rank_ticks
]

ax.set_yticks(
    tick_positions
)

ax.set_yticklabels(
    rank_ticks
)

# Give labels above tallest bars enough space
ax.set_ylim(
    0,
    max_rank + 1.5
)

# ------------------------------------------------------------
# Labels and formatting
# ------------------------------------------------------------

plt.ylabel(
    "Target-Relevance Rank"
)

plt.xlabel(
    "Candidate Predictor"
)

plt.title(
    "Comparison of Predictor–Voltage Relevance Rankings"
)

plt.xticks(
    rotation=70,
    ha="right"
)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.legend(
    title="Method"
)

plt.tight_layout()

plt.show()

<Figure size 1400x700 with 1 Axes>

In [48]:
# ============================================================
# E1.5.4 Save Integrated Target-Relevance Evidence
# ============================================================

integrated_output_file = (
    results_dir
    / "integrated_target_relevance.csv"
)

integrated_target_relevance.to_csv(
    integrated_output_file,
    index=False
)

print(
    "Integrated target-relevance results saved successfully."
)

print(
    "Saved to:",
    integrated_output_file
)

Integrated target-relevance results saved successfully.
Saved to: C:\Users\usman\Desktop\PEMFC_Dissertation\results\feature_selection\integrated_target_relevance.csv


In [ ]:
## E1.6 Predictor Redundancy Assessment

Target-relevance analysis evaluates whether individual predictors contain information about stack voltage. However, highly relevant predictors may also contain substantially overlapping information.

Predictor redundancy is therefore assessed by examining associations among the candidate predictors themselves.

High pairwise association may indicate that two predictors represent closely related operating behaviour or system states. Retaining strongly redundant predictors can increase dimensionality, complicate interpretation, and contribute to multicollinearity without necessarily providing substantial additional predictive information.

Redundancy assessment is conducted using both:

- Pearson correlation, to identify strong linear relationships between predictors;
- Spearman rank correlation, to identify strong monotonic relationships between predictors.

The analysis is performed using development data only. The final test region remains excluded.

Pairwise redundancy is treated as diagnostic evidence rather than an automatic feature-removal rule. Decisions concerning redundant predictors will subsequently consider target relevance, physical interpretation, multicollinearity, and model-based feature-selection evidence.

In [ ]:
### E1.6.1 Pairwise Pearson Redundancy

Pairwise Pearson correlation is used to assess linear redundancy among the candidate predictors.

For each pair of predictors, the Pearson correlation coefficient quantifies the strength and direction of their linear association.

Large absolute correlation values indicate that two predictors may contain substantially overlapping linear information. However, high correlation alone does not justify automatic removal because correlated predictors may differ in physical meaning, target relevance, temporal stability, or usefulness to nonlinear models.

The complete predictor correlation matrix is therefore examined first, followed by extraction of the strongest unique predictor pairs for more detailed redundancy assessment.

In [49]:
# ============================================================
# E1.6.1 Calculate Pairwise Pearson Predictor Correlations
# ============================================================

pearson_predictor_matrix = (
    X_development[
        candidate_predictors
    ]
    .corr(method="pearson")
)

print("=" * 70)
print("Pairwise Pearson Correlation Matrix")
print("=" * 70)

print(
    "Matrix shape:",
    pearson_predictor_matrix.shape
)

display(
    pearson_predictor_matrix.round(3)
)

Pairwise Pearson Correlation Matrix
Matrix shape: (20, 20)


,current,pressure_anode_inlet,pressure_anode_outlet,pressure_cathode_inlet,pressure_cathode_outlet,temp_anode_endplate,temp_anode_dewpoint_water,temp_anode_inlet,temp_anode_outlet,temp_cathode_dewpoint_water,temp_cathode_inlet,temp_cathode_outlet,total_anode_stack_flow,total_cathode_stack_flow,anode_pressure_diff,cathode_pressure_diff,anode_temp_diff,cathode_temp_diff,anode_dewpoint_offset,cathode_dewpoint_offset
current,1.0000,0.1070,-0.1100,0.6110,-0.4450,0.3700,-0.0080,0.4580,0.0200,-0.0140,0.7980,0.3200,0.9800,0.9810,0.6800,0.9390,-0.0140,0.1150,0.4480,0.7820
pressure_anode_inlet,0.1070,1.0000,0.9490,0.1630,0.0540,0.0550,0.0010,0.0160,0.0690,0.0150,0.0540,0.0130,0.1030,0.1030,0.1070,0.0910,0.0690,-0.0010,0.0150,0.0480
pressure_anode_outlet,-0.1100,0.9490,1.0000,0.0450,0.1870,-0.0100,-0.0200,-0.1050,-0.0740,-0.0080,-0.1280,-0.0510,-0.1200,-0.1200,-0.2110,-0.1330,-0.0670,-0.0190,-0.0930,-0.1220
pressure_cathode_inlet,0.6110,0.1630,0.0450,1.0000,0.3770,0.1360,-0.0010,0.0370,-0.0640,0.0500,0.3940,-0.2280,0.5700,0.5700,0.3630,0.5170,-0.0670,-0.3380,0.0360,0.3700
pressure_cathode_outlet,-0.4450,0.0540,0.1870,0.3770,1.0000,-0.2910,0.0110,-0.5230,-0.1120,0.0770,-0.4960,-0.6500,-0.5220,-0.5220,-0.4230,-0.5980,-0.0740,-0.5330,-0.5120,-0.5050
temp_anode_endplate,0.3700,0.0550,-0.0100,0.1360,-0.2910,1.0000,-0.0150,0.2330,0.3610,-0.0270,0.3430,0.3870,0.3920,0.3920,0.2010,0.3860,0.3450,0.3050,0.2320,0.3420
temp_anode_dewpoint_water,-0.0080,0.0010,-0.0200,-0.0010,0.0110,-0.0150,1.0000,0.1650,0.1670,0.5520,0.0390,-0.0480,-0.0080,-0.0080,0.0650,-0.0110,0.1550,-0.0590,-0.2910,-0.1170
temp_anode_inlet,0.4580,0.0160,-0.1050,0.0370,-0.5230,0.2330,0.1650,1.0000,0.1050,0.1140,0.4090,0.4360,0.4920,0.4920,0.3820,0.5150,0.0310,0.3380,0.8960,0.3670
temp_anode_outlet,0.0200,0.0690,-0.0740,-0.0640,-0.1120,0.3610,0.1670,0.1050,1.0000,0.1890,-0.0040,0.2630,0.0290,0.0280,0.4500,0.0480,0.9970,0.2700,0.0270,-0.0570
temp_cathode_dewpoint_water,-0.0140,0.0150,-0.0080,0.0500,0.0770,-0.0270,0.5520,0.1140,0.1890,1.0000,0.0510,-0.0620,-0.0200,-0.0200,0.0720,-0.0280,0.1810,-0.0770,-0.1380,-0.2300


In [50]:
# ============================================================
# E1.6.1 Visualise Pairwise Pearson Redundancy
# ============================================================

plt.figure(
    figsize=(17, 14)
)

sns.heatmap(
    pearson_predictor_matrix,

    # Display Pearson coefficient inside each cell
    annot=True,
    fmt=".2f",
    annot_kws={
        "size": 7
    },

    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,

    linewidths=0.3,

    cbar_kws={
        "label": "Pearson Correlation (r)"
    }
)

plt.title(
    "Pairwise Pearson Correlation Among Candidate Predictors"
)

plt.xlabel(
    "Candidate Predictor"
)

plt.ylabel(
    "Candidate Predictor"
)

plt.xticks(
    rotation=70,
    ha="right"
)

plt.yticks(
    rotation=0
)

plt.tight_layout()

plt.show()

C:\Users\usman\Desktop\PEMFC_Dissertation\pemfc_env\Lib\site-packages\seaborn\matrix.py:243: PendingDeprecationWarning: The set_bad function will be deprecated in a future version. Use cmap.with_extremes(bad=...) or Colormap(bad=...) instead.
  self.cmap.set_bad(bad)


<Figure size 1700x1400 with 2 Axes>

In [51]:
# ============================================================
# E1.6.1 Extract Unique Pearson Predictor Pairs
# ============================================================

pearson_pairwise_results = []

for i in range(
    len(candidate_predictors)
):
    for j in range(
        i + 1,
        len(candidate_predictors)
    ):

        predictor_1 = candidate_predictors[i]
        predictor_2 = candidate_predictors[j]

        correlation_value = (
            pearson_predictor_matrix.loc[
                predictor_1,
                predictor_2
            ]
        )

        pearson_pairwise_results.append(
            {
                "Predictor_1": predictor_1,
                "Predictor_2": predictor_2,
                "Pearson_r": correlation_value,
                "Absolute_r": abs(correlation_value)
            }
        )

pearson_redundancy_table = pd.DataFrame(
    pearson_pairwise_results
)

pearson_redundancy_table = (
    pearson_redundancy_table
    .sort_values(
        by="Absolute_r",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "Number of unique predictor pairs:",
    len(pearson_redundancy_table)
)

display(
    pearson_redundancy_table.round(4)
)

Number of unique predictor pairs: 190


,Predictor_1,Predictor_2,Pearson_r,Absolute_r
0,total_anode_stack_flow,total_cathode_stack_flow,1.0000,1.0000
1,temp_anode_outlet,anode_temp_diff,0.9973,0.9973
2,current,total_cathode_stack_flow,0.9806,0.9806
3,current,total_anode_stack_flow,0.9804,0.9804
4,total_anode_stack_flow,cathode_pressure_diff,0.9756,0.9756
...,...,...,...,...
185,pressure_anode_outlet,temp_cathode_dewpoint_water,-0.0075,0.0075
186,temp_anode_outlet,temp_cathode_inlet,-0.0040,0.0040
187,pressure_anode_inlet,temp_anode_dewpoint_water,0.0012,0.0012
188,pressure_anode_inlet,cathode_temp_diff,-0.0010,0.0010


In [52]:
# ============================================================
# E1.6.1 Identify Highly Correlated Predictor Pairs
# ============================================================

PEARSON_REDUNDANCY_FLAG = 0.80

high_pearson_redundancy = (
    pearson_redundancy_table[
        pearson_redundancy_table[
            "Absolute_r"
        ] >= PEARSON_REDUNDANCY_FLAG
    ]
    .copy()
    .reset_index(drop=True)
)

print("=" * 70)
print(
    f"Predictor pairs with |Pearson r| >= "
    f"{PEARSON_REDUNDANCY_FLAG:.2f}"
)
print("=" * 70)

print(
    "Number of flagged pairs:",
    len(high_pearson_redundancy)
)

display(
    high_pearson_redundancy.round(4)
)

Predictor pairs with |Pearson r| >= 0.80
Number of flagged pairs: 13


,Predictor_1,Predictor_2,Pearson_r,Absolute_r
0,total_anode_stack_flow,total_cathode_stack_flow,1.0000,1.0000
1,temp_anode_outlet,anode_temp_diff,0.9973,0.9973
2,current,total_cathode_stack_flow,0.9806,0.9806
3,current,total_anode_stack_flow,0.9804,0.9804
4,total_anode_stack_flow,cathode_pressure_diff,0.9756,0.9756
5,total_cathode_stack_flow,cathode_pressure_diff,0.9753,0.9753
6,temp_cathode_outlet,cathode_temp_diff,0.9657,0.9657
7,temp_cathode_inlet,cathode_dewpoint_offset,0.9603,0.9603
8,pressure_anode_inlet,pressure_anode_outlet,0.9492,0.9492
9,current,cathode_pressure_diff,0.9394,0.9394


In [53]:
# ============================================================
# E1.6.1 Save Pairwise Pearson Redundancy Results
# ============================================================

pearson_redundancy_output_file = (
    results_dir
    / "pearson_predictor_redundancy.csv"
)

high_pearson_output_file = (
    results_dir
    / "high_pearson_redundancy_pairs.csv"
)

pearson_redundancy_table.to_csv(
    pearson_redundancy_output_file,
    index=False
)

high_pearson_redundancy.to_csv(
    high_pearson_output_file,
    index=False
)

print(
    "Pairwise Pearson redundancy results saved successfully."
)

print(
    "Full pairwise table:",
    pearson_redundancy_output_file
)

print(
    "Flagged redundancy pairs:",
    high_pearson_output_file
)

Pairwise Pearson redundancy results saved successfully.
Full pairwise table: C:\Users\usman\Desktop\PEMFC_Dissertation\results\feature_selection\pearson_predictor_redundancy.csv
Flagged redundancy pairs: C:\Users\usman\Desktop\PEMFC_Dissertation\results\feature_selection\high_pearson_redundancy_pairs.csv


In [ ]:
### E1.6.2 Pairwise Spearman Redundancy

Pairwise Spearman rank correlation is used to assess monotonic redundancy among the candidate predictors.

Whereas Pearson correlation identifies linear relationships between predictors, Spearman correlation evaluates whether two predictors tend to increase or decrease together consistently without requiring their relationship to be linear.

The Spearman rank correlation coefficient, \(\rho\), ranges from -1 to +1. Large absolute values indicate strong monotonic association and therefore potentially overlapping predictor information.

Spearman redundancy analysis complements the preceding Pearson assessment. Predictor pairs that exhibit strong association under both methods provide stronger evidence of redundancy, while disagreement between Pearson and Spearman may indicate nonlinear monotonic relationships.

As in the Pearson analysis, an absolute Spearman correlation of 0.80 or greater is used as a diagnostic flag for potentially high redundancy rather than as an automatic feature-removal threshold.

The analysis uses development data only, while the final test region remains excluded.

In [54]:
# ============================================================
# E1.6.2 Calculate Pairwise Spearman Predictor Correlations
# ============================================================

spearman_predictor_matrix = (
    X_development[
        candidate_predictors
    ]
    .corr(method="spearman")
)

print("=" * 70)
print("Pairwise Spearman Correlation Matrix")
print("=" * 70)

print(
    "Matrix shape:",
    spearman_predictor_matrix.shape
)

display(
    spearman_predictor_matrix.round(3)
)

Pairwise Spearman Correlation Matrix
Matrix shape: (20, 20)


,current,pressure_anode_inlet,pressure_anode_outlet,pressure_cathode_inlet,pressure_cathode_outlet,temp_anode_endplate,temp_anode_dewpoint_water,temp_anode_inlet,temp_anode_outlet,temp_cathode_dewpoint_water,temp_cathode_inlet,temp_cathode_outlet,total_anode_stack_flow,total_cathode_stack_flow,anode_pressure_diff,cathode_pressure_diff,anode_temp_diff,cathode_temp_diff,anode_dewpoint_offset,cathode_dewpoint_offset
current,1.0000,0.3870,-0.3230,0.7650,-0.2600,0.3260,-0.0250,0.3910,0.0210,-0.0030,0.6940,0.2110,0.9150,0.9150,0.5770,0.8620,-0.0030,0.0320,0.3730,0.6830
pressure_anode_inlet,0.3870,1.0000,0.2460,0.5280,0.1980,0.0890,0.0010,0.0760,-0.0040,0.0280,0.1700,-0.0220,0.3990,0.3980,0.4550,0.3380,-0.0100,-0.0620,0.0720,0.1600
pressure_anode_outlet,-0.3230,0.2460,1.0000,-0.1220,0.4910,-0.1390,-0.0650,-0.3270,-0.4250,-0.0510,-0.4050,-0.1920,-0.3370,-0.3360,-0.6570,-0.3760,-0.4060,-0.0890,-0.2830,-0.3840
pressure_cathode_inlet,0.7650,0.5280,-0.1220,1.0000,0.1960,0.2360,-0.0160,0.1510,-0.0330,0.0320,0.5700,-0.0650,0.7820,0.7830,0.4840,0.7040,-0.0450,-0.1960,0.1470,0.5460
pressure_cathode_outlet,-0.2600,0.1980,0.4910,0.1960,1.0000,-0.2330,0.0290,-0.5360,-0.0980,0.0940,-0.4000,-0.6010,-0.2920,-0.2930,-0.2710,-0.4170,-0.0680,-0.4830,-0.5120,-0.4230
temp_anode_endplate,0.3260,0.0890,-0.1390,0.2360,-0.2330,1.0000,0.0020,0.2510,0.3690,-0.0050,0.3330,0.3660,0.3850,0.3860,0.1840,0.3250,0.3550,0.2850,0.2410,0.3360
temp_anode_dewpoint_water,-0.0250,0.0010,-0.0650,-0.0160,0.0290,0.0020,1.0000,0.0610,0.2320,0.4570,-0.0150,0.0160,-0.0360,-0.0370,0.0820,-0.0290,0.2270,0.0200,-0.2810,-0.1390
temp_anode_inlet,0.3910,0.0760,-0.3270,0.1510,-0.5360,0.2510,0.0610,1.0000,0.1080,0.0400,0.3580,0.5430,0.4230,0.4240,0.3220,0.4310,0.0460,0.4460,0.8870,0.3350
temp_anode_outlet,0.0210,-0.0040,-0.4250,-0.0330,-0.0980,0.3690,0.2320,0.1080,1.0000,0.2410,-0.0060,0.2330,0.0240,0.0200,0.4490,0.0280,0.9970,0.2400,0.0210,-0.0650
temp_cathode_dewpoint_water,-0.0030,0.0280,-0.0510,0.0320,0.0940,-0.0050,0.4570,0.0400,0.2410,1.0000,0.0390,-0.0350,-0.0210,-0.0230,0.1000,-0.0210,0.2370,-0.0440,-0.1590,-0.2080


In [55]:
# ============================================================
# E1.6.2 Visualise Pairwise Spearman Redundancy
# ============================================================

plt.figure(
    figsize=(17, 14)
)

sns.heatmap(
    spearman_predictor_matrix,

    # Display Spearman coefficient inside each cell
    annot=True,
    fmt=".2f",
    annot_kws={
        "size": 7
    },

    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,

    linewidths=0.3,

    cbar_kws={
        "label": "Spearman Correlation (ρ)"
    }
)

plt.title(
    "Pairwise Spearman Correlation Among Candidate Predictors"
)

plt.xlabel(
    "Candidate Predictor"
)

plt.ylabel(
    "Candidate Predictor"
)

plt.xticks(
    rotation=70,
    ha="right"
)

plt.yticks(
    rotation=0
)

plt.tight_layout()

plt.show()

C:\Users\usman\Desktop\PEMFC_Dissertation\pemfc_env\Lib\site-packages\seaborn\matrix.py:243: PendingDeprecationWarning: The set_bad function will be deprecated in a future version. Use cmap.with_extremes(bad=...) or Colormap(bad=...) instead.
  self.cmap.set_bad(bad)


<Figure size 1700x1400 with 2 Axes>

In [56]:
# ============================================================
# E1.6.2 Extract Unique Spearman Predictor Pairs
# ============================================================

spearman_pairwise_results = []

for i in range(
    len(candidate_predictors)
):
    for j in range(
        i + 1,
        len(candidate_predictors)
    ):

        predictor_1 = candidate_predictors[i]
        predictor_2 = candidate_predictors[j]

        correlation_value = (
            spearman_predictor_matrix.loc[
                predictor_1,
                predictor_2
            ]
        )

        spearman_pairwise_results.append(
            {
                "Predictor_1": predictor_1,
                "Predictor_2": predictor_2,
                "Spearman_rho": correlation_value,
                "Absolute_rho": abs(correlation_value)
            }
        )

spearman_redundancy_table = pd.DataFrame(
    spearman_pairwise_results
)

spearman_redundancy_table = (
    spearman_redundancy_table
    .sort_values(
        by="Absolute_rho",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "Number of unique predictor pairs:",
    len(spearman_redundancy_table)
)

display(
    spearman_redundancy_table.round(4)
)

Number of unique predictor pairs: 190


,Predictor_1,Predictor_2,Spearman_rho,Absolute_rho
0,total_anode_stack_flow,total_cathode_stack_flow,0.9977,0.9977
1,temp_anode_outlet,anode_temp_diff,0.9971,0.9971
2,temp_cathode_outlet,cathode_temp_diff,0.9656,0.9656
3,temp_cathode_inlet,cathode_dewpoint_offset,0.9510,0.9510
4,current,total_cathode_stack_flow,0.9148,0.9148
...,...,...,...,...
185,current,temp_cathode_dewpoint_water,-0.0025,0.0025
186,temp_anode_endplate,temp_anode_dewpoint_water,0.0022,0.0022
187,cathode_pressure_diff,anode_temp_diff,0.0020,0.0020
188,total_anode_stack_flow,anode_temp_diff,-0.0019,0.0019


In [57]:
# ============================================================
# E1.6.2 Identify Highly Correlated Spearman Predictor Pairs
# ============================================================

SPEARMAN_REDUNDANCY_FLAG = 0.80

high_spearman_redundancy = (
    spearman_redundancy_table[
        spearman_redundancy_table[
            "Absolute_rho"
        ] >= SPEARMAN_REDUNDANCY_FLAG
    ]
    .copy()
    .reset_index(drop=True)
)

print("=" * 70)

print(
    f"Predictor pairs with |Spearman rho| >= "
    f"{SPEARMAN_REDUNDANCY_FLAG:.2f}"
)

print("=" * 70)

print(
    "Number of flagged pairs:",
    len(high_spearman_redundancy)
)

display(
    high_spearman_redundancy.round(4)
)

Predictor pairs with |Spearman rho| >= 0.80
Number of flagged pairs: 10


,Predictor_1,Predictor_2,Spearman_rho,Absolute_rho
0,total_anode_stack_flow,total_cathode_stack_flow,0.9977,0.9977
1,temp_anode_outlet,anode_temp_diff,0.9971,0.9971
2,temp_cathode_outlet,cathode_temp_diff,0.9656,0.9656
3,temp_cathode_inlet,cathode_dewpoint_offset,0.9510,0.9510
4,current,total_cathode_stack_flow,0.9148,0.9148
5,current,total_anode_stack_flow,0.9148,0.9148
6,total_cathode_stack_flow,cathode_pressure_diff,0.9044,0.9044
7,total_anode_stack_flow,cathode_pressure_diff,0.9025,0.9025
8,temp_anode_inlet,anode_dewpoint_offset,0.8871,0.8871
9,current,cathode_pressure_diff,0.8621,0.8621


In [58]:
# ============================================================
# E1.6.2 Save Pairwise Spearman Redundancy Results
# ============================================================

spearman_redundancy_output_file = (
    results_dir
    / "spearman_predictor_redundancy.csv"
)

high_spearman_output_file = (
    results_dir
    / "high_spearman_redundancy_pairs.csv"
)

spearman_redundancy_table.to_csv(
    spearman_redundancy_output_file,
    index=False
)

high_spearman_redundancy.to_csv(
    high_spearman_output_file,
    index=False
)

print(
    "Pairwise Spearman redundancy results saved successfully."
)

print(
    "Full pairwise table:",
    spearman_redundancy_output_file
)

print(
    "Flagged redundancy pairs:",
    high_spearman_output_file
)

Pairwise Spearman redundancy results saved successfully.
Full pairwise table: C:\Users\usman\Desktop\PEMFC_Dissertation\results\feature_selection\spearman_predictor_redundancy.csv
Flagged redundancy pairs: C:\Users\usman\Desktop\PEMFC_Dissertation\results\feature_selection\high_spearman_redundancy_pairs.csv


In [ ]:
### E1.6.3 Redundancy Groups

The pairwise Pearson and Spearman analyses identify individual predictor pairs exhibiting strong linear or monotonic association. However, predictor redundancy may extend across multiple interconnected variables rather than occurring only between isolated pairs.

To provide a more systematic representation of this structure, the Pearson and Spearman redundancy results are integrated and used to identify groups of interconnected predictors.

A predictor pair is flagged as having strong redundancy evidence when an absolute Pearson correlation of at least 0.80 and/or an absolute Spearman correlation of at least 0.80 is observed.

The integrated analysis distinguishes between:

- redundancy supported by both Pearson and Spearman;
- redundancy identified primarily by Pearson;
- redundancy identified primarily by Spearman.

Predictors connected through flagged pairwise relationships are subsequently grouped into redundancy components. These groups represent sets of predictors containing potentially overlapping information.

The identified groups are diagnostic rather than automatic feature-removal groups. Subsequent decisions will also consider target relevance, physical interpretation, multicollinearity, embedded feature selection, nonlinear model-based selection, and temporal stability.

All redundancy evidence is derived exclusively from the development region.

In [59]:
# ============================================================
# E1.6.3 Integrate Pearson and Spearman Redundancy Evidence
# ============================================================

integrated_redundancy = (
    pearson_redundancy_table[
        [
            "Predictor_1",
            "Predictor_2",
            "Pearson_r",
            "Absolute_r"
        ]
    ]
    .merge(
        spearman_redundancy_table[
            [
                "Predictor_1",
                "Predictor_2",
                "Spearman_rho",
                "Absolute_rho"
            ]
        ],
        on=[
            "Predictor_1",
            "Predictor_2"
        ],
        how="inner"
    )
)

print("=" * 75)
print("Integrated Pearson–Spearman Predictor Redundancy")
print("=" * 75)

print(
    "Number of unique predictor pairs:",
    len(integrated_redundancy)
)

display(
    integrated_redundancy.round(4)
)

Integrated Pearson–Spearman Predictor Redundancy
Number of unique predictor pairs: 190


,Predictor_1,Predictor_2,Pearson_r,Absolute_r,Spearman_rho,Absolute_rho
0,total_anode_stack_flow,total_cathode_stack_flow,1.0000,1.0000,0.9977,0.9977
1,temp_anode_outlet,anode_temp_diff,0.9973,0.9973,0.9971,0.9971
2,current,total_cathode_stack_flow,0.9806,0.9806,0.9148,0.9148
3,current,total_anode_stack_flow,0.9804,0.9804,0.9148,0.9148
4,total_anode_stack_flow,cathode_pressure_diff,0.9756,0.9756,0.9025,0.9025
...,...,...,...,...,...,...
185,pressure_anode_outlet,temp_cathode_dewpoint_water,-0.0075,0.0075,-0.0507,0.0507
186,temp_anode_outlet,temp_cathode_inlet,-0.0040,0.0040,-0.0063,0.0063
187,pressure_anode_inlet,temp_anode_dewpoint_water,0.0012,0.0012,0.0005,0.0005
188,pressure_anode_inlet,cathode_temp_diff,-0.0010,0.0010,-0.0623,0.0623


In [60]:
# ============================================================
# E1.6.3 Classify Integrated Redundancy Evidence
# ============================================================

integrated_redundancy["Pearson_Flag"] = (
    integrated_redundancy["Absolute_r"]
    >= PEARSON_REDUNDANCY_FLAG
)

integrated_redundancy["Spearman_Flag"] = (
    integrated_redundancy["Absolute_rho"]
    >= SPEARMAN_REDUNDANCY_FLAG
)

integrated_redundancy["Any_Redundancy_Flag"] = (
    integrated_redundancy[
        [
            "Pearson_Flag",
            "Spearman_Flag"
        ]
    ]
    .any(axis=1)
)

integrated_redundancy["Both_Methods_Flag"] = (
    integrated_redundancy[
        [
            "Pearson_Flag",
            "Spearman_Flag"
        ]
    ]
    .all(axis=1)
)

In [61]:
# ============================================================
# E1.6.3 Identify Redundancy Evidence Type
# ============================================================

def classify_redundancy_evidence(row):

    if row["Pearson_Flag"] and row["Spearman_Flag"]:
        return "Pearson + Spearman"

    elif row["Pearson_Flag"]:
        return "Pearson only"

    elif row["Spearman_Flag"]:
        return "Spearman only"

    else:
        return "Not flagged"


integrated_redundancy["Redundancy_Evidence"] = (
    integrated_redundancy.apply(
        classify_redundancy_evidence,
        axis=1
    )
)

redundancy_evidence_summary = (
    integrated_redundancy[
        "Redundancy_Evidence"
    ]
    .value_counts()
    .rename_axis(
        "Redundancy_Evidence"
    )
    .reset_index(
        name="Number_of_Pairs"
    )
)

display(
    redundancy_evidence_summary
)

,Redundancy_Evidence,Number_of_Pairs
0,Not flagged,177
1,Pearson + Spearman,10
2,Pearson only,3


In [62]:
# ============================================================
# E1.6.3 Extract Integrated High-Redundancy Pairs
# ============================================================

high_integrated_redundancy = (
    integrated_redundancy[
        integrated_redundancy[
            "Any_Redundancy_Flag"
        ]
    ]
    .copy()
)

high_integrated_redundancy[
    "Maximum_Absolute_Association"
] = (
    high_integrated_redundancy[
        [
            "Absolute_r",
            "Absolute_rho"
        ]
    ]
    .max(axis=1)
)

high_integrated_redundancy = (
    high_integrated_redundancy
    .sort_values(
        by="Maximum_Absolute_Association",
        ascending=False
    )
    .reset_index(drop=True)
)

print("=" * 75)
print("Integrated High-Redundancy Predictor Pairs")
print("=" * 75)

print(
    "Number of flagged pairs:",
    len(high_integrated_redundancy)
)

display(
    high_integrated_redundancy[
        [
            "Predictor_1",
            "Predictor_2",
            "Pearson_r",
            "Spearman_rho",
            "Redundancy_Evidence",
            "Maximum_Absolute_Association"
        ]
    ].round(4)
)

Integrated High-Redundancy Predictor Pairs
Number of flagged pairs: 13


,Predictor_1,Predictor_2,Pearson_r,Spearman_rho,Redundancy_Evidence,Maximum_Absolute_Association
0,total_anode_stack_flow,total_cathode_stack_flow,1.0000,0.9977,Pearson + Spearman,1.0000
1,temp_anode_outlet,anode_temp_diff,0.9973,0.9971,Pearson + Spearman,0.9973
2,current,total_cathode_stack_flow,0.9806,0.9148,Pearson + Spearman,0.9806
3,current,total_anode_stack_flow,0.9804,0.9148,Pearson + Spearman,0.9804
4,total_anode_stack_flow,cathode_pressure_diff,0.9756,0.9025,Pearson + Spearman,0.9756
5,total_cathode_stack_flow,cathode_pressure_diff,0.9753,0.9044,Pearson + Spearman,0.9753
6,temp_cathode_outlet,cathode_temp_diff,0.9657,0.9656,Pearson + Spearman,0.9657
7,temp_cathode_inlet,cathode_dewpoint_offset,0.9603,0.9510,Pearson + Spearman,0.9603
8,pressure_anode_inlet,pressure_anode_outlet,0.9492,0.2463,Pearson only,0.9492
9,current,cathode_pressure_diff,0.9394,0.8621,Pearson + Spearman,0.9394


In [63]:
# ============================================================
# E1.6.3 Identify Interconnected Redundancy Groups
# ============================================================

# Build adjacency structure
adjacency = {
    predictor: set()
    for predictor in candidate_predictors
}

for _, row in high_integrated_redundancy.iterrows():

    predictor_1 = row["Predictor_1"]
    predictor_2 = row["Predictor_2"]

    adjacency[predictor_1].add(
        predictor_2
    )

    adjacency[predictor_2].add(
        predictor_1
    )


# ------------------------------------------------------------
# Find connected components
# ------------------------------------------------------------

visited = set()
redundancy_groups = []

for predictor in candidate_predictors:

    if predictor in visited:
        continue

    stack = [predictor]
    component = []

    while stack:

        current_predictor = stack.pop()

        if current_predictor in visited:
            continue

        visited.add(
            current_predictor
        )

        component.append(
            current_predictor
        )

        stack.extend(
            adjacency[current_predictor]
            - visited
        )

    # Keep only groups containing at least two predictors
    if len(component) > 1:

        redundancy_groups.append(
            sorted(component)
        )

In [64]:
# ============================================================
# E1.6.3 Summarise Redundancy Groups
# ============================================================

redundancy_group_records = []

for group_number, group in enumerate(
    redundancy_groups,
    start=1
):

    redundancy_group_records.append(
        {
            "Redundancy_Group":
                f"Group_{group_number}",

            "Number_of_Predictors":
                len(group),

            "Predictors":
                ", ".join(group)
        }
    )

redundancy_groups_table = pd.DataFrame(
    redundancy_group_records
)

print("=" * 75)
print("Predictor Redundancy Groups")
print("=" * 75)

print(
    "Number of redundancy groups:",
    len(redundancy_groups)
)

display(
    redundancy_groups_table
)

Predictor Redundancy Groups
Number of redundancy groups: 5


,Redundancy_Group,Number_of_Predictors,Predictors
0,Group_1,6,"cathode_dewpoint_offset, cathode_pressure_diff..."
1,Group_2,2,"pressure_anode_inlet, pressure_anode_outlet"
2,Group_3,2,"anode_dewpoint_offset, temp_anode_inlet"
3,Group_4,2,"anode_temp_diff, temp_anode_outlet"
4,Group_5,2,"cathode_temp_diff, temp_cathode_outlet"


In [65]:
# ============================================================
# E1.6.3 Identify Predictors Without High Pairwise Redundancy
# ============================================================

grouped_predictors = {
    predictor
    for group in redundancy_groups
    for predictor in group
}

non_redundant_predictors = [
    predictor
    for predictor in candidate_predictors
    if predictor not in grouped_predictors
]

print("=" * 75)
print("Predictors Without Flagged High Pairwise Redundancy")
print("=" * 75)

print(
    "Number of predictors:",
    len(non_redundant_predictors)
)

for predictor in non_redundant_predictors:
    print("-", predictor)

Predictors Without Flagged High Pairwise Redundancy
Number of predictors: 6
- pressure_cathode_inlet
- pressure_cathode_outlet
- temp_anode_endplate
- temp_anode_dewpoint_water
- temp_cathode_dewpoint_water
- anode_pressure_diff


In [66]:
# ============================================================
# E1.6.3 Save Integrated Redundancy Results
# ============================================================

integrated_redundancy_file = (
    results_dir
    / "integrated_predictor_redundancy.csv"
)

high_integrated_redundancy_file = (
    results_dir
    / "high_integrated_redundancy_pairs.csv"
)

redundancy_groups_file = (
    results_dir
    / "predictor_redundancy_groups.csv"
)

integrated_redundancy.to_csv(
    integrated_redundancy_file,
    index=False
)

high_integrated_redundancy.to_csv(
    high_integrated_redundancy_file,
    index=False
)

redundancy_groups_table.to_csv(
    redundancy_groups_file,
    index=False
)

print(
    "Integrated redundancy results saved successfully."
)

print(
    "\nFull integrated evidence:",
    integrated_redundancy_file
)

print(
    "\nHigh-redundancy pairs:",
    high_integrated_redundancy_file
)

print(
    "\nRedundancy groups:",
    redundancy_groups_file
)

Integrated redundancy results saved successfully.

Full integrated evidence: C:\Users\usman\Desktop\PEMFC_Dissertation\results\feature_selection\integrated_predictor_redundancy.csv

High-redundancy pairs: C:\Users\usman\Desktop\PEMFC_Dissertation\results\feature_selection\high_integrated_redundancy_pairs.csv

Redundancy groups: C:\Users\usman\Desktop\PEMFC_Dissertation\results\feature_selection\predictor_redundancy_groups.csv


In [ ]:
# E1.7 Multicollinearity Assessment

## E1.7.1 Variance Inflation Factor (VIF)

Pairwise Pearson and Spearman analyses identify redundancy between predictor pairs, but they cannot fully detect multivariate linear dependence where one predictor can be explained by a combination of several other predictors.

Variance Inflation Factor (VIF) is therefore used to assess multicollinearity across the complete candidate predictor set.

For each predictor, an auxiliary linear regression is fitted using that predictor as the response and all remaining predictors as explanatory variables.

VIF is calculated as:

VIF = 1 / (1 - R²)

where R² measures how strongly the predictor can be explained by the remaining predictors.

If R² is effectively equal to 1, the predictor is exactly or near-exactly explained by the remaining predictors and its VIF tends toward infinity.

Because several engineered predictors were mathematically constructed from original variables, exact linear dependencies are expected and are explicitly identified rather than treated as numerical errors.

This analysis is diagnostic and does not automatically remove predictors.

In [67]:
# ============================================================
# E1.7.2 Prepare Predictor Dataset for VIF
# ============================================================

import numpy as np
import pandas as pd

X_vif = X_development[candidate_predictors].copy()

print("=" * 70)
print("VIF Analysis Dataset")
print("=" * 70)

print(f"Observations       : {X_vif.shape[0]:,}")
print(f"Predictors         : {X_vif.shape[1]}")
print(f"Missing values     : {X_vif.isna().sum().sum():,}")
print(f"Infinite values    : {np.isinf(X_vif.to_numpy()).sum():,}")

constant_predictors = [
    predictor
    for predictor in X_vif.columns
    if X_vif[predictor].nunique(dropna=False) <= 1
]

print(f"Constant predictors: {len(constant_predictors)}")

if constant_predictors:
    print("\nConstant predictors:")
    for predictor in constant_predictors:
        print(f"- {predictor}")

VIF Analysis Dataset
Observations       : 3,049,120
Predictors         : 20
Missing values     : 0
Infinite values    : 0
Constant predictors: 0


In [68]:
# ============================================================
# E1.7.3 Assess Predictor-Matrix Conditioning
# ============================================================

# Standardise only for numerical matrix diagnostics.
# This does not change correlation structure.

X_vif_standardized = (
    X_vif - X_vif.mean()
) / X_vif.std(ddof=0)

vif_correlation_matrix = X_vif_standardized.corr()

matrix_dimension = vif_correlation_matrix.shape[0]
matrix_rank = np.linalg.matrix_rank(vif_correlation_matrix.to_numpy())
rank_deficiency = matrix_dimension - matrix_rank
condition_number = np.linalg.cond(vif_correlation_matrix.to_numpy())

print("=" * 70)
print("VIF Predictor-Matrix Diagnostics")
print("=" * 70)

print(f"Matrix dimension : {matrix_dimension}")
print(f"Matrix rank      : {matrix_rank}")
print(f"Rank deficiency  : {rank_deficiency}")
print(f"Condition number : {condition_number:.4e}")
print(f"Full rank        : {matrix_rank == matrix_dimension}")

VIF Predictor-Matrix Diagnostics
Matrix dimension : 20
Matrix rank      : 18
Rank deficiency  : 2
Condition number : 4.9107e+14
Full rank        : False


In [69]:
# ============================================================
# E1.7.4 Calculate Variance Inflation Factors
# ============================================================

from sklearn.linear_model import LinearRegression

vif_records = []

# Numerical tolerance for treating R² as effectively 1.
VIF_TOLERANCE = 1e-10

for predictor in candidate_predictors:

    other_predictors = [
        other
        for other in candidate_predictors
        if other != predictor
    ]

    # Predictor being assessed
    y_aux = X_vif[predictor].to_numpy()

    # All remaining predictors
    X_aux = X_vif[other_predictors].to_numpy()

    auxiliary_model = LinearRegression()
    auxiliary_model.fit(X_aux, y_aux)

    r_squared = auxiliary_model.score(X_aux, y_aux)

    tolerance = 1.0 - r_squared

    # Exact / near-exact multicollinearity
    if tolerance <= VIF_TOLERANCE:
        vif_value = np.inf
    else:
        vif_value = 1.0 / tolerance

    vif_records.append({
        "Predictor": predictor,
        "Auxiliary_R2": r_squared,
        "Tolerance": tolerance,
        "VIF": vif_value
    })

vif_results_df = pd.DataFrame(vif_records)

# Sort infinite VIFs first, then descending finite VIF
vif_results_df["_sort_vif"] = (
    vif_results_df["VIF"]
    .replace(np.inf, np.finfo(float).max)
)

vif_results_df = (
    vif_results_df
    .sort_values("_sort_vif", ascending=False)
    .drop(columns="_sort_vif")
    .reset_index(drop=True)
)

print("=" * 70)
print("Variance Inflation Factor Results")
print("=" * 70)

display(
    vif_results_df.style.format({
        "Auxiliary_R2": "{:.10f}",
        "Tolerance": "{:.10e}",
        "VIF": lambda value:
            "∞" if np.isinf(value) else f"{value:.4f}"
    })
)

Variance Inflation Factor Results


,Predictor,Auxiliary_R2,Tolerance,VIF
0,pressure_anode_inlet,1.0000000000,0.0000000000e+00,∞
1,pressure_anode_outlet,1.0000000000,0.0000000000e+00,∞
2,pressure_cathode_inlet,1.0000000000,0.0000000000e+00,∞
3,pressure_cathode_outlet,1.0000000000,0.0000000000e+00,∞
4,anode_dewpoint_offset,1.0000000000,0.0000000000e+00,∞
5,temp_anode_dewpoint_water,1.0000000000,0.0000000000e+00,∞
6,temp_anode_inlet,1.0000000000,0.0000000000e+00,∞
7,temp_anode_outlet,1.0000000000,0.0000000000e+00,∞
8,temp_cathode_inlet,1.0000000000,0.0000000000e+00,∞
9,temp_cathode_dewpoint_water,1.0000000000,0.0000000000e+00,∞


In [70]:
# ============================================================
# E1.7.5 Classify VIF Magnitude
# ============================================================

def classify_vif(vif_value):

    if np.isinf(vif_value):
        return "Exact / near-exact multicollinearity"

    if vif_value >= 10:
        return "Very high"

    if vif_value >= 5:
        return "High"

    return "Low"


vif_results_df["Multicollinearity_Level"] = (
    vif_results_df["VIF"]
    .apply(classify_vif)
)

print("=" * 70)
print("VIF Classification")
print("=" * 70)

display(
    vif_results_df[
        [
            "Predictor",
            "Auxiliary_R2",
            "Tolerance",
            "VIF",
            "Multicollinearity_Level"
        ]
    ].style.format({
        "Auxiliary_R2": "{:.10f}",
        "Tolerance": "{:.10e}",
        "VIF": lambda value:
            "∞" if np.isinf(value) else f"{value:.4f}"
    })
)

VIF Classification


,Predictor,Auxiliary_R2,Tolerance,VIF,Multicollinearity_Level
0,pressure_anode_inlet,1.0000000000,0.0000000000e+00,∞,Exact / near-exact multicollinearity
1,pressure_anode_outlet,1.0000000000,0.0000000000e+00,∞,Exact / near-exact multicollinearity
2,pressure_cathode_inlet,1.0000000000,0.0000000000e+00,∞,Exact / near-exact multicollinearity
3,pressure_cathode_outlet,1.0000000000,0.0000000000e+00,∞,Exact / near-exact multicollinearity
4,anode_dewpoint_offset,1.0000000000,0.0000000000e+00,∞,Exact / near-exact multicollinearity
5,temp_anode_dewpoint_water,1.0000000000,0.0000000000e+00,∞,Exact / near-exact multicollinearity
6,temp_anode_inlet,1.0000000000,0.0000000000e+00,∞,Exact / near-exact multicollinearity
7,temp_anode_outlet,1.0000000000,0.0000000000e+00,∞,Exact / near-exact multicollinearity
8,temp_cathode_inlet,1.0000000000,0.0000000000e+00,∞,Exact / near-exact multicollinearity
9,temp_cathode_dewpoint_water,1.0000000000,0.0000000000e+00,∞,Exact / near-exact multicollinearity


In [71]:
# ============================================================
# E1.7.6 Summarise Multicollinearity Levels
# ============================================================

vif_summary_df = (
    vif_results_df["Multicollinearity_Level"]
    .value_counts()
    .rename_axis("Multicollinearity_Level")
    .reset_index(name="Number_of_Predictors")
)

print("=" * 70)
print("VIF Multicollinearity Summary")
print("=" * 70)

display(vif_summary_df)

VIF Multicollinearity Summary


,Multicollinearity_Level,Number_of_Predictors
0,Exact / near-exact multicollinearity,16
1,Very high,3
2,Low,1


In [72]:
# ============================================================
# E1.7.7 Identify Exact / Near-Exact Multicollinearity
# ============================================================

exact_vif_df = vif_results_df[
    np.isinf(vif_results_df["VIF"])
].copy()

print("=" * 70)
print("Predictors with Exact / Near-Exact Multicollinearity")
print("=" * 70)

print(f"Number of predictors: {len(exact_vif_df)}")

if not exact_vif_df.empty:

    display(
        exact_vif_df[
            [
                "Predictor",
                "Auxiliary_R2",
                "Tolerance",
                "VIF"
            ]
        ].style.format({
            "Auxiliary_R2": "{:.10f}",
            "Tolerance": "{:.10e}",
            "VIF": lambda value:
                "∞" if np.isinf(value) else f"{value:.4f}"
        })
    )

Predictors with Exact / Near-Exact Multicollinearity
Number of predictors: 16


,Predictor,Auxiliary_R2,Tolerance,VIF
0,pressure_anode_inlet,1.0000000000,0.0000000000e+00,∞
1,pressure_anode_outlet,1.0000000000,0.0000000000e+00,∞
2,pressure_cathode_inlet,1.0000000000,0.0000000000e+00,∞
3,pressure_cathode_outlet,1.0000000000,0.0000000000e+00,∞
4,anode_dewpoint_offset,1.0000000000,0.0000000000e+00,∞
5,temp_anode_dewpoint_water,1.0000000000,0.0000000000e+00,∞
6,temp_anode_inlet,1.0000000000,0.0000000000e+00,∞
7,temp_anode_outlet,1.0000000000,0.0000000000e+00,∞
8,temp_cathode_inlet,1.0000000000,0.0000000000e+00,∞
9,temp_cathode_dewpoint_water,1.0000000000,0.0000000000e+00,∞


In [73]:
# ============================================================
# E1.7.8 Verify Engineered Feature Linear Dependencies
# ============================================================

dependency_checks = {

    "anode_pressure_diff":
        X_vif["pressure_anode_inlet"]
        - X_vif["pressure_anode_outlet"],

    "cathode_pressure_diff":
        X_vif["pressure_cathode_inlet"]
        - X_vif["pressure_cathode_outlet"],

    "anode_temp_diff":
        X_vif["temp_anode_outlet"]
        - X_vif["temp_anode_inlet"],

    "cathode_temp_diff":
        X_vif["temp_cathode_outlet"]
        - X_vif["temp_cathode_inlet"],

    "anode_dewpoint_offset":
        X_vif["temp_anode_inlet"]
        - X_vif["temp_anode_dewpoint_water"],

    "cathode_dewpoint_offset":
        X_vif["temp_cathode_inlet"]
        - X_vif["temp_cathode_dewpoint_water"]
}

dependency_records = []

DEPENDENCY_TOLERANCE = 1e-10

for engineered_feature, reconstructed_series in dependency_checks.items():

    actual_values = X_vif[engineered_feature].to_numpy()

    reconstructed_values = (
        reconstructed_series.to_numpy()
    )

    absolute_errors = np.abs(
        actual_values - reconstructed_values
    )

    maximum_error = np.max(absolute_errors)
    mean_error = np.mean(absolute_errors)

    dependency_records.append({
        "Engineered_Feature": engineered_feature,
        "Maximum_Reconstruction_Error": maximum_error,
        "Mean_Reconstruction_Error": mean_error,
        "Exact_Within_Tolerance":
            maximum_error < DEPENDENCY_TOLERANCE
    })

dependency_results_df = pd.DataFrame(
    dependency_records
)

print("=" * 70)
print("Engineered Feature Dependency Verification")
print("=" * 70)

display(
    dependency_results_df.style.format({
        "Maximum_Reconstruction_Error": "{:.12e}",
        "Mean_Reconstruction_Error": "{:.12e}"
    })
)

Engineered Feature Dependency Verification


,Engineered_Feature,Maximum_Reconstruction_Error,Mean_Reconstruction_Error,Exact_Within_Tolerance
0,anode_pressure_diff,1.776356839400e-15,2.288509732194e-17,True
1,cathode_pressure_diff,3.552713678801e-15,5.735033320422e-17,True
2,anode_temp_diff,7.105427357601e-15,1.087080027483e-15,True
3,cathode_temp_diff,3.552713678801e-15,4.619349511766e-16,True
4,anode_dewpoint_offset,3.552713678801e-15,5.031744904071e-16,True
5,cathode_dewpoint_offset,1.776356839400e-15,4.274521767254e-17,True


In [74]:
# ============================================================
# E1.7.9 Summarise Engineered Dependency Verification
# ============================================================

number_verified = int(
    dependency_results_df[
        "Exact_Within_Tolerance"
    ].sum()
)

number_checked = len(
    dependency_results_df
)

all_verified = (
    number_verified == number_checked
)

print("=" * 70)
print("Engineered Dependency Verification Summary")
print("=" * 70)

print(f"Engineered features checked : {number_checked}")
print(f"Exact dependencies verified : {number_verified}")
print(
    f"All engineered dependencies verified: "
    f"{all_verified}"
)

Engineered Dependency Verification Summary
Engineered features checked : 6
Exact dependencies verified : 6
All engineered dependencies verified: True


In [75]:
# ============================================================
# E1.7.10 Separate Finite and Infinite VIF Results
# ============================================================

infinite_vif_df = vif_results_df[
    np.isinf(vif_results_df["VIF"])
].copy()

finite_vif_df = vif_results_df[
    np.isfinite(vif_results_df["VIF"])
].copy()

print("=" * 70)
print("Finite / Infinite VIF Summary")
print("=" * 70)

print(
    f"Exact / near-exact multicollinearity : "
    f"{len(infinite_vif_df)} predictors"
)

print(
    f"Finite VIF                           : "
    f"{len(finite_vif_df)} predictors"
)

print("\nFinite VIF predictors:")

display(
    finite_vif_df[
        [
            "Predictor",
            "Auxiliary_R2",
            "Tolerance",
            "VIF",
            "Multicollinearity_Level"
        ]
    ].style.format({
        "Auxiliary_R2": "{:.10f}",
        "Tolerance": "{:.10e}",
        "VIF": "{:.4f}"
    })
)

Finite / Infinite VIF Summary
Exact / near-exact multicollinearity : 16 predictors
Finite VIF                           : 4 predictors

Finite VIF predictors:


,Predictor,Auxiliary_R2,Tolerance,VIF,Multicollinearity_Level
16,total_anode_stack_flow,0.9999797371,2.0262948889e-05,49351.1584,Very high
17,total_cathode_stack_flow,0.9999796922,2.0307826482e-05,49242.0989,Very high
18,current,0.9707283527,2.9271647293e-02,34.1628,Very high
19,temp_anode_endplate,0.4513310096,5.4866899041e-01,1.8226,Low


In [76]:
# ============================================================
# E1.7.11 Visualise Finite Variance Inflation Factors
# ============================================================

import matplotlib.pyplot as plt

finite_vif_plot_df = (
    finite_vif_df
    .sort_values("VIF", ascending=True)
)

plt.figure(figsize=(10, 6))

plt.barh(
    finite_vif_plot_df["Predictor"],
    finite_vif_plot_df["VIF"]
)

plt.axvline(
    5,
    linestyle="--",
    linewidth=1,
    label="VIF = 5"
)

plt.axvline(
    10,
    linestyle="--",
    linewidth=1,
    label="VIF = 10"
)

for index, row in finite_vif_plot_df.iterrows():

    plt.text(
        row["VIF"],
        row["Predictor"],
        f' {row["VIF"]:.2f}',
        va="center"
    )

plt.xlabel("Variance Inflation Factor (VIF)")
plt.ylabel("Candidate Predictor")
plt.title(
    "Finite VIF Values of Candidate Predictors"
)

plt.legend()
plt.tight_layout()
plt.show()

<Figure size 1000x600 with 1 Axes>

In [77]:
# ============================================================
# E1.7.12 Final VIF Diagnostic Summary
# ============================================================

print("=" * 70)
print("Final VIF Diagnostic Summary")
print("=" * 70)

print(f"Candidate predictors            : {len(candidate_predictors)}")
print(f"Predictor-matrix rank           : {matrix_rank}")
print(f"Predictor-matrix dimension      : {matrix_dimension}")
print(f"Rank deficiency                 : {rank_deficiency}")
print(f"Condition number                : {condition_number:.4e}")

print(
    f"Exact / near-exact VIF          : "
    f"{len(infinite_vif_df)}"
)

print(
    f"Finite VIF predictors           : "
    f"{len(finite_vif_df)}"
)

print(
    f"Verified engineered dependencies: "
    f"{number_verified}/{number_checked}"
)

print("\nInterpretation:")
print(
    "- The full candidate predictor set contains severe "
    "multicollinearity."
)

print(
    "- Exact mathematical dependencies are present because "
    "engineered features were retained together with their "
    "source variables."
)

print(
    "- Additional severe empirical collinearity may remain "
    "among finite-VIF predictors."
)

print(
    "- VIF is treated as diagnostic evidence and does not "
    "automatically determine feature removal."
)

Final VIF Diagnostic Summary
Candidate predictors            : 20
Predictor-matrix rank           : 18
Predictor-matrix dimension      : 20
Rank deficiency                 : 2
Condition number                : 4.9107e+14
Exact / near-exact VIF          : 16
Finite VIF predictors           : 4
Verified engineered dependencies: 6/6

Interpretation:
- The full candidate predictor set contains severe multicollinearity.
- Exact mathematical dependencies are present because engineered features were retained together with their source variables.
- Additional severe empirical collinearity may remain among finite-VIF predictors.
- VIF is treated as diagnostic evidence and does not automatically determine feature removal.


In [ ]:
## E1.6 — Preparation for XGBoost-Boruta Feature Selection

The preceding association and redundancy analyses were used as diagnostic
evidence rather than as automatic feature-elimination rules.

The formal feature-selection stage uses a canonical-Boruta-style framework
with XGBoost as the feature-importance estimator. This combines the
XGBoost–shadow-feature principle previously applied to PEMFC stack-voltage
feature selection with repeated shadow comparisons and statistical
confirmation/rejection logic.

Before configuring the XGBoost importance estimator, the eligible predictor
set and exact algebraic relationships introduced through feature engineering
are reviewed explicitly.

In [ ]:
### E1.6.1 — Confirm Candidate Predictors

The predictors that passed the earlier validity and leakage screening are
retrieved here. No predictor is removed on the basis of Pearson correlation,
Spearman correlation, Mutual Information, pairwise redundancy, or VIF alone.

This step confirms the predictor set available for formal feature selection.

In [78]:
# ============================================================
# E1.6.1 — Confirm Candidate Predictors
# ============================================================

print("Number of eligible predictors:", len(candidate_predictors))

print("\nEligible predictors:")
for i, feature in enumerate(candidate_predictors, start=1):
    print(f"{i:>2}. {feature}")

# Basic integrity checks
missing_predictors = [
    feature for feature in candidate_predictors
    if feature not in development_df.columns
]

duplicate_predictors = (
    len(candidate_predictors) != len(set(candidate_predictors))
)

print("\nIntegrity checks")
print("----------------")
print("Missing predictors:", missing_predictors)
print("Duplicate predictor names:", duplicate_predictors)
print("Target included:", "voltage" in candidate_predictors)
print("Power included:", "power" in candidate_predictors)
print("Time included:", "time" in candidate_predictors)
print("Operating hour included:", "operating_hour" in candidate_predictors)

Number of eligible predictors: 20

Eligible predictors:
 1. current
 2. pressure_anode_inlet
 3. pressure_anode_outlet
 4. pressure_cathode_inlet
 5. pressure_cathode_outlet
 6. temp_anode_endplate
 7. temp_anode_dewpoint_water
 8. temp_anode_inlet
 9. temp_anode_outlet
10. temp_cathode_dewpoint_water
11. temp_cathode_inlet
12. temp_cathode_outlet
13. total_anode_stack_flow
14. total_cathode_stack_flow
15. anode_pressure_diff
16. cathode_pressure_diff
17. anode_temp_diff
18. cathode_temp_diff
19. anode_dewpoint_offset
20. cathode_dewpoint_offset

Integrity checks
----------------
Missing predictors: []
Duplicate predictor names: False
Target included: False
Power included: False
Time included: False
Operating hour included: False


In [ ]:
### E1.6.2 — Exact Algebraic Feature Families

Several engineered predictors are deterministic transformations of original
operational variables. These relationships differ from ordinary statistical
correlation because one variable can be reconstructed exactly from others.

The algebraic feature families are identified before XGBoost-Boruta so that
deterministic redundancy can be distinguished from ordinary correlation.
No features are removed at this stage.

In [79]:
# ============================================================
# E1.6.2 — Exact Algebraic Feature Families
# ============================================================

algebraic_feature_families = {
    "Anode pressure": {
        "engineered": "anode_pressure_diff",
        "components": [
            "pressure_anode_inlet",
            "pressure_anode_outlet"
        ],
        "formula": "pressure_anode_inlet - pressure_anode_outlet"
    },

    "Cathode pressure": {
        "engineered": "cathode_pressure_diff",
        "components": [
            "pressure_cathode_inlet",
            "pressure_cathode_outlet"
        ],
        "formula": "pressure_cathode_inlet - pressure_cathode_outlet"
    },

    "Anode temperature": {
        "engineered": "anode_temp_diff",
        "components": [
            "temp_anode_outlet",
            "temp_anode_inlet"
        ],
        "formula": "temp_anode_outlet - temp_anode_inlet"
    },

    "Cathode temperature": {
        "engineered": "cathode_temp_diff",
        "components": [
            "temp_cathode_outlet",
            "temp_cathode_inlet"
        ],
        "formula": "temp_cathode_outlet - temp_cathode_inlet"
    },

    "Anode dewpoint": {
        "engineered": "anode_dewpoint_offset",
        "components": [
            "temp_anode_inlet",
            "temp_anode_dewpoint_water"
        ],
        "formula": "temp_anode_inlet - temp_anode_dewpoint_water"
    },

    "Cathode dewpoint": {
        "engineered": "cathode_dewpoint_offset",
        "components": [
            "temp_cathode_inlet",
            "temp_cathode_dewpoint_water"
        ],
        "formula": "temp_cathode_inlet - temp_cathode_dewpoint_water"
    }
}

print("Exact algebraic feature families")
print("================================")

for family, info in algebraic_feature_families.items():
    print(f"\n{family}")
    print(f"  Components : {', '.join(info['components'])}")
    print(f"  Engineered : {info['engineered']}")
    print(f"  Formula    : {info['engineered']} = {info['formula']}")

print(
    "\nNumber of engineered predictors with exact "
    f"algebraic dependencies: {len(algebraic_feature_families)}"
)

Exact algebraic feature families

Anode pressure
  Components : pressure_anode_inlet, pressure_anode_outlet
  Engineered : anode_pressure_diff
  Formula    : anode_pressure_diff = pressure_anode_inlet - pressure_anode_outlet

Cathode pressure
  Components : pressure_cathode_inlet, pressure_cathode_outlet
  Engineered : cathode_pressure_diff
  Formula    : cathode_pressure_diff = pressure_cathode_inlet - pressure_cathode_outlet

Anode temperature
  Components : temp_anode_outlet, temp_anode_inlet
  Engineered : anode_temp_diff
  Formula    : anode_temp_diff = temp_anode_outlet - temp_anode_inlet

Cathode temperature
  Components : temp_cathode_outlet, temp_cathode_inlet
  Engineered : cathode_temp_diff
  Formula    : cathode_temp_diff = temp_cathode_outlet - temp_cathode_inlet

Anode dewpoint
  Components : temp_anode_inlet, temp_anode_dewpoint_water
  Engineered : anode_dewpoint_offset
  Formula    : anode_dewpoint_offset = temp_anode_inlet - temp_anode_dewpoint_water

Cathode dewpoint

In [ ]:
### E1.6.3 — Pre-Boruta Redundancy Decision

The exact algebraic relationships identified above are documented explicitly,
but no predictor is removed before XGBoost-Boruta.

This decision separates deterministic redundancy assessment from formal
feature relevance selection.

The rationale is:

1. Ordinary correlation is not used as a hard pre-selection rule.
2. VIF is treated as a multicollinearity diagnostic rather than a deletion rule.
3. Choosing one member of an algebraically related family before Boruta would
   require either an arbitrary preference or empirical evidence from the
   development data.
4. XGBoost can operate with correlated and deterministically related predictors
   because it does not require a full-rank design matrix.
5. The formal Boruta procedure will therefore evaluate all valid predictor
   representations against randomized shadow features.
6. Redundancy will be reconsidered after Boruta when parsimonious candidate
   feature sets are constructed and compared using chronological validation.

Therefore, all 20 valid non-leaking predictors are retained as candidates for
the formal XGBoost-Boruta procedure.

In [ ]:
### E1.6.4 — Define and Freeze Boruta Input Predictor Set

Following validity screening and explicit documentation of deterministic
algebraic dependencies, no additional predictor is removed before the formal
feature-selection procedure.

The Boruta input set therefore contains all 20 valid non-leaking predictors.
This fixed predictor set will be used consistently when configuring and
implementing the XGBoost-Boruta procedure.

Association, redundancy, and multicollinearity diagnostics do not directly
alter this input set.

In [81]:
# ============================================================
# E1.6.4 — Define and Freeze Boruta Input Predictor Set
# ============================================================

boruta_candidate_predictors = candidate_predictors.copy()

print("XGBoost-Boruta Input Predictor Set")
print("==================================")

print(
    f"Number of predictors entering Boruta: "
    f"{len(boruta_candidate_predictors)}"
)

print("\nPredictors:")
for i, feature in enumerate(boruta_candidate_predictors, start=1):
    print(f"{i:>2}. {feature}")

# Integrity checks
assert len(boruta_candidate_predictors) == 20
assert len(boruta_candidate_predictors) == len(
    set(boruta_candidate_predictors)
)
assert "voltage" not in boruta_candidate_predictors
assert "power" not in boruta_candidate_predictors
assert "time" not in boruta_candidate_predictors
assert "operating_hour" not in boruta_candidate_predictors

print("\nIntegrity checks passed.")
print("Pre-Boruta predictor removal: None")
print("Boruta input set frozen: True")

XGBoost-Boruta Input Predictor Set
Number of predictors entering Boruta: 20

Predictors:
 1. current
 2. pressure_anode_inlet
 3. pressure_anode_outlet
 4. pressure_cathode_inlet
 5. pressure_cathode_outlet
 6. temp_anode_endplate
 7. temp_anode_dewpoint_water
 8. temp_anode_inlet
 9. temp_anode_outlet
10. temp_cathode_dewpoint_water
11. temp_cathode_inlet
12. temp_cathode_outlet
13. total_anode_stack_flow
14. total_cathode_stack_flow
15. anode_pressure_diff
16. cathode_pressure_diff
17. anode_temp_diff
18. cathode_temp_diff
19. anode_dewpoint_offset
20. cathode_dewpoint_offset

Integrity checks passed.
Pre-Boruta predictor removal: None
Boruta input set frozen: True


In [ ]:
## E1.7 — XGBoost Importance-Estimator Configuration

Before implementing the formal XGBoost-Boruta procedure, a fixed XGBoost
configuration is established for use as the feature-importance estimator.

This is not intended to be full predictive hyperparameter optimization.
Instead, the objective is to identify a configuration that provides:

- stable feature-importance estimates across repeated fits,
- adequate predictive behaviour,
- computational practicality, and
- consistency with published PEMFC XGBoost-Boruta methodology where possible.

Once selected, the same configuration will be fixed for all Boruta iterations
so that changes in feature importance primarily reflect shadow-feature
randomization rather than changes in model settings.

In [ ]:
### E1.7.1 — Define Candidate XGBoost Configurations

A small set of XGBoost configurations is defined to assess the stability of
feature-importance estimation before the formal Boruta procedure.

The purpose is not to optimize the final predictive model. Instead, the
comparison evaluates whether reasonable changes in tree complexity produce
stable feature-importance patterns.

The configurations use squared-error regression and gain-based feature
importance. The number of boosting trees is fixed at 50 and gamma at 0,
providing a literature-informed reference point from previous PEMFC
XGBoost-Boruta research.

Only a limited range of tree depth and learning rate is examined to avoid
turning this stage into full hyperparameter optimization.

In [82]:
# ============================================================
# E1.7.1 — Define Candidate XGBoost Configurations
# ============================================================

xgb_boruta_configurations = {
    "Config_A": {
        "max_depth": 3,
        "learning_rate": 0.05,
        "min_child_weight": 1
    },

    "Config_B": {
        "max_depth": 3,
        "learning_rate": 0.10,
        "min_child_weight": 1
    },

    "Config_C": {
        "max_depth": 6,
        "learning_rate": 0.05,
        "min_child_weight": 1
    },

    "Config_D": {
        "max_depth": 6,
        "learning_rate": 0.10,
        "min_child_weight": 1
    }
}

xgb_boruta_fixed_params = {
    "objective": "reg:squarederror",
    "n_estimators": 50,
    "gamma": 0,
    "subsample": 1.0,
    "colsample_bytree": 1.0,
    "tree_method": "hist",
    "importance_type": "gain",
    "n_jobs": -1,
    "verbosity": 0
}

print("Candidate XGBoost configurations")
print("================================")

for name, params in xgb_boruta_configurations.items():
    print(f"\n{name}")
    for parameter, value in params.items():
        print(f"  {parameter}: {value}")

print("\nFixed parameters")
print("----------------")
for parameter, value in xgb_boruta_fixed_params.items():
    print(f"{parameter}: {value}")

print(
    f"\nNumber of candidate configurations: "
    f"{len(xgb_boruta_configurations)}"
)

Candidate XGBoost configurations

Config_A
  max_depth: 3
  learning_rate: 0.05
  min_child_weight: 1

Config_B
  max_depth: 3
  learning_rate: 0.1
  min_child_weight: 1

Config_C
  max_depth: 6
  learning_rate: 0.05
  min_child_weight: 1

Config_D
  max_depth: 6
  learning_rate: 0.1
  min_child_weight: 1

Fixed parameters
----------------
objective: reg:squarederror
n_estimators: 50
gamma: 0
subsample: 1.0
colsample_bytree: 1.0
tree_method: hist
importance_type: gain
n_jobs: -1
verbosity: 0

Number of candidate configurations: 4


In [ ]:
### E1.7.2 — XGBoost Configuration Assessment Framework

The candidate XGBoost configurations are evaluated to select a stable
feature-importance estimator for the subsequent Boruta procedure.

This stage is not a full predictive hyperparameter optimization. The objective
is to identify a configuration that provides sufficiently accurate and stable
feature-importance estimation.

Each candidate configuration will therefore be assessed using:

1. Predictive adequacy:
   RMSE, MAE, and R² are used to confirm that the estimator can adequately
   represent the relationship between the candidate predictors and stack
   voltage.

2. Feature-importance stability:
   repeated model fits are used to determine whether gain-based feature
   importance and feature rankings remain reasonably consistent across
   repeated runs.

3. Computational practicality:
   training time is considered because the selected estimator will subsequently
   be fitted repeatedly during the Boruta procedure.

The configuration is selected by considering these criteria jointly rather
than simply choosing the configuration with the lowest prediction error.

Only development-training data are used for this configuration assessment.
The final test stages remain completely untouched.

After selection, the chosen XGBoost configuration is frozen and used
consistently throughout the formal XGBoost-Boruta procedure.

In [ ]:
### E1.7.3 — Define Training-Only Configuration Assessment Data

The XGBoost importance-estimator configuration is assessed using only the
earliest common training period represented by durability-stage labels
50–450.

These observations constitute the training set of the first chronological
outer fold and are also contained within the training sets of all subsequent
outer folds.

Restricting configuration assessment to this common training period prevents
the outer-validation stages from influencing the choice of XGBoost importance
estimator.

The stage labels 500–850 therefore remain outside this configuration-selection
step, while the final-test stage labels 900–1000 remain completely untouched.

The 50–450 training period will subsequently be divided chronologically for
the configuration assessment rather than using a random train-validation split.

In [83]:
# ============================================================
# E1.7.3 — Define Training-Only Configuration Assessment Data
# ============================================================

configuration_stage_labels = [
    50, 100, 150, 200, 250,
    300, 350, 400, 450
]

configuration_df = development_df[
    development_df["operating_hour"].isin(configuration_stage_labels)
].copy()

X_configuration = configuration_df[boruta_candidate_predictors].copy()
y_configuration = configuration_df["voltage"].copy()

print("XGBoost Configuration Assessment Data")
print("=====================================")

print("Stage labels used:")
print(configuration_stage_labels)

print(f"\nNumber of stages: {len(configuration_stage_labels)}")
print(f"Number of observations: {len(configuration_df):,}")

print(f"\nX shape: {X_configuration.shape}")
print(f"y shape: {y_configuration.shape}")

print("\nIntegrity checks")
print("----------------")
print(
    "Only requested stages:",
    sorted(configuration_df["operating_hour"].unique().tolist())
    == configuration_stage_labels
)

print(
    "Outer validation stages present:",
    configuration_df["operating_hour"]
    .isin([500, 550, 600, 650, 700, 750, 800, 850])
    .any()
)

print(
    "Final test stages present:",
    configuration_df["operating_hour"]
    .isin([900, 950, 1000])
    .any()
)

print("Missing X values:", X_configuration.isna().sum().sum())
print("Missing y values:", y_configuration.isna().sum())

XGBoost Configuration Assessment Data
Stage labels used:
[50, 100, 150, 200, 250, 300, 350, 400, 450]

Number of stages: 9
Number of observations: 1,614,240

X shape: (1614240, 20)
y shape: (1614240,)

Integrity checks
----------------
Only requested stages: True
Outer validation stages present: False
Final test stages present: False
Missing X values: 0
Missing y values: 0


In [ ]:
### E1.7.4 — Internal Chronological Configuration Split

The training-only configuration dataset is divided chronologically to assess
the candidate XGBoost importance-estimator configurations without using the
outer-validation periods.

Stage labels 50–350 are used for fitting the candidate configurations, while
stage labels 400–450 form an internal configuration-validation period.

This preserves temporal ordering and ensures that predictive adequacy is
assessed on later durability stages than those used for fitting.

The internal validation period is used only to select the fixed XGBoost
importance-estimator configuration. It is not part of the outer chronological
validation used later to compare selected feature sets.

In [84]:
# ============================================================
# E1.7.4 — Internal Chronological Configuration Split
# ============================================================

config_train_labels = [
    50, 100, 150, 200, 250, 300, 350
]

config_validation_labels = [
    400, 450
]

config_train_df = configuration_df[
    configuration_df["operating_hour"].isin(config_train_labels)
].copy()

config_validation_df = configuration_df[
    configuration_df["operating_hour"].isin(config_validation_labels)
].copy()

X_config_train = config_train_df[boruta_candidate_predictors].copy()
y_config_train = config_train_df["voltage"].copy()

X_config_validation = config_validation_df[
    boruta_candidate_predictors
].copy()

y_config_validation = config_validation_df["voltage"].copy()


print("Internal Chronological Configuration Split")
print("==========================================")

print("\nTraining stage labels:")
print(config_train_labels)

print("\nConfiguration-validation stage labels:")
print(config_validation_labels)

print("\nDataset sizes")
print("-------------")
print(f"Training observations: {len(config_train_df):,}")
print(f"Validation observations: {len(config_validation_df):,}")

print(f"\nX_train shape: {X_config_train.shape}")
print(f"y_train shape: {y_config_train.shape}")

print(f"X_validation shape: {X_config_validation.shape}")
print(f"y_validation shape: {y_config_validation.shape}")


# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

train_stages = set(
    config_train_df["operating_hour"].unique()
)

validation_stages = set(
    config_validation_df["operating_hour"].unique()
)

print("\nIntegrity checks")
print("----------------")

print(
    "Train/validation stage overlap:",
    len(train_stages.intersection(validation_stages)) > 0
)

print(
    "Chronological ordering preserved:",
    max(config_train_labels) < min(config_validation_labels)
)

print(
    "Outer validation stages present:",
    configuration_df["operating_hour"]
    .isin([500, 550, 600, 650, 700, 750, 800, 850])
    .any()
)

print(
    "Final test stages present:",
    configuration_df["operating_hour"]
    .isin([900, 950, 1000])
    .any()
)

Internal Chronological Configuration Split

Training stage labels:
[50, 100, 150, 200, 250, 300, 350]

Configuration-validation stage labels:
[400, 450]

Dataset sizes
-------------
Training observations: 1,255,520
Validation observations: 358,720

X_train shape: (1255520, 20)
y_train shape: (1255520,)
X_validation shape: (358720, 20)
y_validation shape: (358720,)

Integrity checks
----------------
Train/validation stage overlap: False
Chronological ordering preserved: True
Outer validation stages present: False
Final test stages present: False


In [ ]:
### E1.7.5 — Repeated-Seed Configuration Assessment Design

Feature-importance stability is assessed through repeated XGBoost fits under
controlled stochastic variation.

Because full row and feature sampling can make repeated XGBoost fits nearly
deterministic, subsample and colsample_bytree are set to 0.8 during this
assessment. Different random seeds therefore generate modest variation in the
training process while preserving the same data, model family, and candidate
predictor set.

Each candidate configuration is fitted repeatedly using the same predefined
random seeds.

For each fit, the following are recorded:

- RMSE on the internal chronological validation period,
- MAE,
- R²,
- gain-based importance for each predictor, and
- feature-importance ranking.

Configuration assessment therefore considers both predictive adequacy and
the reproducibility of feature importance under controlled model variation.

In [85]:
# ============================================================
# E1.7.5 — Repeated-Seed Configuration Assessment Design
# ============================================================

configuration_seeds = [
    11, 22, 33, 44, 55,
    66, 77, 88, 99, 110
]

# Controlled stochasticity for stability assessment
xgb_boruta_fixed_params["subsample"] = 0.8
xgb_boruta_fixed_params["colsample_bytree"] = 0.8

print("Repeated-Seed Configuration Assessment")
print("======================================")

print(f"Number of repeated fits per configuration: {len(configuration_seeds)}")
print("Random seeds:")
print(configuration_seeds)

print("\nSampling parameters used for stability assessment")
print("-------------------------------------------------")
print("subsample:", xgb_boruta_fixed_params["subsample"])
print(
    "colsample_bytree:",
    xgb_boruta_fixed_params["colsample_bytree"]
)

print("\nCandidate configurations:", len(xgb_boruta_configurations))

total_fits = (
    len(configuration_seeds)
    * len(xgb_boruta_configurations)
)

print("Total XGBoost fits required:", total_fits)

Repeated-Seed Configuration Assessment
Number of repeated fits per configuration: 10
Random seeds:
[11, 22, 33, 44, 55, 66, 77, 88, 99, 110]

Sampling parameters used for stability assessment
-------------------------------------------------
subsample: 0.8
colsample_bytree: 0.8

Candidate configurations: 4
Total XGBoost fits required: 40


In [ ]:
### E1.7.6 — Run Repeated XGBoost Configuration Assessment

Each candidate XGBoost configuration is fitted ten times using the predefined
random seeds.

Models are fitted using stage labels 50–350 and evaluated on the internal
chronological validation period represented by stage labels 400–450.

For every repeated fit, predictive performance, computational time, and
gain-based feature importance are recorded.

No configuration is selected at this stage. The resulting evidence is
subsequently summarized to compare predictive adequacy and feature-importance
stability across configurations.

In [86]:
# ============================================================
# E1.7.6 — Run Repeated XGBoost Configuration Assessment
# ============================================================

import time
import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


performance_records = []
importance_records = []


for config_name, config_params in xgb_boruta_configurations.items():

    print(f"\nRunning {config_name}")
    print("-" * 50)

    for run_number, seed in enumerate(configuration_seeds, start=1):

        # ----------------------------------------------------
        # Build model parameters
        # ----------------------------------------------------

        model_params = {
            **xgb_boruta_fixed_params,
            **config_params,
            "random_state": seed
        }

        model = XGBRegressor(**model_params)

        # ----------------------------------------------------
        # Fit model
        # ----------------------------------------------------

        start_time = time.perf_counter()

        model.fit(
            X_config_train,
            y_config_train
        )

        fit_time = time.perf_counter() - start_time

        # ----------------------------------------------------
        # Chronological validation prediction
        # ----------------------------------------------------

        y_pred = model.predict(X_config_validation)

        rmse = np.sqrt(
            mean_squared_error(
                y_config_validation,
                y_pred
            )
        )

        mae = mean_absolute_error(
            y_config_validation,
            y_pred
        )

        r2 = r2_score(
            y_config_validation,
            y_pred
        )

        # ----------------------------------------------------
        # Store predictive performance
        # ----------------------------------------------------

        performance_records.append({
            "configuration": config_name,
            "run": run_number,
            "seed": seed,
            "rmse": rmse,
            "mae": mae,
            "r2": r2,
            "fit_time_seconds": fit_time
        })

        # ----------------------------------------------------
        # Extract gain importance
        # ----------------------------------------------------

        booster = model.get_booster()

        gain_scores = booster.get_score(
            importance_type="gain"
        )

        feature_gains = {
            feature: gain_scores.get(feature, 0.0)
            for feature in boruta_candidate_predictors
        }

        # Rank: 1 = highest gain
        feature_ranks = pd.Series(
            feature_gains
        ).rank(
            ascending=False,
            method="min"
        )

        for feature in boruta_candidate_predictors:

            importance_records.append({
                "configuration": config_name,
                "run": run_number,
                "seed": seed,
                "feature": feature,
                "gain": feature_gains[feature],
                "rank": feature_ranks[feature]
            })

        print(
            f"Run {run_number:>2}/10 | "
            f"Seed {seed:>3} | "
            f"RMSE = {rmse:.6f} | "
            f"R² = {r2:.6f} | "
            f"Time = {fit_time:.2f}s"
        )


# ============================================================
# Convert results to DataFrames
# ============================================================

xgb_config_performance = pd.DataFrame(
    performance_records
)

xgb_config_importance = pd.DataFrame(
    importance_records
)


print("\n" + "=" * 60)
print("Configuration assessment complete")
print("=" * 60)

print(
    "Performance records:",
    xgb_config_performance.shape
)

print(
    "Importance records:",
    xgb_config_importance.shape
)

print(
    "\nExpected performance records:",
    len(xgb_boruta_configurations)
    * len(configuration_seeds)
)

print(
    "Expected importance records:",
    len(xgb_boruta_configurations)
    * len(configuration_seeds)
    * len(boruta_candidate_predictors)
)


Running Config_A
--------------------------------------------------
Run  1/10 | Seed  11 | RMSE = 0.022755 | R² = 0.945609 | Time = 3.95s
Run  2/10 | Seed  22 | RMSE = 0.022781 | R² = 0.945483 | Time = 4.13s
Run  3/10 | Seed  33 | RMSE = 0.022929 | R² = 0.944773 | Time = 4.12s
Run  4/10 | Seed  44 | RMSE = 0.023013 | R² = 0.944368 | Time = 4.12s
Run  5/10 | Seed  55 | RMSE = 0.022963 | R² = 0.944609 | Time = 4.12s
Run  6/10 | Seed  66 | RMSE = 0.022903 | R² = 0.944900 | Time = 4.11s
Run  7/10 | Seed  77 | RMSE = 0.022963 | R² = 0.944607 | Time = 4.04s
Run  8/10 | Seed  88 | RMSE = 0.022958 | R² = 0.944631 | Time = 4.04s
Run  9/10 | Seed  99 | RMSE = 0.022994 | R² = 0.944461 | Time = 3.89s
Run 10/10 | Seed 110 | RMSE = 0.022976 | R² = 0.944547 | Time = 4.13s

Running Config_B
--------------------------------------------------
Run  1/10 | Seed  11 | RMSE = 0.021661 | R² = 0.950714 | Time = 4.15s
Run  2/10 | Seed  22 | RMSE = 0.022020 | R² = 0.949063 | Time = 4.32s
Run  3/10 | Seed  33 |

In [ ]:
### E1.7.7 — Predictive Adequacy Across Configurations

Predictive performance across the repeated fits is summarized for each
candidate configuration.

Mean performance represents the typical predictive behaviour of each
configuration, while standard deviation quantifies sensitivity to the
controlled stochastic variation introduced through row and feature
subsampling.

These results are used to establish predictive adequacy rather than to
identify the final predictive XGBoost model. Configuration selection will
also consider feature-importance stability.

In [87]:
# ============================================================
# E1.7.7 — Summarize Predictive Adequacy
# ============================================================

xgb_config_performance_summary = (
    xgb_config_performance
    .groupby("configuration")
    .agg(
        mean_rmse=("rmse", "mean"),
        sd_rmse=("rmse", "std"),
        mean_mae=("mae", "mean"),
        sd_mae=("mae", "std"),
        mean_r2=("r2", "mean"),
        sd_r2=("r2", "std"),
        mean_fit_time=("fit_time_seconds", "mean"),
        sd_fit_time=("fit_time_seconds", "std")
    )
    .reset_index()
)

# Coefficient of variation of RMSE
xgb_config_performance_summary["rmse_cv_percent"] = (
    xgb_config_performance_summary["sd_rmse"]
    / xgb_config_performance_summary["mean_rmse"]
    * 100
)

# Sort by mean validation RMSE
xgb_config_performance_summary = (
    xgb_config_performance_summary
    .sort_values("mean_rmse")
    .reset_index(drop=True)
)

print("Predictive Adequacy Summary")
print("===========================")

print(
    xgb_config_performance_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

Predictive Adequacy Summary
configuration  mean_rmse  sd_rmse  mean_mae   sd_mae  mean_r2    sd_r2  mean_fit_time  sd_fit_time  rmse_cv_percent
     Config_B   0.022117 0.000390  0.018275 0.000370 0.948602 0.001829       4.246933     0.216910         1.765336
     Config_D   0.022534 0.000135  0.016792 0.000121 0.946659 0.000637       5.565818     0.270588         0.598298
     Config_A   0.022923 0.000088  0.017783 0.000085 0.944799 0.000421       4.065516     0.086036         0.382525
     Config_C   0.024783 0.000083  0.017820 0.000088 0.935482 0.000431       5.819199     0.549545         0.334087


In [ ]:
### E1.7.8 — Feature-Importance Ranking Stability

Feature-importance stability is evaluated by comparing gain-based feature
rankings across the ten repeated fits of each candidate configuration.

For each configuration, pairwise Spearman rank correlations are calculated
between the feature rankings obtained from every pair of repeated runs.

A high mean Spearman correlation indicates that the configuration produces
a reproducible ordering of predictor importance despite controlled stochastic
variation in XGBoost training.

The minimum and standard deviation of pairwise correlations are also reported
to identify configurations whose average stability may conceal individual
unstable runs.

This analysis complements predictive adequacy and is particularly important
because the selected XGBoost configuration will subsequently act as the
importance estimator within the Boruta procedure.

In [89]:
# ============================================================
# E1.7.8 — Feature-Importance Ranking Stability
# ============================================================

from itertools import combinations
from scipy.stats import spearmanr

rank_stability_records = []

for config_name in xgb_boruta_configurations.keys():

    config_importance = xgb_config_importance[
        xgb_config_importance["configuration"] == config_name
    ]

    # Rows = features, columns = repeated runs
    rank_matrix = config_importance.pivot(
        index="feature",
        columns="run",
        values="rank"
    )

    pairwise_correlations = []

    for run_a, run_b in combinations(rank_matrix.columns, 2):

        rho, _ = spearmanr(
            rank_matrix[run_a],
            rank_matrix[run_b]
        )

        pairwise_correlations.append(rho)

    rank_stability_records.append({
        "configuration": config_name,
        "mean_rank_spearman": np.mean(pairwise_correlations),
        "sd_rank_spearman": np.std(
            pairwise_correlations,
            ddof=1
        ),
        "min_rank_spearman": np.min(pairwise_correlations),
        "max_rank_spearman": np.max(pairwise_correlations),
        "n_pairwise_comparisons": len(pairwise_correlations)
    })


xgb_rank_stability_summary = pd.DataFrame(
    rank_stability_records
).sort_values(
    "mean_rank_spearman",
    ascending=False
).reset_index(drop=True)


print("Feature-Importance Ranking Stability")
print("====================================")

print(
    xgb_rank_stability_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

Feature-Importance Ranking Stability
configuration  mean_rank_spearman  sd_rank_spearman  min_rank_spearman  max_rank_spearman  n_pairwise_comparisons
     Config_D            0.946232          0.024322           0.884211           0.989474                      45
     Config_C            0.935138          0.041038           0.831579           0.989474                      45
     Config_B            0.922401          0.039016           0.831639           0.980753                      45
     Config_A            0.906427          0.054315           0.775966           0.996396                      45


In [ ]:
### E1.7.9 — Select and Freeze XGBoost-Boruta Configuration

Config D was selected as the fixed XGBoost importance estimator for the
subsequent Boruta procedure.

The selection does not imply that Config D is the globally optimal XGBoost
predictive model. Instead, it was selected from the predefined candidate
configurations because it provided the strongest feature-importance ranking
stability while maintaining adequate chronological validation performance.

Across repeated stochastic fits, Config D achieved a mean pairwise Spearman
correlation of approximately 0.946 between feature-importance rankings,
together with a mean validation RMSE of approximately 0.0225 V and mean
R² of approximately 0.947.

Because the purpose of XGBoost at this stage is feature-importance estimation
rather than final voltage-model optimization, reproducibility of feature
importance was prioritized once adequate predictive performance had been
established.

The selected configuration is frozen before the formal XGBoost-Boruta
procedure and will remain unchanged across Boruta iterations.

In [90]:
# ============================================================
# E1.7.9 — Select and Freeze XGBoost-Boruta Configuration
# ============================================================

selected_xgb_boruta_config = {
    **xgb_boruta_fixed_params,
    **xgb_boruta_configurations["Config_D"]
}

selected_xgb_boruta_config["random_state"] = 42

print("Selected XGBoost-Boruta Importance Estimator")
print("============================================")

print("Selected configuration: Config_D")

print("\nFixed parameters:")
for parameter, value in selected_xgb_boruta_config.items():
    print(f"{parameter}: {value}")

print("\nSelection evidence")
print("------------------")

config_d_performance = (
    xgb_config_performance_summary[
        xgb_config_performance_summary["configuration"] == "Config_D"
    ].iloc[0]
)

config_d_stability = (
    xgb_rank_stability_summary[
        xgb_rank_stability_summary["configuration"] == "Config_D"
    ].iloc[0]
)

print(f"Mean RMSE: {config_d_performance['mean_rmse']:.6f} V")
print(f"Mean MAE:  {config_d_performance['mean_mae']:.6f} V")
print(f"Mean R²:   {config_d_performance['mean_r2']:.6f}")

print(
    "Mean rank stability: "
    f"{config_d_stability['mean_rank_spearman']:.6f}"
)

print(
    "Minimum rank stability: "
    f"{config_d_stability['min_rank_spearman']:.6f}"
)

print("\nConfiguration frozen for Boruta: True")

Selected XGBoost-Boruta Importance Estimator
Selected configuration: Config_D

Fixed parameters:
objective: reg:squarederror
n_estimators: 50
gamma: 0
subsample: 0.8
colsample_bytree: 0.8
tree_method: hist
importance_type: gain
n_jobs: -1
verbosity: 0
max_depth: 6
learning_rate: 0.1
min_child_weight: 1
random_state: 42

Selection evidence
------------------
Mean RMSE: 0.022534 V
Mean MAE:  0.016792 V
Mean R²:   0.946659
Mean rank stability: 0.946232
Minimum rank stability: 0.884211

Configuration frozen for Boruta: True


In [ ]:
### E1.8.1 — Formal XGBoost-Boruta Decision Framework

The formal feature-selection procedure combines XGBoost gain importance with
canonical Boruta-style shadow-feature and statistical decision logic.

For each Boruta iteration:

1. One shadow copy of every real predictor is created by independently
   permuting its observations within the current training data.

2. The real and shadow predictors are combined and supplied simultaneously
   to one XGBoost model using the fixed configuration selected in E1.7.

3. Gain-based feature importance is calculated for all real and shadow
   predictors.

4. The maximum gain obtained by any shadow feature is used as the iteration's
   random-information benchmark.

5. A real predictor records a hit when its gain exceeds the maximum shadow
   gain.

The process is repeated across multiple independent shadow permutations.

After the repeated iterations, each predictor's accumulated hit count is
compared with the hit frequency expected under the Boruta null hypothesis.
Statistical testing is then used to classify predictors as:

- Confirmed: consistently performs better than randomized shadow information.
- Rejected: consistently performs worse than the shadow benchmark.
- Tentative: available evidence is insufficient for confirmation or rejection.

Multiple-testing control is applied when making simultaneous feature-level
decisions.

This implementation is a canonical-Boruta-style XGBoost adaptation rather
than an exact reproduction of the XGBoost-Boruta Z-score formulation reported
in previous PEMFC research.

In [91]:
# ============================================================
# E1.8.1 — Formal XGBoost-Boruta Decision Framework
# ============================================================

boruta_settings = {
    "n_iterations": 50,
    "alpha": 0.05,
    "importance_type": "gain",
    "shadow_threshold": "maximum_shadow_gain",
    "decision_logic": "canonical_boruta_style",
    "multiple_testing": "bonferroni",
    "base_random_seed": 42
}

print("XGBoost-Boruta Decision Framework")
print("=================================")

print(f"Number of real predictors: {len(boruta_candidate_predictors)}")
print(f"Boruta iterations: {boruta_settings['n_iterations']}")
print(f"Significance level: {boruta_settings['alpha']}")

print(
    "Shadow benchmark:",
    boruta_settings["shadow_threshold"]
)

print(
    "Feature importance:",
    boruta_settings["importance_type"]
)

print(
    "Decision logic:",
    boruta_settings["decision_logic"]
)

print(
    "Multiple-testing control:",
    boruta_settings["multiple_testing"]
)

print("\nHit definition:")
print(
    "real feature gain > maximum shadow-feature gain"
)

print("\nPossible final states:")
print("Confirmed")
print("Tentative")
print("Rejected")

XGBoost-Boruta Decision Framework
Number of real predictors: 20
Boruta iterations: 50
Significance level: 0.05
Shadow benchmark: maximum_shadow_gain
Feature importance: gain
Decision logic: canonical_boruta_style
Multiple-testing control: bonferroni

Hit definition:
real feature gain > maximum shadow-feature gain

Possible final states:
Confirmed
Tentative
Rejected


In [ ]:
### E1.8.2 — Statistical Hit-Count Decision Rule

Each Boruta iteration produces a binary outcome for every real predictor:

- Hit = 1 if the predictor's XGBoost gain exceeds the maximum gain of all
  shadow features in that iteration.
- Hit = 0 otherwise.

Under the Boruta null hypothesis, a predictor that is no more informative
than randomized shadow information is treated as having a hit probability
of 0.5.

After 50 iterations, the accumulated hit count for each predictor is therefore
evaluated using a binomial distribution with:

- number of trials = 50
- null hit probability = 0.5

Two one-sided tests are used:

1. Confirmation test:
   tests whether the observed number of hits is significantly greater than
   expected under the null hypothesis.

2. Rejection test:
   tests whether the observed number of hits is significantly lower than
   expected under the null hypothesis.

Because 20 predictors are assessed simultaneously, Bonferroni correction is
applied using the predefined significance level α = 0.05.

A predictor is classified as:

- Confirmed if its upper-tail corrected p-value is significant.
- Rejected if its lower-tail corrected p-value is significant.
- Tentative otherwise.

The statistical decision rule is fixed before the Boruta feature-selection
results are examined.

In [92]:
# ============================================================
# E1.8.2 — Statistical Hit-Count Decision Rule
# ============================================================

from scipy.stats import binom

n_boruta_iterations = boruta_settings["n_iterations"]
boruta_alpha = boruta_settings["alpha"]
n_boruta_features = len(boruta_candidate_predictors)

# Bonferroni-adjusted significance threshold
boruta_alpha_corrected = boruta_alpha / n_boruta_features


def boruta_hit_pvalues(
    hits,
    n_iterations=n_boruta_iterations,
    null_probability=0.5
):
    """
    Calculate one-sided binomial p-values for a Boruta hit count.

    Confirmation:
        P(X >= observed hits)

    Rejection:
        P(X <= observed hits)

    under X ~ Binomial(n_iterations, 0.5).
    """

    # Upper-tail probability, inclusive of observed hit count
    p_confirm = binom.sf(
        hits - 1,
        n_iterations,
        null_probability
    )

    # Lower-tail probability, inclusive of observed hit count
    p_reject = binom.cdf(
        hits,
        n_iterations,
        null_probability
    )

    return p_confirm, p_reject


def classify_boruta_hits(hits):
    """
    Convert an accumulated hit count into a Boruta decision
    using Bonferroni-corrected one-sided binomial tests.
    """

    p_confirm, p_reject = boruta_hit_pvalues(hits)

    if p_confirm < boruta_alpha_corrected:
        decision = "Confirmed"

    elif p_reject < boruta_alpha_corrected:
        decision = "Rejected"

    else:
        decision = "Tentative"

    return decision


print("Boruta Statistical Decision Rule")
print("================================")

print(f"Number of iterations: {n_boruta_iterations}")
print(f"Number of predictors: {n_boruta_features}")
print(f"Original alpha: {boruta_alpha}")
print(
    f"Bonferroni-corrected alpha: "
    f"{boruta_alpha_corrected:.6f}"
)

print("\nNull hit probability: 0.5")
print("Confirmation test: P(X >= observed hits)")
print("Rejection test:    P(X <= observed hits)")


# ------------------------------------------------------------
# Determine the hit-count boundaries implied by the test
# ------------------------------------------------------------

decision_table = []

for hits in range(n_boruta_iterations + 1):

    p_confirm, p_reject = boruta_hit_pvalues(hits)

    decision_table.append({
        "hits": hits,
        "p_confirm": p_confirm,
        "p_reject": p_reject,
        "decision": classify_boruta_hits(hits)
    })

boruta_decision_table = pd.DataFrame(decision_table)

confirmed_hits = boruta_decision_table.loc[
    boruta_decision_table["decision"] == "Confirmed",
    "hits"
]

rejected_hits = boruta_decision_table.loc[
    boruta_decision_table["decision"] == "Rejected",
    "hits"
]

print("\nDecision boundaries")
print("-------------------")

if len(confirmed_hits) > 0:
    print(
        "Minimum hits required for confirmation:",
        int(confirmed_hits.min())
    )

if len(rejected_hits) > 0:
    print(
        "Maximum hits resulting in rejection:",
        int(rejected_hits.max())
    )

tentative_hits = boruta_decision_table.loc[
    boruta_decision_table["decision"] == "Tentative",
    "hits"
]

if len(tentative_hits) > 0:
    print(
        "Tentative hit range:",
        f"{int(tentative_hits.min())}"
        f"–{int(tentative_hits.max())}"
    )

Boruta Statistical Decision Rule
Number of iterations: 50
Number of predictors: 20
Original alpha: 0.05
Bonferroni-corrected alpha: 0.002500

Null hit probability: 0.5
Confirmation test: P(X >= observed hits)
Rejection test:    P(X <= observed hits)

Decision boundaries
-------------------
Minimum hits required for confirmation: 36
Maximum hits resulting in rejection: 14
Tentative hit range: 15–35


In [ ]:
### E1.8.3 — Shadow Feature Generation

Boruta evaluates each real predictor relative to randomized versions of the
predictor information.

For every Boruta iteration, one shadow feature is generated for each real
predictor by independently permuting the observations of that predictor.

Permutation preserves the marginal distribution of the original variable but
breaks its correspondence with the target and with the original row structure.

The resulting shadow variables therefore represent randomized information
against which the importance of the real predictors can be compared.

All 20 real predictors receive their own independently permuted shadow feature
during every iteration.

Shadow variables are regenerated independently for each Boruta iteration and
are used only temporarily during model fitting.

In [93]:
# ============================================================
# E1.8.3 — Shadow Feature Generation
# ============================================================

def create_shadow_features(X, random_seed):
    """
    Create one independently permuted shadow feature for every
    real predictor.

    Parameters
    ----------
    X : pandas.DataFrame
        Real predictor matrix.

    random_seed : int
        Seed used to make the permutation reproducible.

    Returns
    -------
    pandas.DataFrame
        Shadow predictor matrix with columns prefixed by
        'shadow__'.
    """

    rng = np.random.default_rng(random_seed)

    shadow_data = {}

    for feature in X.columns:

        shadow_data[f"shadow__{feature}"] = rng.permutation(
            X[feature].to_numpy()
        )

    shadow_df = pd.DataFrame(
        shadow_data,
        index=X.index
    )

    return shadow_df


# ------------------------------------------------------------
# Test the shadow-generation function
# ------------------------------------------------------------

shadow_test_seed = boruta_settings["base_random_seed"]

X_shadow_test = create_shadow_features(
    X_config_train[boruta_candidate_predictors],
    random_seed=shadow_test_seed
)

print("Shadow Feature Generation Test")
print("==============================")

print(
    "Number of real features:",
    len(boruta_candidate_predictors)
)

print(
    "Number of shadow features:",
    X_shadow_test.shape[1]
)

print(
    "Number of observations:",
    X_shadow_test.shape[0]
)

print(
    "Index preserved:",
    X_shadow_test.index.equals(
        X_config_train.index
    )
)

print(
    "Missing shadow values:",
    int(X_shadow_test.isna().sum().sum())
)

print(
    "Duplicate shadow column names:",
    X_shadow_test.columns.duplicated().any()
)

print("\nExample shadow columns:")
for feature in X_shadow_test.columns[:5]:
    print(feature)

Shadow Feature Generation Test
Number of real features: 20
Number of shadow features: 20
Number of observations: 1255520
Index preserved: True
Missing shadow values: 0
Duplicate shadow column names: False

Example shadow columns:
shadow__current
shadow__pressure_anode_inlet
shadow__pressure_anode_outlet
shadow__pressure_cathode_inlet
shadow__pressure_cathode_outlet


In [94]:
# ------------------------------------------------------------
# Verify permutation behaviour
# ------------------------------------------------------------

verification_records = []

for feature in boruta_candidate_predictors:

    original_values = X_config_train[feature].to_numpy()

    shadow_values = X_shadow_test[
        f"shadow__{feature}"
    ].to_numpy()

    same_multiset = np.array_equal(
        np.sort(original_values),
        np.sort(shadow_values)
    )

    same_row_order = np.array_equal(
        original_values,
        shadow_values
    )

    verification_records.append({
        "feature": feature,
        "values_preserved": same_multiset,
        "row_order_unchanged": same_row_order
    })


shadow_verification = pd.DataFrame(
    verification_records
)

print("\nPermutation Verification")
print("========================")

print(
    "All feature values preserved:",
    shadow_verification[
        "values_preserved"
    ].all()
)

print(
    "Any feature retained identical row order:",
    shadow_verification[
        "row_order_unchanged"
    ].any()
)


Permutation Verification
All feature values preserved: True
Any feature retained identical row order: False


In [ ]:
### E1.8.4 — Single-Iteration XGBoost-Boruta Validation

Before executing the complete repeated Boruta procedure, one iteration is
performed to verify the XGBoost-Boruta implementation.

The 20 real predictors are combined with 20 independently permuted shadow
predictors and supplied simultaneously to the fixed XGBoost importance
estimator selected in E1.7.

Gain importance is extracted for every real and shadow predictor. Features
not used in any tree split are assigned a gain importance of zero.

The maximum gain obtained by any shadow predictor forms the random-information
benchmark for the iteration.

Each real predictor receives:

- Hit = 1 when its gain is greater than the maximum shadow gain.
- Hit = 0 otherwise.

This single iteration is used only to validate the implementation. No feature
is confirmed, rejected, or removed on the basis of this test iteration.

In [95]:
# ============================================================
# E1.8.4 — Single-Iteration XGBoost-Boruta Validation
# ============================================================

from xgboost import XGBRegressor

# ------------------------------------------------------------
# 1. Generate shadow predictors
# ------------------------------------------------------------

test_iteration_seed = boruta_settings["base_random_seed"]

X_shadow_iteration = create_shadow_features(
    X_config_train[boruta_candidate_predictors],
    random_seed=test_iteration_seed
)

# ------------------------------------------------------------
# 2. Combine real and shadow predictors
# ------------------------------------------------------------

X_boruta_iteration = pd.concat(
    [
        X_config_train[boruta_candidate_predictors],
        X_shadow_iteration
    ],
    axis=1
)

# ------------------------------------------------------------
# 3. Fit the frozen XGBoost importance estimator
# ------------------------------------------------------------

test_model_params = selected_xgb_boruta_config.copy()
test_model_params["random_state"] = test_iteration_seed

boruta_test_model = XGBRegressor(
    **test_model_params
)

boruta_test_model.fit(
    X_boruta_iteration,
    y_config_train
)

# ------------------------------------------------------------
# 4. Extract Gain importance
# ------------------------------------------------------------

gain_scores = boruta_test_model.get_booster().get_score(
    importance_type="gain"
)

all_gain = pd.Series(
    {
        feature: gain_scores.get(feature, 0.0)
        for feature in X_boruta_iteration.columns
    },
    dtype=float
)

real_gain = all_gain[
    boruta_candidate_predictors
]

shadow_gain = all_gain[
    X_shadow_iteration.columns
]

# ------------------------------------------------------------
# 5. Determine maximum shadow benchmark
# ------------------------------------------------------------

max_shadow_gain = shadow_gain.max()
max_shadow_feature = shadow_gain.idxmax()

# ------------------------------------------------------------
# 6. Assign hits to real predictors
# ------------------------------------------------------------

single_iteration_results = pd.DataFrame({
    "feature": boruta_candidate_predictors,
    "real_gain": [
        real_gain[feature]
        for feature in boruta_candidate_predictors
    ]
})

single_iteration_results["max_shadow_gain"] = (
    max_shadow_gain
)

single_iteration_results["hit"] = (
    single_iteration_results["real_gain"]
    > max_shadow_gain
).astype(int)

single_iteration_results = (
    single_iteration_results
    .sort_values(
        "real_gain",
        ascending=False
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 7. Report results
# ------------------------------------------------------------

print("Single XGBoost-Boruta Iteration")
print("================================")

print(
    "Real predictors:",
    len(boruta_candidate_predictors)
)

print(
    "Shadow predictors:",
    X_shadow_iteration.shape[1]
)

print(
    "Combined predictors:",
    X_boruta_iteration.shape[1]
)

print(
    f"\nMaximum shadow gain: "
    f"{max_shadow_gain:.6f}"
)

print(
    "Maximum shadow feature:",
    max_shadow_feature
)

print(
    "Number of real-feature hits:",
    int(single_iteration_results["hit"].sum())
)

print("\nReal Feature Results")
print("--------------------")

print(
    single_iteration_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

Single XGBoost-Boruta Iteration
Real predictors: 20
Shadow predictors: 20
Combined predictors: 40

Maximum shadow gain: 0.058739
Maximum shadow feature: shadow__temp_cathode_dewpoint_water
Number of real-feature hits: 19

Real Feature Results
--------------------
                    feature  real_gain  max_shadow_gain  hit
                    current 143.621994         0.058739    1
   total_cathode_stack_flow  63.525532         0.058739    1
     total_anode_stack_flow  14.521477         0.058739    1
      cathode_pressure_diff  11.559439         0.058739    1
     pressure_cathode_inlet   2.708781         0.058739    1
    pressure_cathode_outlet   2.266582         0.058739    1
        temp_anode_endplate   0.567776         0.058739    1
        anode_pressure_diff   0.414286         0.058739    1
      pressure_anode_outlet   0.331694         0.058739    1
         temp_cathode_inlet   0.204169         0.058739    1
            anode_temp_diff   0.190411         0.058739    1
    

In [ ]:
### E1.8.5 — Repeated XGBoost-Boruta Procedure

The validated XGBoost-Boruta procedure is now repeated for 50 independent
iterations using the training-only data.

During each iteration:

1. A new independently permuted shadow version of every real predictor is
   generated.
2. The 20 real and 20 shadow predictors are supplied simultaneously to the
   fixed XGBoost importance estimator.
3. Gain importance is extracted for all predictors.
4. The maximum shadow-feature gain is recorded as the iteration-specific
   random-information benchmark.
5. Each real predictor receives one hit when its gain exceeds this benchmark.

No predictor is removed during the 50 iterations. This ensures that every
candidate predictor is evaluated over the same number of repeated shadow
comparisons.

After all iterations, the accumulated hit counts are passed to the statistical
decision rule defined in E1.8.2 to classify predictors as Confirmed, Tentative,
or Rejected.

Only the training stages used for configuration development (50–350) are used
in this procedure. The internal configuration-validation stages, outer
chronological validation stages, and final test stages remain excluded.

In [96]:
# ============================================================
# E1.8.5 — Repeated XGBoost-Boruta Procedure
# ============================================================

import time

n_iterations = boruta_settings["n_iterations"]
base_seed = boruta_settings["base_random_seed"]

iteration_records = []
feature_iteration_records = []

boruta_start_time = time.perf_counter()


for iteration in range(1, n_iterations + 1):

    iteration_seed = base_seed + iteration

    # --------------------------------------------------------
    # 1. Generate new shadow predictors
    # --------------------------------------------------------

    X_shadow = create_shadow_features(
        X_config_train[boruta_candidate_predictors],
        random_seed=iteration_seed
    )

    # --------------------------------------------------------
    # 2. Combine real and shadow predictors
    # --------------------------------------------------------

    X_boruta = pd.concat(
        [
            X_config_train[boruta_candidate_predictors],
            X_shadow
        ],
        axis=1
    )

    # --------------------------------------------------------
    # 3. Fit fixed XGBoost importance estimator
    # --------------------------------------------------------

    model_params = selected_xgb_boruta_config.copy()
    model_params["random_state"] = iteration_seed

    model = XGBRegressor(
        **model_params
    )

    iteration_start = time.perf_counter()

    model.fit(
        X_boruta,
        y_config_train
    )

    fit_time = time.perf_counter() - iteration_start

    # --------------------------------------------------------
    # 4. Extract Gain importance
    # --------------------------------------------------------

    gain_scores = model.get_booster().get_score(
        importance_type="gain"
    )

    real_gains = {
        feature: gain_scores.get(feature, 0.0)
        for feature in boruta_candidate_predictors
    }

    shadow_gains = {
        feature: gain_scores.get(feature, 0.0)
        for feature in X_shadow.columns
    }

    # --------------------------------------------------------
    # 5. Maximum shadow benchmark
    # --------------------------------------------------------

    max_shadow_feature = max(
        shadow_gains,
        key=shadow_gains.get
    )

    max_shadow_gain = shadow_gains[
        max_shadow_feature
    ]

    # --------------------------------------------------------
    # 6. Record feature-level results
    # --------------------------------------------------------

    iteration_hit_count = 0

    for feature in boruta_candidate_predictors:

        feature_gain = real_gains[feature]

        hit = int(
            feature_gain > max_shadow_gain
        )

        iteration_hit_count += hit

        feature_iteration_records.append({
            "iteration": iteration,
            "seed": iteration_seed,
            "feature": feature,
            "real_gain": feature_gain,
            "max_shadow_gain": max_shadow_gain,
            "hit": hit
        })

    # --------------------------------------------------------
    # 7. Record iteration-level diagnostics
    # --------------------------------------------------------

    iteration_records.append({
        "iteration": iteration,
        "seed": iteration_seed,
        "max_shadow_feature": max_shadow_feature,
        "max_shadow_gain": max_shadow_gain,
        "real_feature_hits": iteration_hit_count,
        "fit_time_seconds": fit_time
    })

    print(
        f"Iteration {iteration:02d}/{n_iterations} | "
        f"Hits: {iteration_hit_count:02d}/{len(boruta_candidate_predictors)} | "
        f"Max shadow gain: {max_shadow_gain:.6f} | "
        f"Fit time: {fit_time:.2f} s"
    )


# ------------------------------------------------------------
# Convert records to DataFrames
# ------------------------------------------------------------

boruta_iteration_summary = pd.DataFrame(
    iteration_records
)

boruta_feature_iterations = pd.DataFrame(
    feature_iteration_records
)

boruta_total_time = (
    time.perf_counter() - boruta_start_time
)


print("\nRepeated XGBoost-Boruta Complete")
print("================================")

print(
    "Iterations completed:",
    len(boruta_iteration_summary)
)

print(
    "Feature-level records:",
    len(boruta_feature_iterations)
)

print(
    "Expected feature-level records:",
    n_iterations * len(boruta_candidate_predictors)
)

print(
    f"Total runtime: "
    f"{boruta_total_time:.2f} seconds"
)

print(
    f"Mean fit time per iteration: "
    f"{boruta_iteration_summary['fit_time_seconds'].mean():.2f} seconds"
)

Iteration 01/50 | Hits: 19/20 | Max shadow gain: 0.060525 | Fit time: 8.33 s
Iteration 02/50 | Hits: 19/20 | Max shadow gain: 0.044080 | Fit time: 8.09 s
Iteration 03/50 | Hits: 20/20 | Max shadow gain: 0.032621 | Fit time: 8.28 s
Iteration 04/50 | Hits: 19/20 | Max shadow gain: 0.054714 | Fit time: 7.90 s
Iteration 05/50 | Hits: 20/20 | Max shadow gain: 0.014468 | Fit time: 7.84 s
Iteration 06/50 | Hits: 20/20 | Max shadow gain: 0.021898 | Fit time: 8.00 s
Iteration 07/50 | Hits: 20/20 | Max shadow gain: 0.024252 | Fit time: 8.55 s
Iteration 08/50 | Hits: 20/20 | Max shadow gain: 0.021734 | Fit time: 8.52 s
Iteration 09/50 | Hits: 19/20 | Max shadow gain: 0.037270 | Fit time: 8.39 s
Iteration 10/50 | Hits: 20/20 | Max shadow gain: 0.035104 | Fit time: 8.24 s
Iteration 11/50 | Hits: 20/20 | Max shadow gain: 0.019919 | Fit time: 7.70 s
Iteration 12/50 | Hits: 17/20 | Max shadow gain: 0.064390 | Fit time: 8.30 s
Iteration 13/50 | Hits: 20/20 | Max shadow gain: 0.010895 | Fit time: 8.13 s

In [ ]:
### E1.8.6 — Aggregate Boruta Hits and Statistical Decisions

The 50 repeated XGBoost-Boruta iterations are aggregated at predictor level.

For each predictor, the total number and proportion of iterations in which its
gain exceeded the maximum shadow-feature gain are calculated.

The accumulated hit count is then evaluated using the statistical decision
rule defined before execution of the Boruta procedure.

With 50 iterations, 20 simultaneously evaluated predictors, α = 0.05 and
Bonferroni correction, the predefined decision boundaries are:

- 36–50 hits: Confirmed
- 15–35 hits: Tentative
- 0–14 hits: Rejected

The corresponding one-sided binomial p-values are also retained so that the
final classification is based on the formal statistical test rather than only
the displayed hit-count boundaries.

At this stage, the classifications describe feature relevance within the
initial training period only. Their stability across the outer chronological
training folds will be evaluated separately.

In [97]:
# ============================================================
# E1.8.6 — Aggregate Boruta Hits and Statistical Decisions
# ============================================================

boruta_feature_summary = (
    boruta_feature_iterations
    .groupby("feature", as_index=False)
    .agg(
        hits=("hit", "sum"),
        mean_real_gain=("real_gain", "mean"),
        sd_real_gain=("real_gain", "std"),
        mean_max_shadow_gain=("max_shadow_gain", "mean")
    )
)

# ------------------------------------------------------------
# Hit proportion
# ------------------------------------------------------------

boruta_feature_summary["hit_rate"] = (
    boruta_feature_summary["hits"]
    / n_boruta_iterations
)

# ------------------------------------------------------------
# Statistical p-values
# ------------------------------------------------------------

p_confirm_values = []
p_reject_values = []
decisions = []

for hits in boruta_feature_summary["hits"]:

    p_confirm, p_reject = boruta_hit_pvalues(
        int(hits)
    )

    p_confirm_values.append(p_confirm)
    p_reject_values.append(p_reject)

    decisions.append(
        classify_boruta_hits(int(hits))
    )

boruta_feature_summary["p_confirm"] = (
    p_confirm_values
)

boruta_feature_summary["p_reject"] = (
    p_reject_values
)

boruta_feature_summary["decision"] = (
    decisions
)

# ------------------------------------------------------------
# Decision ordering for presentation
# ------------------------------------------------------------

decision_order = {
    "Confirmed": 0,
    "Tentative": 1,
    "Rejected": 2
}

boruta_feature_summary["decision_order"] = (
    boruta_feature_summary["decision"]
    .map(decision_order)
)

boruta_feature_summary = (
    boruta_feature_summary
    .sort_values(
        ["decision_order", "hits", "mean_real_gain"],
        ascending=[True, False, False]
    )
    .drop(columns="decision_order")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Summary counts
# ------------------------------------------------------------

decision_counts = (
    boruta_feature_summary["decision"]
    .value_counts()
    .reindex(
        ["Confirmed", "Tentative", "Rejected"],
        fill_value=0
    )
)

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("XGBoost-Boruta Feature Decisions")
print("================================")

print(f"Iterations: {n_boruta_iterations}")
print(
    f"Bonferroni-corrected alpha: "
    f"{boruta_alpha_corrected:.6f}"
)

print("\nDecision counts")
print("---------------")

for decision, count in decision_counts.items():
    print(f"{decision}: {count}")

print("\nFeature-Level Results")
print("---------------------")

display_columns = [
    "feature",
    "hits",
    "hit_rate",
    "mean_real_gain",
    "mean_max_shadow_gain",
    "p_confirm",
    "p_reject",
    "decision"
]

print(
    boruta_feature_summary[
        display_columns
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

assert len(boruta_feature_summary) == len(
    boruta_candidate_predictors
)

assert boruta_feature_summary["hits"].between(
    0,
    n_boruta_iterations
).all()

assert set(
    boruta_feature_summary["decision"]
).issubset(
    {"Confirmed", "Tentative", "Rejected"}
)

print("\nIntegrity checks passed.")

XGBoost-Boruta Feature Decisions
Iterations: 50
Bonferroni-corrected alpha: 0.002500

Decision counts
---------------
Confirmed: 19
Tentative: 1
Rejected: 0

Feature-Level Results
---------------------
                    feature  hits  hit_rate  mean_real_gain  mean_max_shadow_gain  p_confirm  p_reject  decision
                    current    50  1.000000      141.589941              0.037385   0.000000  1.000000 Confirmed
   total_cathode_stack_flow    50  1.000000       50.408273              0.037385   0.000000  1.000000 Confirmed
     total_anode_stack_flow    50  1.000000       16.869577              0.037385   0.000000  1.000000 Confirmed
    pressure_cathode_outlet    50  1.000000        1.531927              0.037385   0.000000  1.000000 Confirmed
      cathode_pressure_diff    50  1.000000        1.287020              0.037385   0.000000  1.000000 Confirmed
        temp_anode_endplate    50  1.000000        0.535634              0.037385   0.000000  1.000000 Confirmed
     pr

In [ ]:
### E1.8.7 — Define Fold-Wise Boruta Training Sets

The initial XGBoost-Boruta analysis established feature relevance using the
early training period. Feature-selection stability is now assessed using the
four predefined expanding chronological training folds.

The training history expands progressively while the corresponding future
validation stages remain excluded from feature selection:

- Fold 1: train on stage labels 50–450; validation 500–550
- Fold 2: train on stage labels 50–550; validation 600–650
- Fold 3: train on stage labels 50–650; validation 700–750
- Fold 4: train on stage labels 50–750; validation 800–850

The XGBoost configuration, predictor set, number of Boruta iterations, shadow
benchmark, significance level and statistical decision rule remain unchanged.

Boruta is performed using only the training portion of each chronological
fold. The corresponding validation stages are not used during feature
selection.

This analysis evaluates whether predictor relevance remains stable as the
available PEMFC durability history expands.

In [98]:
# ============================================================
# E1.8.7 — Define Fold-Wise Boruta Training Sets
# ============================================================

boruta_outer_folds = {
    "Fold_1": {
        "train_labels": list(range(50, 451, 50)),
        "validation_labels": [500, 550]
    },
    "Fold_2": {
        "train_labels": list(range(50, 551, 50)),
        "validation_labels": [600, 650]
    },
    "Fold_3": {
        "train_labels": list(range(50, 651, 50)),
        "validation_labels": [700, 750]
    },
    "Fold_4": {
        "train_labels": list(range(50, 751, 50)),
        "validation_labels": [800, 850]
    }
}


fold_definition_records = []

print("Fold-Wise Boruta Training Framework")
print("====================================")

for fold_name, fold_info in boruta_outer_folds.items():

    train_labels = fold_info["train_labels"]
    validation_labels = fold_info["validation_labels"]

    fold_train_df = development_df[
        development_df["operating_hour"].isin(train_labels)
    ]

    fold_validation_df = development_df[
        development_df["operating_hour"].isin(validation_labels)
    ]

    overlap = set(train_labels).intersection(validation_labels)

    fold_definition_records.append({
        "fold": fold_name,
        "train_start_label": min(train_labels),
        "train_end_label": max(train_labels),
        "n_train_stages": len(train_labels),
        "n_train_observations": len(fold_train_df),
        "validation_labels": str(validation_labels),
        "n_validation_observations": len(fold_validation_df),
        "stage_overlap": len(overlap) > 0
    })

    print(f"\n{fold_name}")
    print("-" * len(fold_name))

    print("Training labels:", train_labels)
    print("Validation labels:", validation_labels)

    print(
        "Training observations:",
        f"{len(fold_train_df):,}"
    )

    print(
        "Validation observations:",
        f"{len(fold_validation_df):,}"
    )

    print(
        "Train/validation stage overlap:",
        len(overlap) > 0
    )

    print(
        "Chronological ordering preserved:",
        max(train_labels) < min(validation_labels)
    )


boruta_fold_definitions = pd.DataFrame(
    fold_definition_records
)


# ------------------------------------------------------------
# Global integrity checks
# ------------------------------------------------------------

all_validation_labels = []

for fold_info in boruta_outer_folds.values():
    all_validation_labels.extend(
        fold_info["validation_labels"]
    )

assert all(
    max(info["train_labels"])
    < min(info["validation_labels"])
    for info in boruta_outer_folds.values()
)

assert not set([900, 950, 1000]).intersection(
    set(
        label
        for info in boruta_outer_folds.values()
        for label in (
            info["train_labels"]
            + info["validation_labels"]
        )
    )
)

assert boruta_candidate_predictors == list(
    X_development.columns
)

print("\nGlobal Integrity Checks")
print("=======================")

print("Chronological ordering preserved: True")
print("Final test stages excluded: True")
print("Frozen 20-predictor Boruta input retained: True")
print("Frozen XGBoost configuration retained: Config_D")
print(
    "Boruta iterations per fold:",
    boruta_settings["n_iterations"]
)

Fold-Wise Boruta Training Framework

Fold_1
------
Training labels: [50, 100, 150, 200, 250, 300, 350, 400, 450]
Validation labels: [500, 550]
Training observations: 1,614,240
Validation observations: 358,720
Train/validation stage overlap: False
Chronological ordering preserved: True

Fold_2
------
Training labels: [50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550]
Validation labels: [600, 650]
Training observations: 1,972,960
Validation observations: 358,720
Train/validation stage overlap: False
Chronological ordering preserved: True

Fold_3
------
Training labels: [50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650]
Validation labels: [700, 750]
Training observations: 2,331,680
Validation observations: 358,720
Train/validation stage overlap: False
Chronological ordering preserved: True

Fold_4
------
Training labels: [50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700, 750]
Validation labels: [800, 850]
Training observations: 2,690,400
Validation obs

In [ ]:
### E1.8.8 — Reusable Fold-Wise XGBoost-Boruta Function

A reusable function is defined to apply the fixed XGBoost-Boruta procedure
consistently to each chronological training fold.

For every fold, the function:

1. extracts only the fold's training-stage observations;
2. retains the same frozen 20-predictor input set;
3. performs 50 independent shadow-feature iterations;
4. fits the same fixed Config D XGBoost importance estimator;
5. records whether each real feature exceeds the maximum shadow-feature gain;
6. aggregates hit counts across all iterations; and
7. applies the same predefined statistical decision rule.

No configuration or statistical setting is changed between folds.

This ensures that differences in Boruta decisions across folds reflect changes
in the available PEMFC durability history rather than changes in the
feature-selection procedure.

In [99]:
# ============================================================
# E1.8.8 — Reusable Fold-Wise XGBoost-Boruta Function
# ============================================================

def run_xgb_boruta_fold(
    fold_name,
    train_df,
    predictors,
    target_column="voltage",
    n_iterations=50,
    base_seed=42
):
    """
    Run the frozen XGBoost-Boruta procedure on one chronological
    training fold.

    Returns
    -------
    feature_summary : pandas.DataFrame
        Aggregated feature-level Boruta results.

    iteration_summary : pandas.DataFrame
        Iteration-level diagnostics.
    """

    X_fold = train_df[predictors]
    y_fold = train_df[target_column]

    feature_records = []
    iteration_records = []

    fold_start_time = time.perf_counter()

    for iteration in range(1, n_iterations + 1):

        iteration_seed = base_seed + iteration

        # ----------------------------------------------------
        # 1. Create shadow predictors
        # ----------------------------------------------------

        X_shadow = create_shadow_features(
            X_fold,
            random_seed=iteration_seed
        )

        # ----------------------------------------------------
        # 2. Combine real + shadow predictors
        # ----------------------------------------------------

        X_boruta = pd.concat(
            [X_fold, X_shadow],
            axis=1
        )

        # ----------------------------------------------------
        # 3. Fit frozen XGBoost estimator
        # ----------------------------------------------------

        model_params = selected_xgb_boruta_config.copy()
        model_params["random_state"] = iteration_seed

        model = XGBRegressor(
            **model_params
        )

        iteration_start = time.perf_counter()

        model.fit(
            X_boruta,
            y_fold
        )

        fit_time = (
            time.perf_counter()
            - iteration_start
        )

        # ----------------------------------------------------
        # 4. Extract gain importance
        # ----------------------------------------------------

        gain_scores = (
            model
            .get_booster()
            .get_score(
                importance_type="gain"
            )
        )

        real_gains = {
            feature: gain_scores.get(
                feature,
                0.0
            )
            for feature in predictors
        }

        shadow_gains = {
            feature: gain_scores.get(
                feature,
                0.0
            )
            for feature in X_shadow.columns
        }

        # ----------------------------------------------------
        # 5. Maximum shadow benchmark
        # ----------------------------------------------------

        max_shadow_feature = max(
            shadow_gains,
            key=shadow_gains.get
        )

        max_shadow_gain = (
            shadow_gains[
                max_shadow_feature
            ]
        )

        # ----------------------------------------------------
        # 6. Record hits
        # ----------------------------------------------------

        iteration_hits = 0

        for feature in predictors:

            feature_gain = real_gains[
                feature
            ]

            hit = int(
                feature_gain
                > max_shadow_gain
            )

            iteration_hits += hit

            feature_records.append({
                "fold": fold_name,
                "iteration": iteration,
                "seed": iteration_seed,
                "feature": feature,
                "real_gain": feature_gain,
                "max_shadow_gain": max_shadow_gain,
                "hit": hit
            })

        # ----------------------------------------------------
        # 7. Iteration diagnostics
        # ----------------------------------------------------

        iteration_records.append({
            "fold": fold_name,
            "iteration": iteration,
            "seed": iteration_seed,
            "max_shadow_feature": max_shadow_feature,
            "max_shadow_gain": max_shadow_gain,
            "real_feature_hits": iteration_hits,
            "fit_time_seconds": fit_time
        })

        print(
            f"{fold_name} | "
            f"Iteration {iteration:02d}/{n_iterations} | "
            f"Hits: {iteration_hits:02d}/{len(predictors)} | "
            f"Max shadow gain: {max_shadow_gain:.6f} | "
            f"Fit time: {fit_time:.2f} s"
        )

    # --------------------------------------------------------
    # Convert records to DataFrames
    # --------------------------------------------------------

    feature_iterations = pd.DataFrame(
        feature_records
    )

    iteration_summary = pd.DataFrame(
        iteration_records
    )

    # --------------------------------------------------------
    # Aggregate feature-level results
    # --------------------------------------------------------

    feature_summary = (
        feature_iterations
        .groupby(
            ["fold", "feature"],
            as_index=False
        )
        .agg(
            hits=("hit", "sum"),
            mean_real_gain=("real_gain", "mean"),
            sd_real_gain=("real_gain", "std"),
            mean_max_shadow_gain=(
                "max_shadow_gain",
                "mean"
            )
        )
    )

    feature_summary["hit_rate"] = (
        feature_summary["hits"]
        / n_iterations
    )

    # --------------------------------------------------------
    # Statistical decisions
    # --------------------------------------------------------

    feature_summary["p_confirm"] = (
        feature_summary["hits"]
        .apply(
            lambda h:
            boruta_hit_pvalues(
                int(h)
            )[0]
        )
    )

    feature_summary["p_reject"] = (
        feature_summary["hits"]
        .apply(
            lambda h:
            boruta_hit_pvalues(
                int(h)
            )[1]
        )
    )

    feature_summary["decision"] = (
        feature_summary["hits"]
        .apply(
            lambda h:
            classify_boruta_hits(
                int(h)
            )
        )
    )

    total_runtime = (
        time.perf_counter()
        - fold_start_time
    )

    print(
        f"\n{fold_name} complete"
    )

    print(
        f"Total runtime: "
        f"{total_runtime:.2f} seconds"
    )

    print(
        f"Confirmed: "
        f"{(feature_summary['decision'] == 'Confirmed').sum()}"
    )

    print(
        f"Tentative: "
        f"{(feature_summary['decision'] == 'Tentative').sum()}"
    )

    print(
        f"Rejected: "
        f"{(feature_summary['decision'] == 'Rejected').sum()}"
    )

    return (
        feature_summary,
        iteration_summary
    )

In [ ]:
### E1.8.9 — Fold 1 XGBoost-Boruta Analysis

The reusable Boruta procedure is first applied to Fold 1.

Fold 1 uses stage labels 50–450 for feature selection, while stage labels
500–550 remain completely excluded from the Boruta procedure.

This serves both as the first chronological feature-selection result and as
a final verification of the reusable fold-wise implementation before it is
applied to the remaining expanding folds.

In [100]:
# ============================================================
# E1.8.9 — Fold 1 XGBoost-Boruta Analysis
# ============================================================

fold_1_train_labels = (
    boruta_outer_folds[
        "Fold_1"
    ]["train_labels"]
)

fold_1_train_df = development_df[
    development_df[
        "operating_hour"
    ].isin(
        fold_1_train_labels
    )
].copy()

fold_1_boruta_summary, fold_1_iteration_summary = (
    run_xgb_boruta_fold(
        fold_name="Fold_1",
        train_df=fold_1_train_df,
        predictors=boruta_candidate_predictors,
        target_column="voltage",
        n_iterations=boruta_settings[
            "n_iterations"
        ],
        base_seed=boruta_settings[
            "base_random_seed"
        ]
    )
)


print(
    "\nFold 1 Feature Decisions"
)
print(
    "========================"
)

fold_1_display = (
    fold_1_boruta_summary
    .sort_values(
        ["decision", "hits"],
        ascending=[True, False]
    )
)

print(
    fold_1_display[
        [
            "feature",
            "hits",
            "hit_rate",
            "p_confirm",
            "p_reject",
            "decision"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

Fold_1 | Iteration 01/50 | Hits: 20/20 | Max shadow gain: 0.030425 | Fit time: 15.29 s
Fold_1 | Iteration 02/50 | Hits: 14/20 | Max shadow gain: 0.104781 | Fit time: 15.39 s
Fold_1 | Iteration 03/50 | Hits: 20/20 | Max shadow gain: 0.023857 | Fit time: 16.43 s
Fold_1 | Iteration 04/50 | Hits: 19/20 | Max shadow gain: 0.063686 | Fit time: 15.78 s
Fold_1 | Iteration 05/50 | Hits: 20/20 | Max shadow gain: 0.034747 | Fit time: 15.43 s
Fold_1 | Iteration 06/50 | Hits: 20/20 | Max shadow gain: 0.048408 | Fit time: 13.19 s
Fold_1 | Iteration 07/50 | Hits: 20/20 | Max shadow gain: 0.023641 | Fit time: 15.41 s
Fold_1 | Iteration 08/50 | Hits: 19/20 | Max shadow gain: 0.081804 | Fit time: 16.15 s
Fold_1 | Iteration 09/50 | Hits: 20/20 | Max shadow gain: 0.030477 | Fit time: 13.73 s
Fold_1 | Iteration 10/50 | Hits: 20/20 | Max shadow gain: 0.051036 | Fit time: 15.97 s
Fold_1 | Iteration 11/50 | Hits: 20/20 | Max shadow gain: 0.072576 | Fit time: 16.11 s
Fold_1 | Iteration 12/50 | Hits: 19/20 | Ma

In [ ]:
### E1.8.10 — Remaining Fold-Wise XGBoost-Boruta Analyses

Following successful validation of the fold-wise implementation on Fold 1,
the same frozen XGBoost-Boruta procedure is applied to Folds 2–4.

The available training history expands progressively:

- Fold 2: stage labels 50–550
- Fold 3: stage labels 50–650
- Fold 4: stage labels 50–750

For each fold, the corresponding future validation stages remain excluded
from feature selection.

No XGBoost configuration, Boruta parameter, statistical threshold, predictor
definition, or decision rule is modified between folds.

The resulting feature decisions will subsequently be compared across folds
to assess the temporal stability of predictor relevance as progressively
later PEMFC durability information becomes available.

In [101]:
# ============================================================
# E1.8.10 — Remaining Fold-Wise XGBoost-Boruta Analyses
# ============================================================

remaining_fold_summaries = {}
remaining_iteration_summaries = {}

for fold_name in ["Fold_2", "Fold_3", "Fold_4"]:

    train_labels = (
        boruta_outer_folds[
            fold_name
        ]["train_labels"]
    )

    fold_train_df = development_df[
        development_df[
            "operating_hour"
        ].isin(train_labels)
    ].copy()

    print("\n")
    print("=" * 70)
    print(
        f"Starting {fold_name} | "
        f"Training labels {min(train_labels)}–{max(train_labels)}"
    )
    print("=" * 70)

    feature_summary, iteration_summary = (
        run_xgb_boruta_fold(
            fold_name=fold_name,
            train_df=fold_train_df,
            predictors=boruta_candidate_predictors,
            target_column="voltage",
            n_iterations=boruta_settings[
                "n_iterations"
            ],
            base_seed=boruta_settings[
                "base_random_seed"
            ]
        )
    )

    remaining_fold_summaries[
        fold_name
    ] = feature_summary

    remaining_iteration_summaries[
        fold_name
    ] = iteration_summary


print("\n")
print("Remaining Fold-Wise Boruta Runs Complete")
print("=========================================")

for fold_name in ["Fold_2", "Fold_3", "Fold_4"]:

    summary = remaining_fold_summaries[
        fold_name
    ]

    print(f"\n{fold_name}")

    print(
        "Confirmed:",
        int(
            (
                summary["decision"]
                == "Confirmed"
            ).sum()
        )
    )

    print(
        "Tentative:",
        int(
            (
                summary["decision"]
                == "Tentative"
            ).sum()
        )
    )

    print(
        "Rejected:",
        int(
            (
                summary["decision"]
                == "Rejected"
            ).sum()
        )
    )



Starting Fold_2 | Training labels 50–550
Fold_2 | Iteration 01/50 | Hits: 20/20 | Max shadow gain: 0.071578 | Fit time: 18.43 s
Fold_2 | Iteration 02/50 | Hits: 15/20 | Max shadow gain: 0.107212 | Fit time: 18.64 s
Fold_2 | Iteration 03/50 | Hits: 20/20 | Max shadow gain: 0.042589 | Fit time: 19.23 s
Fold_2 | Iteration 04/50 | Hits: 20/20 | Max shadow gain: 0.037536 | Fit time: 18.07 s
Fold_2 | Iteration 05/50 | Hits: 20/20 | Max shadow gain: 0.046472 | Fit time: 19.25 s
Fold_2 | Iteration 06/50 | Hits: 20/20 | Max shadow gain: 0.028999 | Fit time: 19.51 s
Fold_2 | Iteration 07/50 | Hits: 20/20 | Max shadow gain: 0.027490 | Fit time: 19.39 s
Fold_2 | Iteration 08/50 | Hits: 20/20 | Max shadow gain: 0.081570 | Fit time: 19.27 s
Fold_2 | Iteration 09/50 | Hits: 19/20 | Max shadow gain: 0.091827 | Fit time: 18.66 s
Fold_2 | Iteration 10/50 | Hits: 20/20 | Max shadow gain: 0.045984 | Fit time: 19.40 s
Fold_2 | Iteration 11/50 | Hits: 20/20 | Max shadow gain: 0.036456 | Fit time: 18.10 s


In [107]:
# ============================================================
# E1.8.11 — Combined Cross-Fold XGBoost-Boruta Summary
# ============================================================

all_fold_summaries = {
    "Fold_1": fold_1_boruta_summary,
    "Fold_2": remaining_fold_summaries["Fold_2"],
    "Fold_3": remaining_fold_summaries["Fold_3"],
    "Fold_4": remaining_fold_summaries["Fold_4"]
}

combined_fold_results = []

for fold_name, summary in all_fold_summaries.items():

    temp = summary.copy()
    temp["fold"] = fold_name

    combined_fold_results.append(temp)

combined_fold_results = pd.concat(
    combined_fold_results,
    ignore_index=True
)

# ------------------------------------------------------------
# Hit counts by fold
# ------------------------------------------------------------

cross_fold_hits = (
    combined_fold_results
    .pivot(
        index="feature",
        columns="fold",
        values="hits"
    )
    .reset_index()
)

# Rename hit columns for clarity
cross_fold_hits = cross_fold_hits.rename(
    columns={
        "Fold_1": "Fold_1_hits",
        "Fold_2": "Fold_2_hits",
        "Fold_3": "Fold_3_hits",
        "Fold_4": "Fold_4_hits"
    }
)

# ------------------------------------------------------------
# Hit rates by fold
# ------------------------------------------------------------

cross_fold_rates = (
    combined_fold_results
    .pivot(
        index="feature",
        columns="fold",
        values="hit_rate"
    )
    .reset_index()
)

cross_fold_rates = cross_fold_rates.rename(
    columns={
        "Fold_1": "Fold_1_hit_rate",
        "Fold_2": "Fold_2_hit_rate",
        "Fold_3": "Fold_3_hit_rate",
        "Fold_4": "Fold_4_hit_rate"
    }
)

# ------------------------------------------------------------
# Merge hit counts and hit rates
# ------------------------------------------------------------

cross_fold_summary = cross_fold_hits.merge(
    cross_fold_rates,
    on="feature",
    how="left"
)

# ------------------------------------------------------------
# Total hits across all folds
# ------------------------------------------------------------

hit_columns = [
    "Fold_1_hits",
    "Fold_2_hits",
    "Fold_3_hits",
    "Fold_4_hits"
]

cross_fold_summary[
    "total_hits"
] = (
    cross_fold_summary[
        hit_columns
    ]
    .sum(axis=1)
)

# ------------------------------------------------------------
# Overall hit ratio
# ------------------------------------------------------------

n_folds = 4

n_iterations = boruta_settings[
    "n_iterations"
]

max_possible_hits = (
    n_folds
    * n_iterations
)

cross_fold_summary[
    "hit_ratio"
] = (
    cross_fold_summary[
        "total_hits"
    ]
    / max_possible_hits
)

# ------------------------------------------------------------
# Mean and minimum fold hit rate
# ------------------------------------------------------------

rate_columns = [
    "Fold_1_hit_rate",
    "Fold_2_hit_rate",
    "Fold_3_hit_rate",
    "Fold_4_hit_rate"
]

cross_fold_summary[
    "mean_hit_rate"
] = (
    cross_fold_summary[
        rate_columns
    ]
    .mean(axis=1)
)

cross_fold_summary[
    "min_hit_rate"
] = (
    cross_fold_summary[
        rate_columns
    ]
    .min(axis=1)
)

# ------------------------------------------------------------
# Count confirmed folds
# ------------------------------------------------------------

confirmed_counts = (
    combined_fold_results
    .assign(
        confirmed=(
            combined_fold_results[
                "decision"
            ]
            == "Confirmed"
        )
    )
    .groupby(
        "feature"
    )["confirmed"]
    .sum()
)

cross_fold_summary[
    "confirmed_folds"
] = (
    cross_fold_summary[
        "feature"
    ]
    .map(
        confirmed_counts
    )
)

# ------------------------------------------------------------
# Overall decision
# ------------------------------------------------------------

cross_fold_summary[
    "overall_decision"
] = (
    cross_fold_summary[
        "confirmed_folds"
    ]
    .apply(
        lambda x:
        "Stable Confirmed"
        if x == 4
        else "Inconsistent"
    )
)

# ------------------------------------------------------------
# Sort by strongest overall Boruta support
# ------------------------------------------------------------

cross_fold_summary = (
    cross_fold_summary
    .sort_values(
        [
            "hit_ratio",
            "min_hit_rate"
        ],
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

# ------------------------------------------------------------
# Display compact summary
# ------------------------------------------------------------

display_columns = [
    "feature",
    "Fold_1_hits",
    "Fold_2_hits",
    "Fold_3_hits",
    "Fold_4_hits",
    "total_hits",
    "hit_ratio",
    "mean_hit_rate",
    "min_hit_rate",
    "confirmed_folds",
    "overall_decision"
]

print(
    "\nCross-Fold XGBoost-Boruta Stability Summary"
)

print(
    "=" * 140
)

print(
    cross_fold_summary[
        display_columns
    ]
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


Cross-Fold XGBoost-Boruta Stability Summary
                    feature  Fold_1_hits  Fold_2_hits  Fold_3_hits  Fold_4_hits  total_hits  hit_ratio  mean_hit_rate  min_hit_rate  confirmed_folds overall_decision
        anode_pressure_diff           50           50           50           50         200     1.0000         1.0000        1.0000                4 Stable Confirmed
            anode_temp_diff           50           50           50           50         200     1.0000         1.0000        1.0000                4 Stable Confirmed
      cathode_pressure_diff           50           50           50           50         200     1.0000         1.0000        1.0000                4 Stable Confirmed
                    current           50           50           50           50         200     1.0000         1.0000        1.0000                4 Stable Confirmed
      pressure_anode_outlet           50           50           50           50         200     1.0000         1.0000        

In [108]:
# ============================================================
# E1.8.12 — Final Cross-Fold Boruta Consensus
# ============================================================

# ------------------------------------------------------------
# Features confirmed in every outer fold
# ------------------------------------------------------------

boruta_stable_features = (
    cross_fold_summary.loc[
        cross_fold_summary[
            "confirmed_folds"
        ] == 4,
        "feature"
    ]
    .tolist()
)

# ------------------------------------------------------------
# Features not consistently confirmed
# ------------------------------------------------------------

boruta_inconsistent_features = (
    cross_fold_summary.loc[
        cross_fold_summary[
            "confirmed_folds"
        ] < 4,
        "feature"
    ]
    .tolist()
)

# ------------------------------------------------------------
# Final Boruta consensus table
# ------------------------------------------------------------

boruta_consensus_summary = (
    cross_fold_summary[
        [
            "feature",
            "total_hits",
            "hit_ratio",
            "mean_hit_rate",
            "min_hit_rate",
            "confirmed_folds",
            "overall_decision"
        ]
    ]
    .copy()
)

boruta_consensus_summary[
    "boruta_selection"
] = (
    boruta_consensus_summary[
        "confirmed_folds"
    ]
    .apply(
        lambda x:
        "Retain"
        if x == 4
        else "Review"
    )
)

boruta_consensus_summary = (
    boruta_consensus_summary
    .sort_values(
        [
            "confirmed_folds",
            "hit_ratio",
            "min_hit_rate"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print(
    "Final Cross-Fold Boruta Consensus"
)
print(
    "=" * 95
)

print(
    boruta_consensus_summary
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\n")
print(
    f"Stable Boruta features retained: "
    f"{len(boruta_stable_features)}"
)

print(
    f"Inconsistent features requiring review: "
    f"{len(boruta_inconsistent_features)}"
)

print("\nRetained Features")
print("=================")

for feature in boruta_stable_features:
    print(feature)

Final Cross-Fold Boruta Consensus
                    feature  total_hits  hit_ratio  mean_hit_rate  min_hit_rate  confirmed_folds overall_decision boruta_selection
        anode_pressure_diff         200     1.0000         1.0000        1.0000                4 Stable Confirmed           Retain
            anode_temp_diff         200     1.0000         1.0000        1.0000                4 Stable Confirmed           Retain
      cathode_pressure_diff         200     1.0000         1.0000        1.0000                4 Stable Confirmed           Retain
                    current         200     1.0000         1.0000        1.0000                4 Stable Confirmed           Retain
      pressure_anode_outlet         200     1.0000         1.0000        1.0000                4 Stable Confirmed           Retain
    pressure_cathode_outlet         200     1.0000         1.0000        1.0000                4 Stable Confirmed           Retain
  temp_anode_dewpoint_water         200     1.000

In [115]:
# ============================================================
# E1.8 — Integrated Feature Evidence
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Pearson target evidence
# ------------------------------------------------------------

pearson_evidence = (
    pearson_target_table[
        [
            "Predictor",
            "Pearson_r",
            "Absolute_r",
            "p_value",
            "Association_Strength"
        ]
    ]
    .rename(
        columns={
            "Predictor": "feature",
            "Absolute_r": "Absolute_Pearson_r",
            "p_value": "Pearson_p_value",
            "Association_Strength":
                "Pearson_Strength"
        }
    )
    .copy()
)


# ------------------------------------------------------------
# 2. Spearman target evidence
# ------------------------------------------------------------

spearman_evidence = (
    spearman_target_table[
        [
            "Predictor",
            "Spearman_rho",
            "Absolute_rho",
            "p_value",
            "Association_Strength"
        ]
    ]
    .rename(
        columns={
            "Predictor": "feature",
            "Absolute_rho":
                "Absolute_Spearman_rho",
            "p_value":
                "Spearman_p_value",
            "Association_Strength":
                "Spearman_Strength"
        }
    )
    .copy()
)


# ------------------------------------------------------------
# 3. Mutual Information evidence
# ------------------------------------------------------------

mi_evidence = (
    mi_target_table[
        [
            "Predictor",
            "Mutual_Information",
            "MI_Rank",
            "Relative_MI"
        ]
    ]
    .rename(
        columns={
            "Predictor": "feature"
        }
    )
    .copy()
)


# ------------------------------------------------------------
# 4. VIF evidence
# ------------------------------------------------------------

vif_evidence = (
    vif_results_df[
        [
            "Predictor",
            "Auxiliary_R2",
            "Tolerance",
            "VIF",
            "Multicollinearity_Level"
        ]
    ]
    .rename(
        columns={
            "Predictor": "feature"
        }
    )
    .copy()
)


# ------------------------------------------------------------
# 5. Exact linear dependency flag
#
# Infinite VIF indicates that a predictor participates in an
# exact or near-exact linear dependency within the full
# predictor set.
#
# IMPORTANT:
# This does NOT mean that every flagged variable is itself an
# engineered variable.
#
# Example:
# pressure_anode_inlet,
# pressure_anode_outlet,
# anode_pressure_diff
#
# can form an exact dependency because:
#
# anode_pressure_diff
# = pressure_anode_inlet
# - pressure_anode_outlet
# ------------------------------------------------------------

vif_evidence[
    "Exact_Linear_Dependency"
] = (
    np.isinf(
        vif_evidence[
            "VIF"
        ]
    )
)


# ------------------------------------------------------------
# 6. Pearson redundancy flag
#
# Uses the predictor-predictor redundancy results already
# calculated earlier.
# ------------------------------------------------------------

pearson_redundant_features = set(
    high_pearson_redundancy[
        "Predictor_1"
    ]
).union(
    set(
        high_pearson_redundancy[
            "Predictor_2"
        ]
    )
)


# ------------------------------------------------------------
# 7. Spearman redundancy flag
# ------------------------------------------------------------

spearman_redundant_features = set(
    high_spearman_redundancy[
        "Predictor_1"
    ]
).union(
    set(
        high_spearman_redundancy[
            "Predictor_2"
        ]
    )
)


# ------------------------------------------------------------
# 8. Feature-level redundancy evidence
# ------------------------------------------------------------

redundancy_evidence = pd.DataFrame(
    {
        "feature":
            boruta_candidate_predictors
    }
)


redundancy_evidence[
    "Pearson_Redundancy_Flag"
] = (
    redundancy_evidence[
        "feature"
    ]
    .isin(
        pearson_redundant_features
    )
)


redundancy_evidence[
    "Spearman_Redundancy_Flag"
] = (
    redundancy_evidence[
        "feature"
    ]
    .isin(
        spearman_redundant_features
    )
)


# ------------------------------------------------------------
# Combined redundancy flag
#
# True when a feature was flagged by either Pearson or
# Spearman predictor-predictor redundancy analysis.
#
# This is diagnostic evidence only.
# It is NOT an automatic deletion decision.
# ------------------------------------------------------------

redundancy_evidence[
    "Redundancy_Flag"
] = (
    redundancy_evidence[
        "Pearson_Redundancy_Flag"
    ]
    |
    redundancy_evidence[
        "Spearman_Redundancy_Flag"
    ]
)


# ------------------------------------------------------------
# 9. Boruta cross-fold evidence
#
# Uses the Final Cross-Fold Boruta Consensus already
# calculated.
# ------------------------------------------------------------

boruta_evidence = (
    cross_fold_summary[
        [
            "feature",
            "total_hits",
            "hit_ratio",
            "mean_hit_rate",
            "min_hit_rate",
            "confirmed_folds",
            "overall_decision"
        ]
    ]
    .copy()
)


boruta_evidence = (
    boruta_evidence
    .rename(
        columns={
            "hit_ratio":
                "Boruta_Hit_Rate",

            "mean_hit_rate":
                "Boruta_Mean_Hit_Rate",

            "min_hit_rate":
                "Boruta_Min_Hit_Rate"
        }
    )
)


# ------------------------------------------------------------
# 10. Fold-selection frequency
#
# Four chronological Boruta folds were used.
# ------------------------------------------------------------

n_boruta_folds = 4

boruta_evidence[
    "Fold_Selection_Frequency"
] = (
    boruta_evidence[
        "confirmed_folds"
    ]
    / n_boruta_folds
)


# ------------------------------------------------------------
# 11. Final Boruta status
# ------------------------------------------------------------

def extract_boruta_status(value):

    value = str(value)

    if "Confirmed" in value:
        return "Confirmed"

    elif "Tentative" in value:
        return "Tentative"

    elif "Rejected" in value:
        return "Rejected"

    else:
        return value


boruta_evidence[
    "Boruta_Status"
] = (
    boruta_evidence[
        "overall_decision"
    ]
    .apply(
        extract_boruta_status
    )
)


# ------------------------------------------------------------
# 12. Merge all existing evidence
# ------------------------------------------------------------

integrated_feature_evidence = (
    pearson_evidence

    .merge(
        spearman_evidence,
        on="feature",
        how="outer"
    )

    .merge(
        mi_evidence,
        on="feature",
        how="outer"
    )

    .merge(
        redundancy_evidence,
        on="feature",
        how="outer"
    )

    .merge(
        vif_evidence,
        on="feature",
        how="outer"
    )

    .merge(
        boruta_evidence,
        on="feature",
        how="outer"
    )
)


# ------------------------------------------------------------
# 13. Final concise E1.8 evidence table
#
# Boruta evidence is placed at the end because it carries the
# highest methodological weight in the feature-selection
# decision.
# ------------------------------------------------------------

final_integrated_feature_evidence = (
    integrated_feature_evidence[
        [
            "feature",

            # Association evidence
            "Pearson_r",
            "Spearman_rho",
            "Mutual_Information",

            # Redundancy evidence
            "Pearson_Redundancy_Flag",
            "Spearman_Redundancy_Flag",
            "Redundancy_Flag",

            # Multicollinearity / dependency evidence
            "VIF",
            "Multicollinearity_Level",
            "Exact_Linear_Dependency",

            # Boruta evidence
            "Boruta_Hit_Rate",
            "Boruta_Status",
            "Fold_Selection_Frequency"
        ]
    ]
    .sort_values(
        [
            "Fold_Selection_Frequency",
            "Boruta_Hit_Rate"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 14. Display
# ------------------------------------------------------------

print(
    "E1.8 — Integrated Feature Evidence"
)

print(
    "=" * 175
)

print(
    final_integrated_feature_evidence
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

E1.8 — Integrated Feature Evidence
                    feature  Pearson_r  Spearman_rho  Mutual_Information  Pearson_Redundancy_Flag  Spearman_Redundancy_Flag  Redundancy_Flag        VIF              Multicollinearity_Level  Exact_Linear_Dependency  Boruta_Hit_Rate Boruta_Status  Fold_Selection_Frequency
        anode_pressure_diff    -0.6263       -0.5539              0.5952                    False                     False            False        inf Exact / near-exact multicollinearity                     True           1.0000     Confirmed                    1.0000
            anode_temp_diff    -0.0170       -0.0458              0.2695                     True                      True             True        inf Exact / near-exact multicollinearity                     True           1.0000     Confirmed                    1.0000
      cathode_pressure_diff    -0.8539       -0.8190              1.4327                     True                      True             True        inf 

In [116]:
# ============================================================
# E1.9.1 — Planned Candidate Feature Sets
# ============================================================

# ------------------------------------------------------------
# A — Full valid feature set
# ------------------------------------------------------------

candidate_set_A = (
    boruta_candidate_predictors.copy()
)


# ------------------------------------------------------------
# B — Boruta Confirmed
# ------------------------------------------------------------

candidate_set_B = (
    final_integrated_feature_evidence.loc[
        final_integrated_feature_evidence[
            "Boruta_Status"
        ] == "Confirmed",
        "feature"
    ]
    .tolist()
)


# ------------------------------------------------------------
# C — Boruta Confirmed + Tentative
# ------------------------------------------------------------

candidate_set_C = (
    final_integrated_feature_evidence.loc[
        final_integrated_feature_evidence[
            "Boruta_Status"
        ]
        .isin(
            [
                "Confirmed",
                "Tentative"
            ]
        ),
        "feature"
    ]
    .tolist()
)


candidate_feature_sets = {
    "A_Full_Valid":
        candidate_set_A,

    "B_Boruta_Confirmed":
        candidate_set_B,

    "C_Boruta_Confirmed_Plus_Tentative":
        candidate_set_C
}


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print(
    "E1.9 — Planned Candidate Feature Sets"
)

print("=" * 80)

for (
    set_name,
    features
) in candidate_feature_sets.items():

    print(
        f"\n{set_name}"
    )

    print(
        f"Number of features: "
        f"{len(features)}"
    )

    print(
        ", ".join(
            features
        )
    )

E1.9 — Planned Candidate Feature Sets

A_Full_Valid
Number of features: 20
current, pressure_anode_inlet, pressure_anode_outlet, pressure_cathode_inlet, pressure_cathode_outlet, temp_anode_endplate, temp_anode_dewpoint_water, temp_anode_inlet, temp_anode_outlet, temp_cathode_dewpoint_water, temp_cathode_inlet, temp_cathode_outlet, total_anode_stack_flow, total_cathode_stack_flow, anode_pressure_diff, cathode_pressure_diff, anode_temp_diff, cathode_temp_diff, anode_dewpoint_offset, cathode_dewpoint_offset

B_Boruta_Confirmed
Number of features: 20
anode_pressure_diff, anode_temp_diff, cathode_pressure_diff, current, pressure_anode_outlet, pressure_cathode_outlet, temp_anode_dewpoint_water, temp_anode_endplate, temp_cathode_inlet, total_anode_stack_flow, total_cathode_stack_flow, cathode_dewpoint_offset, temp_cathode_dewpoint_water, temp_anode_outlet, pressure_cathode_inlet, temp_cathode_outlet, pressure_anode_inlet, temp_anode_inlet, cathode_temp_diff, anode_dewpoint_offset

C_Boruta_Co

In [117]:
# ============================================================
# E1.9.2 — Candidate-Set Equivalence Check
# ============================================================

candidate_set_comparison = []

set_names = list(
    candidate_feature_sets.keys()
)

for i in range(
    len(set_names)
):

    for j in range(
        i + 1,
        len(set_names)
    ):

        set_1_name = (
            set_names[i]
        )

        set_2_name = (
            set_names[j]
        )

        set_1 = set(
            candidate_feature_sets[
                set_1_name
            ]
        )

        set_2 = set(
            candidate_feature_sets[
                set_2_name
            ]
        )

        candidate_set_comparison.append(
            {
                "Set_1":
                    set_1_name,

                "Set_2":
                    set_2_name,

                "Identical":
                    set_1 == set_2,

                "Set_1_Features":
                    len(set_1),

                "Set_2_Features":
                    len(set_2),

                "Only_in_Set_1":
                    ", ".join(
                        sorted(
                            set_1
                            - set_2
                        )
                    )
                    or "None",

                "Only_in_Set_2":
                    ", ".join(
                        sorted(
                            set_2
                            - set_1
                        )
                    )
                    or "None"
            }
        )


candidate_set_equivalence = pd.DataFrame(
    candidate_set_comparison
)


print(
    "Candidate-Set Equivalence"
)

print("=" * 100)

print(
    candidate_set_equivalence
    .to_string(
        index=False
    )
)

Candidate-Set Equivalence
             Set_1                             Set_2  Identical  Set_1_Features  Set_2_Features Only_in_Set_1 Only_in_Set_2
      A_Full_Valid                B_Boruta_Confirmed       True              20              20          None          None
      A_Full_Valid C_Boruta_Confirmed_Plus_Tentative       True              20              20          None          None
B_Boruta_Confirmed C_Boruta_Confirmed_Plus_Tentative       True              20              20          None          None


In [118]:
# ============================================================
# E1.9.3 — Define Final Boruta-Selected Feature Set
# ============================================================

all_candidate_sets_identical = (
    set(candidate_set_A)
    == set(candidate_set_B)
    == set(candidate_set_C)
)


if all_candidate_sets_identical:

    selected_feature_set = (
        candidate_set_B.copy()
    )

    selected_feature_set_name = (
        "Boruta_Confirmed"
    )

    print(
        "All planned candidate feature sets are identical."
    )

    print(
        "No separate A/B/C predictive comparison is required."
    )

else:

    selected_feature_set = None

    selected_feature_set_name = None

    print(
        "Candidate sets differ."
    )

    print(
        "Separate candidate-set validation is required."
    )


print("\nFinal Selected Feature Set")
print("=" * 65)

print(
    f"Selection basis: "
    f"{selected_feature_set_name}"
)

print(
    f"Number of predictors: "
    f"{len(selected_feature_set)}"
)

print("\nPredictors:")

for i, feature in enumerate(
    selected_feature_set,
    start=1
):
    print(
        f"{i:>2}. {feature}"
    )

All planned candidate feature sets are identical.
No separate A/B/C predictive comparison is required.

Final Selected Feature Set
Selection basis: Boruta_Confirmed
Number of predictors: 20

Predictors:
 1. anode_pressure_diff
 2. anode_temp_diff
 3. cathode_pressure_diff
 4. current
 5. pressure_anode_outlet
 6. pressure_cathode_outlet
 7. temp_anode_dewpoint_water
 8. temp_anode_endplate
 9. temp_cathode_inlet
10. total_anode_stack_flow
11. total_cathode_stack_flow
12. cathode_dewpoint_offset
13. temp_cathode_dewpoint_water
14. temp_anode_outlet
15. pressure_cathode_inlet
16. temp_cathode_outlet
17. pressure_anode_inlet
18. temp_anode_inlet
19. cathode_temp_diff
20. anode_dewpoint_offset


In [119]:
# ============================================================
# E1.10.1 — Inspect Chronological Fold Structure
# ============================================================

print(
    "E1.10 — Existing Chronological Fold Structure"
)

print("=" * 75)


for fold_name, fold_info in boruta_outer_folds.items():

    print(f"\n{fold_name}")
    print("-" * 40)

    print(
        "Available keys:",
        list(
            fold_info.keys()
        )
    )

    for key, value in fold_info.items():

        if isinstance(
            value,
            (list, tuple, np.ndarray, pd.Series)
        ):

            print(
                f"{key}: "
                f"{len(value)} labels"
            )

            if len(value) > 0:

                print(
                    f"    Range: "
                    f"{min(value)}–{max(value)}"
                )

                print(
                    f"    Values: "
                    f"{list(value)}"
                )

        else:

            print(
                f"{key}: {value}"
            )

E1.10 — Existing Chronological Fold Structure

Fold_1
----------------------------------------
Available keys: ['train_labels', 'validation_labels']
train_labels: 9 labels
    Range: 50–450
    Values: [50, 100, 150, 200, 250, 300, 350, 400, 450]
validation_labels: 2 labels
    Range: 500–550
    Values: [500, 550]

Fold_2
----------------------------------------
Available keys: ['train_labels', 'validation_labels']
train_labels: 11 labels
    Range: 50–550
    Values: [50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550]
validation_labels: 2 labels
    Range: 600–650
    Values: [600, 650]

Fold_3
----------------------------------------
Available keys: ['train_labels', 'validation_labels']
train_labels: 13 labels
    Range: 50–650
    Values: [50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650]
validation_labels: 2 labels
    Range: 700–750
    Values: [700, 750]

Fold_4
----------------------------------------
Available keys: ['train_labels', 'validation_labels']
train_

In [120]:
# ============================================================
# E1.10.2 — Chronological Validation of Selected Feature Set
# Ridge + XGBoost
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from xgboost import XGBRegressor


# ------------------------------------------------------------
# 1. Validation settings
# ------------------------------------------------------------

target_column = "voltage"

validation_features = (
    selected_feature_set.copy()
)

random_seed = (
    boruta_settings[
        "base_random_seed"
    ]
)


print(
    "E1.10 — Chronological Validation"
)

print("=" * 75)

print(
    f"Selected predictors: "
    f"{len(validation_features)}"
)

print(
    f"Target: {target_column}"
)

print(
    f"Chronological folds: "
    f"{len(boruta_outer_folds)}"
)


# ------------------------------------------------------------
# 2. Storage for fold-wise results
# ------------------------------------------------------------

chronological_validation_results = []


# ------------------------------------------------------------
# 3. Loop through expanding chronological folds
# ------------------------------------------------------------

for fold_name, fold_info in (
    boruta_outer_folds.items()
):

    train_labels = (
        fold_info[
            "train_labels"
        ]
    )

    validation_labels = (
        fold_info[
            "validation_labels"
        ]
    )


    print("\n")
    print("=" * 75)

    print(
        f"{fold_name}"
    )

    print("-" * 75)

    print(
        f"Training stages: "
        f"{min(train_labels)}–"
        f"{max(train_labels)} h"
    )

    print(
        f"Validation stages: "
        f"{min(validation_labels)}–"
        f"{max(validation_labels)} h"
    )


    # --------------------------------------------------------
    # 4. Construct chronological training data
    # --------------------------------------------------------

    train_df = (
        development_df[
            development_df[
                "operating_hour"
            ].isin(
                train_labels
            )
        ]
        .copy()
    )


    # --------------------------------------------------------
    # 5. Construct unseen later-stage validation data
    # --------------------------------------------------------

    validation_df = (
        development_df[
            development_df[
                "operating_hour"
            ].isin(
                validation_labels
            )
        ]
        .copy()
    )


    X_train = (
        train_df[
            validation_features
        ]
    )

    y_train = (
        train_df[
            target_column
        ]
    )


    X_validation = (
        validation_df[
            validation_features
        ]
    )

    y_validation = (
        validation_df[
            target_column
        ]
    )


    print(
        f"Training rows: "
        f"{len(X_train):,}"
    )

    print(
        f"Validation rows: "
        f"{len(X_validation):,}"
    )


    # ========================================================
    # 6. Ridge
    #
    # Standardisation is fitted ONLY on training data through
    # the pipeline, preventing leakage from validation stages.
    # ========================================================

    ridge_model = Pipeline(
        steps=[
            (
                "scaler",
                StandardScaler()
            ),
            (
                "ridge",
                Ridge(
                    alpha=1.0
                )
            )
        ]
    )


    ridge_start = time.time()

    ridge_model.fit(
        X_train,
        y_train
    )

    ridge_predictions = (
        ridge_model.predict(
            X_validation
        )
    )

    ridge_runtime = (
        time.time()
        - ridge_start
    )


    ridge_rmse = np.sqrt(
        mean_squared_error(
            y_validation,
            ridge_predictions
        )
    )

    ridge_mae = (
        mean_absolute_error(
            y_validation,
            ridge_predictions
        )
    )

    ridge_r2 = (
        r2_score(
            y_validation,
            ridge_predictions
        )
    )


    chronological_validation_results.append(
        {
            "Fold":
                fold_name,

            "Model":
                "Ridge",

            "Train_Start_h":
                min(train_labels),

            "Train_End_h":
                max(train_labels),

            "Validation_Start_h":
                min(validation_labels),

            "Validation_End_h":
                max(validation_labels),

            "Training_Rows":
                len(X_train),

            "Validation_Rows":
                len(X_validation),

            "Number_of_Features":
                len(validation_features),

            "RMSE":
                ridge_rmse,

            "MAE":
                ridge_mae,

            "R2":
                ridge_r2,

            "Runtime_Seconds":
                ridge_runtime
        }
    )


    print("\nRidge")

    print(
        f"RMSE: {ridge_rmse:.6f}"
    )

    print(
        f"MAE : {ridge_mae:.6f}"
    )

    print(
        f"R²  : {ridge_r2:.6f}"
    )


    # ========================================================
    # 7. XGBoost
    #
    # Fixed configuration is used across all folds so that
    # changes in performance reflect chronological
    # generalisation rather than fold-specific tuning.
    # ========================================================

    xgb_model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        subsample=1.0,
        colsample_bytree=1.0,
        random_state=random_seed,
        n_jobs=-1,
        tree_method="hist"
    )


    xgb_start = time.time()

    xgb_model.fit(
        X_train,
        y_train
    )

    xgb_predictions = (
        xgb_model.predict(
            X_validation
        )
    )

    xgb_runtime = (
        time.time()
        - xgb_start
    )


    xgb_rmse = np.sqrt(
        mean_squared_error(
            y_validation,
            xgb_predictions
        )
    )

    xgb_mae = (
        mean_absolute_error(
            y_validation,
            xgb_predictions
        )
    )

    xgb_r2 = (
        r2_score(
            y_validation,
            xgb_predictions
        )
    )


    chronological_validation_results.append(
        {
            "Fold":
                fold_name,

            "Model":
                "XGBoost",

            "Train_Start_h":
                min(train_labels),

            "Train_End_h":
                max(train_labels),

            "Validation_Start_h":
                min(validation_labels),

            "Validation_End_h":
                max(validation_labels),

            "Training_Rows":
                len(X_train),

            "Validation_Rows":
                len(X_validation),

            "Number_of_Features":
                len(validation_features),

            "RMSE":
                xgb_rmse,

            "MAE":
                xgb_mae,

            "R2":
                xgb_r2,

            "Runtime_Seconds":
                xgb_runtime
        }
    )


    print("\nXGBoost")

    print(
        f"RMSE: {xgb_rmse:.6f}"
    )

    print(
        f"MAE : {xgb_mae:.6f}"
    )

    print(
        f"R²  : {xgb_r2:.6f}"
    )


# ------------------------------------------------------------
# 8. Final results table
# ------------------------------------------------------------

chronological_validation_df = (
    pd.DataFrame(
        chronological_validation_results
    )
)


print("\n")
print(
    "E1.10 — Fold-Wise Chronological Validation Results"
)

print("=" * 120)

print(
    chronological_validation_df[
        [
            "Fold",
            "Model",
            "Train_End_h",
            "Validation_Start_h",
            "Validation_End_h",
            "Number_of_Features",
            "RMSE",
            "MAE",
            "R2",
            "Runtime_Seconds"
        ]
    ]
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

E1.10 — Chronological Validation
Selected predictors: 20
Target: voltage
Chronological folds: 4


Fold_1
---------------------------------------------------------------------------
Training stages: 50–450 h
Validation stages: 500–550 h
Training rows: 1,614,240
Validation rows: 358,720

Ridge
RMSE: 0.018224
MAE : 0.014882
R²  : 0.965270

XGBoost
RMSE: 0.007352
MAE : 0.005210
R²  : 0.994348


Fold_2
---------------------------------------------------------------------------
Training stages: 50–550 h
Validation stages: 600–650 h
Training rows: 1,972,960
Validation rows: 358,720

Ridge
RMSE: 0.016332
MAE : 0.012935
R²  : 0.973288

XGBoost
RMSE: 0.010015
MAE : 0.006738
R²  : 0.989955


Fold_3
---------------------------------------------------------------------------
Training stages: 50–650 h
Validation stages: 700–750 h
Training rows: 2,331,680
Validation rows: 358,720

Ridge
RMSE: 0.019800
MAE : 0.016798
R²  : 0.962488

XGBoost
RMSE: 0.011608
MAE : 0.009366
R²  : 0.987106


Fold_4
-------

In [121]:
# ============================================================
# E1.11.1 — Feature-Selection Temporal Stability
# ============================================================

# ------------------------------------------------------------
# Combine the four existing fold-wise Boruta summaries
# ------------------------------------------------------------

all_fold_boruta_summaries = {
    "Fold_1":
        fold_1_boruta_summary,

    "Fold_2":
        remaining_fold_summaries[
            "Fold_2"
        ],

    "Fold_3":
        remaining_fold_summaries[
            "Fold_3"
        ],

    "Fold_4":
        remaining_fold_summaries[
            "Fold_4"
        ]
}


# ------------------------------------------------------------
# Extract fold-wise hit rates
# ------------------------------------------------------------

fold_hit_rate_tables = []

for fold_name, summary in (
    all_fold_boruta_summaries.items()
):

    fold_table = (
        summary[
            [
                "feature",
                "hit_rate",
                "decision"
            ]
        ]
        .copy()
    )

    fold_table = (
        fold_table
        .rename(
            columns={
                "hit_rate":
                    f"{fold_name}_Hit_Rate",

                "decision":
                    f"{fold_name}_Decision"
            }
        )
    )

    fold_hit_rate_tables.append(
        fold_table
    )


# ------------------------------------------------------------
# Merge fold-wise results
# ------------------------------------------------------------

feature_temporal_stability = (
    fold_hit_rate_tables[0]
)

for fold_table in (
    fold_hit_rate_tables[1:]
):

    feature_temporal_stability = (
        feature_temporal_stability
        .merge(
            fold_table,
            on="feature",
            how="outer"
        )
    )


# ------------------------------------------------------------
# Calculate temporal stability statistics
# ------------------------------------------------------------

hit_rate_columns = [
    "Fold_1_Hit_Rate",
    "Fold_2_Hit_Rate",
    "Fold_3_Hit_Rate",
    "Fold_4_Hit_Rate"
]


feature_temporal_stability[
    "Mean_Hit_Rate"
] = (
    feature_temporal_stability[
        hit_rate_columns
    ]
    .mean(
        axis=1
    )
)


feature_temporal_stability[
    "Min_Hit_Rate"
] = (
    feature_temporal_stability[
        hit_rate_columns
    ]
    .min(
        axis=1
    )
)


feature_temporal_stability[
    "Max_Hit_Rate"
] = (
    feature_temporal_stability[
        hit_rate_columns
    ]
    .max(
        axis=1
    )
)


feature_temporal_stability[
    "Hit_Rate_Range"
] = (
    feature_temporal_stability[
        "Max_Hit_Rate"
    ]
    -
    feature_temporal_stability[
        "Min_Hit_Rate"
    ]
)


feature_temporal_stability[
    "Hit_Rate_SD"
] = (
    feature_temporal_stability[
        hit_rate_columns
    ]
    .std(
        axis=1,
        ddof=0
    )
)


# ------------------------------------------------------------
# Count number of folds where feature was confirmed
# ------------------------------------------------------------

decision_columns = [
    "Fold_1_Decision",
    "Fold_2_Decision",
    "Fold_3_Decision",
    "Fold_4_Decision"
]


feature_temporal_stability[
    "Confirmed_Folds"
] = (
    feature_temporal_stability[
        decision_columns
    ]
    .eq(
        "Confirmed"
    )
    .sum(
        axis=1
    )
)


feature_temporal_stability[
    "Selection_Frequency"
] = (
    feature_temporal_stability[
        "Confirmed_Folds"
    ]
    / 4
)


# ------------------------------------------------------------
# Final display table
# ------------------------------------------------------------

feature_temporal_stability_display = (
    feature_temporal_stability[
        [
            "feature",
            "Fold_1_Hit_Rate",
            "Fold_2_Hit_Rate",
            "Fold_3_Hit_Rate",
            "Fold_4_Hit_Rate",
            "Mean_Hit_Rate",
            "Min_Hit_Rate",
            "Hit_Rate_Range",
            "Hit_Rate_SD",
            "Confirmed_Folds",
            "Selection_Frequency"
        ]
    ]
    .sort_values(
        [
            "Selection_Frequency",
            "Mean_Hit_Rate"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "E1.11.1 — Feature-Selection Temporal Stability"
)

print("=" * 150)

print(
    feature_temporal_stability_display
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

E1.11.1 — Feature-Selection Temporal Stability
                    feature  Fold_1_Hit_Rate  Fold_2_Hit_Rate  Fold_3_Hit_Rate  Fold_4_Hit_Rate  Mean_Hit_Rate  Min_Hit_Rate  Hit_Rate_Range  Hit_Rate_SD  Confirmed_Folds  Selection_Frequency
        anode_pressure_diff           1.0000           1.0000           1.0000           1.0000         1.0000        1.0000          0.0000       0.0000                4               1.0000
            anode_temp_diff           1.0000           1.0000           1.0000           1.0000         1.0000        1.0000          0.0000       0.0000                4               1.0000
      cathode_pressure_diff           1.0000           1.0000           1.0000           1.0000         1.0000        1.0000          0.0000       0.0000                4               1.0000
                    current           1.0000           1.0000           1.0000           1.0000         1.0000        1.0000          0.0000       0.0000                4               

In [122]:
# ============================================================
# E1.11.2 — Predictive Temporal Stability
# ============================================================

# ------------------------------------------------------------
# Summarise fold-wise chronological validation performance
# already obtained in E1.10.
# ------------------------------------------------------------

predictive_temporal_stability = (
    chronological_validation_df
    .groupby(
        "Model"
    )
    .agg(
        Mean_RMSE=(
            "RMSE",
            "mean"
        ),
        Min_RMSE=(
            "RMSE",
            "min"
        ),
        Max_RMSE=(
            "RMSE",
            "max"
        ),
        RMSE_SD=(
            "RMSE",
            "std"
        ),

        Mean_MAE=(
            "MAE",
            "mean"
        ),
        Min_MAE=(
            "MAE",
            "min"
        ),
        Max_MAE=(
            "MAE",
            "max"
        ),
        MAE_SD=(
            "MAE",
            "std"
        ),

        Mean_R2=(
            "R2",
            "mean"
        ),
        Min_R2=(
            "R2",
            "min"
        ),
        Max_R2=(
            "R2",
            "max"
        ),
        R2_SD=(
            "R2",
            "std"
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# Add metric ranges
# ------------------------------------------------------------

predictive_temporal_stability[
    "RMSE_Range"
] = (
    predictive_temporal_stability[
        "Max_RMSE"
    ]
    -
    predictive_temporal_stability[
        "Min_RMSE"
    ]
)


predictive_temporal_stability[
    "MAE_Range"
] = (
    predictive_temporal_stability[
        "Max_MAE"
    ]
    -
    predictive_temporal_stability[
        "Min_MAE"
    ]
)


predictive_temporal_stability[
    "R2_Range"
] = (
    predictive_temporal_stability[
        "Max_R2"
    ]
    -
    predictive_temporal_stability[
        "Min_R2"
    ]
)


# ------------------------------------------------------------
# Reorder columns for interpretation
# ------------------------------------------------------------

predictive_temporal_stability = (
    predictive_temporal_stability[
        [
            "Model",

            "Mean_RMSE",
            "Min_RMSE",
            "Max_RMSE",
            "RMSE_Range",
            "RMSE_SD",

            "Mean_MAE",
            "Min_MAE",
            "Max_MAE",
            "MAE_Range",
            "MAE_SD",

            "Mean_R2",
            "Min_R2",
            "Max_R2",
            "R2_Range",
            "R2_SD"
        ]
    ]
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print(
    "E1.11.2 — Predictive Temporal Stability"
)

print("=" * 165)

print(
    predictive_temporal_stability
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

E1.11.2 — Predictive Temporal Stability
  Model  Mean_RMSE  Min_RMSE  Max_RMSE  RMSE_Range  RMSE_SD  Mean_MAE  Min_MAE  Max_MAE  MAE_Range   MAE_SD  Mean_R2   Min_R2   Max_R2  R2_Range    R2_SD
  Ridge   0.017102  0.014051  0.019800    0.005748 0.002479  0.013666 0.010049 0.016798   0.006749 0.002881 0.970199 0.962488 0.979747  0.017259 0.007841
XGBoost   0.009636  0.007352  0.011608    0.004257 0.001757  0.007127 0.005210 0.009366   0.004155 0.001717 0.990505 0.987106 0.994348  0.007243 0.002980


In [123]:
# ============================================================
# E1.12.1 — Final Feature-Selection Decision
# ============================================================

# ------------------------------------------------------------
# Final decision rule
#
# Boruta carries the primary methodological weight.
#
# Association, redundancy and VIF are retained as supporting
# diagnostic evidence rather than automatic exclusion rules.
#
# Temporal evidence is provided by:
#   1. Fold-wise Boruta selection stability
#   2. Chronological predictive validation
# ------------------------------------------------------------


final_selected_features = (
    selected_feature_set.copy()
)


final_rejected_features = (
    final_integrated_feature_evidence.loc[
        final_integrated_feature_evidence[
            "Boruta_Status"
        ] == "Rejected",
        "feature"
    ]
    .tolist()
)


final_tentative_features = (
    final_integrated_feature_evidence.loc[
        final_integrated_feature_evidence[
            "Boruta_Status"
        ] == "Tentative",
        "feature"
    ]
    .tolist()
)


# ------------------------------------------------------------
# Decision summary
# ------------------------------------------------------------

final_feature_selection_summary = pd.DataFrame(
    [
        {
            "Decision_Category":
                "Selected",

            "Number_of_Features":
                len(
                    final_selected_features
                ),

            "Features":
                ", ".join(
                    final_selected_features
                )
        },

        {
            "Decision_Category":
                "Tentative",

            "Number_of_Features":
                len(
                    final_tentative_features
                ),

            "Features":
                (
                    ", ".join(
                        final_tentative_features
                    )
                    if final_tentative_features
                    else "None"
                )
        },

        {
            "Decision_Category":
                "Rejected",

            "Number_of_Features":
                len(
                    final_rejected_features
                ),

            "Features":
                (
                    ", ".join(
                        final_rejected_features
                    )
                    if final_rejected_features
                    else "None"
                )
        }
    ]
)


print(
    "E1.12 — Final Feature-Selection Decision"
)

print("=" * 100)

print(
    final_feature_selection_summary
    .to_string(
        index=False
    )
)


# ------------------------------------------------------------
# Additional decision evidence
# ------------------------------------------------------------

print("\nFinal Decision Evidence")
print("=" * 70)

print(
    f"Candidate predictors assessed: "
    f"{len(boruta_candidate_predictors)}"
)

print(
    f"Final selected predictors: "
    f"{len(final_selected_features)}"
)

print(
    f"Tentative predictors: "
    f"{len(final_tentative_features)}"
)

print(
    f"Rejected predictors: "
    f"{len(final_rejected_features)}"
)

print(
    "\nAll selected in every chronological Boruta fold:",
    bool(
        (
            feature_temporal_stability_display[
                "Selection_Frequency"
            ] == 1.0
        ).all()
    )
)


print("\nChronological Predictive Stability")
print("-" * 70)

print(
    predictive_temporal_stability[
        [
            "Model",
            "Mean_RMSE",
            "RMSE_SD",
            "Mean_MAE",
            "MAE_SD",
            "Mean_R2",
            "R2_SD"
        ]
    ]
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

E1.12 — Final Feature-Selection Decision
Decision_Category  Number_of_Features                                                                                                                                                                                                                                                                                                                                                                                                                                           Features
         Selected                  20 anode_pressure_diff, anode_temp_diff, cathode_pressure_diff, current, pressure_anode_outlet, pressure_cathode_outlet, temp_anode_dewpoint_water, temp_anode_endplate, temp_cathode_inlet, total_anode_stack_flow, total_cathode_stack_flow, cathode_dewpoint_offset, temp_cathode_dewpoint_water, temp_anode_outlet, pressure_cathode_inlet, temp_cathode_outlet, pressure_anode_inlet, temp_anode_inlet, cathode_temp_diff, anode_dewpoint_offset
        Tenta

In [124]:
# ============================================================
# E1.13.1 — Save Final Feature-Selection Outputs
# ============================================================

from pathlib import Path
import json
import platform
import sklearn
import xgboost
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Create output directory
# ------------------------------------------------------------

feature_selection_output_dir = Path(
    "../results/feature_selection"
)

feature_selection_output_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 2. Save integrated feature evidence
# ------------------------------------------------------------

final_integrated_feature_evidence.to_csv(
    feature_selection_output_dir
    / "integrated_feature_evidence.csv",
    index=False
)


# ------------------------------------------------------------
# 3. Save cross-fold Boruta consensus
# ------------------------------------------------------------

cross_fold_summary.to_csv(
    feature_selection_output_dir
    / "boruta_cross_fold_summary.csv",
    index=False
)


# ------------------------------------------------------------
# 4. Save feature-selection temporal stability
# ------------------------------------------------------------

feature_temporal_stability_display.to_csv(
    feature_selection_output_dir
    / "feature_selection_temporal_stability.csv",
    index=False
)


# ------------------------------------------------------------
# 5. Save chronological predictive validation
# ------------------------------------------------------------

chronological_validation_df.to_csv(
    feature_selection_output_dir
    / "chronological_validation_results.csv",
    index=False
)


# ------------------------------------------------------------
# 6. Save predictive temporal stability
# ------------------------------------------------------------

predictive_temporal_stability.to_csv(
    feature_selection_output_dir
    / "predictive_temporal_stability.csv",
    index=False
)


# ------------------------------------------------------------
# 7. Save final decision summary
# ------------------------------------------------------------

final_feature_selection_summary.to_csv(
    feature_selection_output_dir
    / "final_feature_selection_summary.csv",
    index=False
)


# ------------------------------------------------------------
# 8. Save fold-wise Boruta results
# ------------------------------------------------------------

for (
    fold_name,
    fold_summary
) in all_fold_boruta_summaries.items():

    fold_summary.to_csv(
        feature_selection_output_dir
        / f"{fold_name.lower()}_boruta_results.csv",
        index=False
    )


# ------------------------------------------------------------
# 9. Helper for JSON-compatible values
# ------------------------------------------------------------

def make_json_serializable(value):

    if isinstance(
        value,
        (np.integer,)
    ):
        return int(value)

    if isinstance(
        value,
        (np.floating,)
    ):
        return float(value)

    if isinstance(
        value,
        (np.bool_,)
    ):
        return bool(value)

    if isinstance(
        value,
        np.ndarray
    ):
        return value.tolist()

    if isinstance(
        value,
        Path
    ):
        return str(value)

    return value


# ------------------------------------------------------------
# 10. Save final specification
# ------------------------------------------------------------

final_feature_selection_specification = {

    "target":
        "voltage",

    "selection_method":
        "Fold-wise XGBoost-Boruta",

    "selection_priority":
        "Boruta assigned highest methodological weight",

    "number_of_candidate_features":
        len(
            boruta_candidate_predictors
        ),

    "number_of_selected_features":
        len(
            final_selected_features
        ),

    "selected_features":
        final_selected_features,

    "tentative_features":
        final_tentative_features,

    "rejected_features":
        final_rejected_features,

    "boruta_settings": {
        key:
            make_json_serializable(value)

        for key, value
        in boruta_settings.items()
    },

    "chronological_folds": {
        fold_name: {
            key: [
                make_json_serializable(x)
                for x in value
            ]

            for key, value
            in fold_info.items()
        }

        for fold_name, fold_info
        in boruta_outer_folds.items()
    },

    "temporal_selection_result": {
        "all_features_confirmed_in_all_folds":
            bool(
                (
                    feature_temporal_stability_display[
                        "Selection_Frequency"
                    ] == 1.0
                ).all()
            ),

        "minimum_mean_boruta_hit_rate":
            float(
                feature_temporal_stability_display[
                    "Mean_Hit_Rate"
                ].min()
            ),

        "maximum_mean_boruta_hit_rate":
            float(
                feature_temporal_stability_display[
                    "Mean_Hit_Rate"
                ].max()
            )
    },

    "software_versions": {
        "python":
            platform.python_version(),

        "pandas":
            pd.__version__,

        "numpy":
            np.__version__,

        "scikit_learn":
            sklearn.__version__,

        "xgboost":
            xgboost.__version__
    }
}


# ------------------------------------------------------------
# 11. Write JSON specification
# ------------------------------------------------------------

with open(
    feature_selection_output_dir
    / "final_feature_selection_specification.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        final_feature_selection_specification,
        file,
        indent=4
    )


# ------------------------------------------------------------
# 12. Confirmation
# ------------------------------------------------------------

print(
    "E1.13 — Final Feature-Selection Specification Saved"
)

print("=" * 70)

print(
    f"Output directory: "
    f"{feature_selection_output_dir.resolve()}"
)

print(
    f"\nSelected features: "
    f"{len(final_selected_features)}"
)

print(
    f"Tentative features: "
    f"{len(final_tentative_features)}"
)

print(
    f"Rejected features: "
    f"{len(final_rejected_features)}"
)

print(
    "\nSaved files:"
)

for file_path in sorted(
    feature_selection_output_dir.iterdir()
):

    print(
        f" - {file_path.name}"
    )

E1.13 — Final Feature-Selection Specification Saved
Output directory: C:\Users\usman\Desktop\PEMFC_Dissertation\results\feature_selection

Selected features: 20
Tentative features: 0
Rejected features: 0

Saved files:
 - boruta_cross_fold_summary.csv
 - chronological_split_definition.json
 - chronological_validation_results.csv
 - feature_selection_temporal_stability.csv
 - final_feature_selection_specification.json
 - final_feature_selection_summary.csv
 - fold_1_boruta_results.csv
 - fold_2_boruta_results.csv
 - fold_3_boruta_results.csv
 - fold_4_boruta_results.csv
 - high_integrated_redundancy_pairs.csv
 - high_pearson_redundancy_pairs.csv
 - high_spearman_redundancy_pairs.csv
 - integrated_feature_evidence.csv
 - integrated_predictor_redundancy.csv
 - integrated_target_relevance.csv
 - mutual_information_target_relevance.csv
 - pearson_predictor_redundancy.csv
 - predictive_temporal_stability.csv
 - predictor_redundancy_groups.csv
 - spearman_predictor_redundancy.csv
 - spearman_t

In [ ]:
# E1.14 — Feature Selection Summary

## Objective

The purpose of this notebook was to identify a scientifically and statistically defensible predictor set for data-driven PEMFC voltage prediction while accounting for nonlinear relationships, predictor redundancy, multicollinearity, and temporal changes across the durability experiment.

Voltage was retained as the prediction target. Power was excluded from the predictor set because Power = Voltage × Current, meaning that its inclusion would introduce direct target leakage.

A total of 20 valid candidate predictors, including measured operating variables and engineered PEMFC indicators, were assessed.

---

## 1. Association Evidence

Three complementary association measures were used to examine predictor relationships with Voltage:

- Pearson correlation assessed linear association.
- Spearman correlation assessed monotonic association.
- Mutual Information assessed broader potentially nonlinear dependence.

The results demonstrated substantial differences in association strength among predictors.

Current showed particularly strong association with Voltage:

- Pearson r = -0.9681
- Spearman ρ = -0.9271
- MI = 1.5338

Other variables, including cathode pressure difference, reactant flows, cathode pressure, and thermal/water-management indicators, also demonstrated meaningful associations.

Importantly, some predictors showed weak Pearson and Spearman relationships but remained informative according to Mutual Information and subsequent Boruta analysis. Therefore, pairwise association strength alone was not used as an automatic feature-elimination criterion.

---

## 2. Predictor Redundancy

Predictor-to-predictor Pearson and Spearman analyses were used to identify strongly overlapping variables.

Several strong relationships were identified, including:

- total anode stack flow ↔ total cathode stack flow
- current ↔ anode/cathode stack flow
- cathode pressure difference ↔ current/stack flow
- anode inlet pressure ↔ anode outlet pressure
- temperature-difference variables ↔ their constituent temperature measurements
- dewpoint-offset variables ↔ associated thermal measurements

These relationships are consistent with the physically coupled operation of the PEMFC system and with the inclusion of engineered variables derived from measured quantities.

Redundancy was therefore treated as diagnostic evidence rather than an automatic reason for predictor removal.

---

## 3. Multicollinearity Diagnostics

Variance Inflation Factor analysis identified substantial multicollinearity within the candidate predictor space.

Several predictors participated in exact or near-exact linear dependencies, particularly where engineered difference/offset variables were included alongside their constituent measurements.

The two stack-flow variables also exhibited extremely high finite VIF values.

These findings demonstrate that the predictor set contains considerable statistical dependence.

However, VIF was used as a diagnostic rather than an automatic exclusion rule. This was considered particularly important because:

- some dependencies arise directly from physically meaningful engineered variables;
- Ridge regression can regularize correlated predictors;
- nonlinear tree-based models such as XGBoost are not subject to the same coefficient-instability problem as ordinary linear regression;
- high redundancy does not necessarily imply absence of predictive relevance.

---

## 4. XGBoost-Boruta Feature Selection

XGBoost-Boruta was used as the primary all-relevant feature-selection method.

Boruta compared the importance of each real predictor against randomized shadow features over repeated iterations.

To account for the temporal nature of PEMFC degradation, Boruta was executed independently across four expanding chronological folds rather than relying on a single randomly mixed dataset.

The folds progressively incorporated later durability stages into training:

- Fold 1: training 50–450 h
- Fold 2: training 50–550 h
- Fold 3: training 50–650 h
- Fold 4: training 50–750 h

Each fold used 50 Boruta iterations.

---

## 5. Boruta Results

All 20 candidate predictors were classified as Confirmed in every chronological fold.

Therefore:

- Confirmed predictors = 20
- Tentative predictors = 0
- Rejected predictors = 0
- Selection frequency = 1.00 for every predictor

Cross-fold mean Boruta hit rates ranged from:

- maximum = 1.000
- minimum = 0.895

Eleven predictors achieved a perfect 1.000 hit rate across all 200 fold-iteration combinations.

The lowest overall hit rate was observed for `anode_dewpoint_offset` at 0.895. Despite its comparatively lower hit strength, it remained Confirmed in all four chronological folds.

The Boruta results therefore provided no statistical basis for rejecting any of the 20 candidate predictors.

---

## 6. Integrated Feature Evidence

Pearson, Spearman, Mutual Information, redundancy, VIF, exact-linear-dependency diagnostics, and Boruta results were integrated into a single feature-evidence table.

Boruta was assigned the highest methodological weight for the final selection decision because it evaluates feature relevance within a multivariate nonlinear predictive framework rather than relying solely on pairwise association.

The integrated evidence showed that predictive relevance and predictor redundancy can coexist.

Consequently, predictors were not removed solely because they:

- had weak Pearson correlation;
- participated in strong predictor-to-predictor correlations;
- exhibited high VIF;
- formed part of an exact linear dependency.

---

## 7. Candidate Feature Sets

Three candidate specifications were originally considered:

- A — Full valid predictor set
- B — Boruta-confirmed predictors
- C — Boruta-confirmed + tentative predictors

Because Boruta classified all 20 predictors as Confirmed:

A = B = C

All three specifications therefore contained exactly the same 20 predictors.

Running separate predictive experiments on these identical feature sets would not provide additional methodological evidence. No artificial reduction was introduced solely to force the creation of a smaller feature set.

---

## 8. Chronological Predictive Validation

The Boruta-selected 20-feature specification was subsequently evaluated using Ridge regression and XGBoost across the same expanding chronological validation framework.

The purpose was to determine whether the selected predictors retained predictive usefulness when applied to later unseen durability stages.

### Ridge

Across the four chronological folds:

- Mean RMSE = 0.017102
- Mean MAE = 0.013666
- Mean R² = 0.970199
- R² SD = 0.007841

### XGBoost

Across the four chronological folds:

- Mean RMSE = 0.009636
- Mean MAE = 0.007127
- Mean R² = 0.990505
- R² SD = 0.002980

XGBoost outperformed Ridge in every chronological fold.

Both models nevertheless maintained high predictive performance across later unseen durability stages.

---

## 9. Temporal Stability

Temporal stability was assessed from two complementary perspectives.

### Feature-selection stability

Every predictor was Confirmed in all four chronological Boruta folds:

- Confirmed folds = 4/4 for every predictor
- Selection frequency = 1.00 for every predictor

Most predictors exhibited very small variation in Boruta hit rates across folds.

`anode_dewpoint_offset` showed the greatest variation:

- Mean hit rate = 0.895
- Minimum hit rate = 0.820
- Hit-rate range = 0.140
- Hit-rate SD = 0.0572

Despite this variation, its selection status remained Confirmed throughout.

### Predictive stability

Ridge:

- RMSE SD = 0.002479
- MAE SD = 0.002881
- R² SD = 0.007841

XGBoost:

- RMSE SD = 0.001757
- MAE SD = 0.001717
- R² SD = 0.002980

There was no monotonic collapse in predictive performance as validation progressed towards later durability stages.

The strongest temporal predictive stability was observed with XGBoost.

These conclusions apply specifically to the chronological validation stages evaluated here, extending from 500 h to 850 h.

---

## 10. Final Feature-Selection Decision

The final decision was to retain all 20 Boruta-confirmed predictors.

No predictor was removed solely because of pairwise redundancy, high VIF, exact linear dependence, or comparatively weak univariate association.

This decision was supported by:

1. confirmation of all predictors by XGBoost-Boruta;
2. confirmation in all four chronological folds;
3. selection frequency of 1.00 for every predictor;
4. consistently high Boruta hit rates;
5. strong chronological predictive performance;
6. stable predictive performance across progressively later validation stages;
7. complementary association evidence;
8. the physical relevance of the measured and engineered PEMFC variables.

The association, redundancy, and VIF analyses therefore remain important components of the methodology even though they did not ultimately cause feature removal. They characterize the statistical structure of the predictor space and provide context for interpreting the Boruta selection results.

---

## Final Outcome

Target:

`voltage`

Final selected predictors:

20

Tentative predictors:

0

Rejected predictors:

0

The complete 20-feature Boruta-confirmed specification will therefore be carried forward into the subsequent modelling stage.

The existing engineered dataset can continue to be used because feature selection did not remove any of the 20 candidate predictors. The saved feature-selection specification records which variables constitute the final predictor set, while `operating_hour` remains available for chronological stage identification rather than being automatically treated as a prediction feature.